# 026. Inverse Cloze Task と passage 単位の 2 段階検索 / Inverse Cloze Task and Two-Stage Retrieval at Passage Level

## 1. 概要 / Overview

検索器(retriever)は、**何を関連とみなして学習するか(関連性の定義)** によって、学ぶものが変わる。[024](./024_text_embedding_and_retriever.ipynb) は「同じ記事から切り出した別の区間」を正例にした
(記事単位の定義)。この定義では、話題が合っていれば解けるので、同じ記事の他の passage との区別は学習の信号に現れない。本トピックは、**passage 単位の定義** の学習として、
**Inverse Cloze Task**(Lee et al., 2019)を実装する。passage から 1 文を抜いて query にし、その文を除いた残りの文脈を正例にする自己教師ありの学習である。
同じ記事の他の passage(話題が同じ)が困難な負例になり、局所的な内容の一致が要る。

2 つの定義の比較は、**入力の形式を揃えて、関連性の定義だけを変える** 設計にする。学習では、どちらの条件も query は「passage(窓)から抜いた 1 文」で、正例の passage は文を 1 つ除いた形である。
passage 単位の定義(R2)の正例は、その文を抜いた元の窓の文脈で、記事単位の定義(R1)の正例は、同じ記事の別の窓の文を除いた形である。さらに、in-batch negatives に同じ記事の組を意図的に入れ、
**関連性の定義を、正例と in-batch の負例の両方に一貫して当てはめる**(R1 は同じ記事の組の passage を負例から除き、R2 は負例として数える)。評価では、同じ索引(すべての passage を 1 文を除いた形にしたもの)と同じ query(抜いた文)を使い、
関連のラベルだけを変える(passage 単位のラベルは正解が query の passage だけ、記事単位のラベルは正解が同じ記事の他の passage)。

そのうえで、後続のアプリ(`apps/`の RAG App)で使う **2 段階の検索**(retrieve-then-rerank)を組む。第 1 段は dual encoder(全探索と HNSW)、第 2 段は query と passage を連結して符号化する cross-encoder で、
cross-encoder は第 1 段の上位の候補だけを並べ替える。並べ替えモデルの学習の入力の分布は、推論時の候補(第 1 段の上位)に揃える。

[025](./025_ann_search_and_reranking.ipynb) の実験 D・E(cross-encoder 対 dual encoder、困難な負例 対 ランダムな負例)は、cross-encoder の学習が成立せず、前提不成立だった(その結論は 025 のとおりで、本トピックはやり直さない)。
025 の事後的な解釈の候補の一つは、記事単位の関連性の定義が、query と passage の語どうしの相互作用を必要としていなかった可能性である。passage 単位の定義は、局所的な内容の一致を要求する点で、この候補と関係する。
**ただし、本トピックの実験はその解釈を検証するものではない。** 本トピックの実験 C は、「アプリで並べ替えを足す価値があるか」という実用の問いである。

検証するのは次の 3 つである(いずれも判定つき。各実験は 1 つの検証事項を扱う)。

- **実験 A**: passage 単位のラベルの評価で、passage 単位の定義(Inverse Cloze Task)で学習した検索器(R2)の Recall@10 が、記事単位の定義で学習した検索器(R1)より高いか。
- **実験 B**: 同じ索引・同じ query の、記事単位のラベルの評価で、R1 の Recall@10 が R2 より高いか。
- **実験 C**: passage 単位のラベルの評価で、R2 の上位の候補を cross-encoder で並べ替えた構成(R3)の Recall@10 が、R2 だけ(並べ替えなし)より高いか。

実験 A と B を合わせると「定義が学ぶものを決める」という交差(各定義で学習した検索器が、その定義のラベルの評価で優れる)を表すが、部分ごとに別の検証事項として判定し、組み合わせの読み方は 6.1 節で事前に宣言する。
あわせて、判定を設けない観察として、(D)R2 の埋め込みの HNSW 索引で第 1 段を近似したときの 2 段階の検索の費用(距離計算の回数・順伝播の回数)と検索指標を、並べ替える候補の数 $K$ の数点で示し、
(E)同じ query に対する R1 と R2 の検索結果の例を示す。診断量として、024 と同じ形式の記事単位の評価(024 の評価用の索引と query)も、R1・R2・024 の Hub のモデルの 3 つで測る(判定には使わない)。

### 1.1 実行の手順

本番実行は Google Colab T4 の **1 つのセッションで完結** させる。5.1 節のセットアップセルで`SMOKE_TEST = False`にして、「すべてのセルを実行」する。

実行時間の予算は 1 セッションあたり **T4 で 120 分** とする。学習を始める前に(6.2 節)、T4 上でのスケーリングの計測から、各処理の時間を見積もる。続いて、cross-encoder の学習ステップ数を決める
パイロット(6.3 節。本番と同じセッションの冒頭で行い、その時間も予算に含める)を行い、パイロットの後の残りの時間で、**実行計画**(6.1 節。削る段階の組)のそれぞれの残りの実行時間を見積もり(6.4 節)、
予算に収まる計画のうち優先順位の最も高いものを自動で選ぶ。選択は見積もりと経過時間のみに基づき、どの実験の結果も参照しない。**選ばれた計画は 6.4 節の出力に印字される。**
事前の計測のための別の実行やセッションの分割はしない。見積もりが最も下位の計画でも予算を超える場合は、本番の学習の前に停止する。

学習したモデルのうち、後続のアプリ(`apps/`)の入力にする候補の **R2 のシード 0**(`kojikojiprg/ai-theories-text-embedding-en`の新しいブランチ`inverse-cloze-task`)と
**cross-encoder のシード 0**(新しいリポジトリ`kojikojiprg/ai-theories-cross-encoder-en`)は、公開の条件(判定ではない。6.1 節)を満たし、かつ`UPLOAD_ARTIFACTS`が`True`のときだけ、
Hugging Face Hub にアップロードする(6.14 節。既定は`False`)。他のモデルは条件間の比較のためのものであり、保存もアップロードもしない。

### 1.2 本番実行の結果の要約

本番の実行の後に、実行環境、選ばれた計画、前提条件の成否、判定の一覧、要点をここに記す。

### 1.3 本番実行の後に更新が必要な箇所

- 1.2 節(本番実行の結果の要約)。
- 6.1 節の「検出力の事前確認」と「前提条件の閾値の下限」の、本番の値との対応(7.2 節に記す)。
- 7 章(結果・考察)のすべての節の本文。
- 6.14 節のアップロードの結果(`UPLOAD_ARTIFACTS`を`True`にして実行した場合は、7.10 節に確認の出力を記す)。

## 2. 参考論文 / References

1. Lee, K., Chang, M.-W., Toutanova, K.,
   "Latent Retrieval for Weakly Supervised Open Domain Question Answering", ACL 2019. https://arxiv.org/abs/1906.00300
   (Inverse Cloze Task。passage から 1 文を抜いて query にし、残りを正例にする。抜いた文を文脈に残す割合 10% の理由。3.2 節・実験 A・B)
2. Chang, W.-C., Yu, F. X., Chang, Y.-W., Yang, Y., Kumar, S.,
   "Pre-training Tasks for Embedding-based Large-scale Retrieval", ICLR 2020. https://arxiv.org/abs/2002.03932
   (Inverse Cloze Task・Body First Selection・Wiki Link Prediction の 3 つの事前学習の課題の比較。3.6 節、位置づけのみ)
3. Karpukhin, V., Oğuz, B., Min, S., Lewis, P., Wu, L., Edunov, S., Chen, D., Yih, W.,
   "Dense Passage Retrieval for Open-Domain Question Answering", EMNLP 2020. https://arxiv.org/abs/2004.04906
   (DPR(Dense Passage Retrieval)。質問と答えを含む passage の組による学習。in-batch negatives と困難な負例。3.3・3.6 節)
4. Izacard, G., Caron, M., Hosseini, L., Riedel, S., Bojanowski, P., Joulin, A., Grave, E.,
   "Unsupervised Dense Information Retrieval with Contrastive Learning", TMLR 2022. https://arxiv.org/abs/2112.09118
   (Contriever。同じ文書の別の区間を正例にする定義。記事単位の定義の原型。3.1 節)
5. Nogueira, R., Cho, K.,
   "Passage Re-ranking with BERT", arXiv:1901.04085, 2019. https://arxiv.org/abs/1901.04085
   (cross-encoder による再順位づけ。3.5 節)
6. Kusupati, A., Bhatt, G., Rege, A., Wallingford, M., Sinha, A., Ramanujan, V., Howard-Snyder, W., Chen, K., Kakade, S., Jain, P., Farhadi, A.,
   "Matryoshka Representation Learning", NeurIPS 2022. https://arxiv.org/abs/2205.13147
   (入れ子の次元の損失。第 1 段の学習の損失は 024 の条件 C5 と同じ)
7. Malkov, Y. A., Yashunin, D. A.,
   "Efficient and Robust Approximate Nearest Neighbor Search Using Hierarchical Navigable Small World Graphs", IEEE TPAMI 2020. https://arxiv.org/abs/1603.09320
   (HNSW。観察 D で第 1 段を近似する。025 で実装)
8. Ma, J., Korotkov, I., Yang, Y., Hall, K., McDonald, R.,
   "Zero-shot Neural Passage Retrieval via Domain-targeted Synthetic Question Generation", arXiv:2004.14503, 2020. https://arxiv.org/abs/2004.14503
   (文章から合成した質問と文章の組による学習。3.6 節、位置づけのみ)
9. Dai, Z., Zhao, V. Y., Ma, J., Luan, Y., Ni, J., Lu, J., Bakalov, A., Guu, K., Hall, K. B., Chang, M.-W.,
   "Promptagator: Few-shot Dense Retrieval From 8 Examples", arXiv:2209.11755, 2022(ICLR 2023). https://arxiv.org/abs/2209.11755
   (大規模言語モデルによる合成の質問の生成で検索器を学習する。3.6 節、位置づけのみ)

本文で用いる既存トピックの部品: 小型 GPT は [006](../02_pretraining/006_pretraining_small_gpt.ipynb)、AdamW・warmup + cosine・gradient clipping は [007](../02_pretraining/007_training_stabilization.ipynb)、
本トピックが起点にする学習済みの小型 GPT は [008](../02_pretraining/008_decoding_strategies.ipynb)、FP16 の autocast と動的損失スケーリングは [011](../03_efficient_training/011_mixed_precision_training.ipynb)、
記事を単位とするクラスタブートストラップは [015](../03_efficient_training/015_long_context_extension.ipynb)、dual encoder・InfoNCE・in-batch negatives・Matryoshka Representation Learning・評価の手順は
[024](./024_text_embedding_and_retriever.ipynb)、HNSW・cross-encoder・2 段階の検索の費用と学習の入力の分布の揃え方は [025](./025_ann_search_and_reranking.ipynb) で扱った。

## 3. 理論 / Theory

### 3.1 動機: 関連性の定義が、検索器の学ぶものを決める

検索器の学習は、query $q$ に対して「どの passage を関連とみなすか」(正解の集合 $G_q$)を決めて初めて定まる。正解の決め方が違えば、同じ構造・同じ損失でも、学ぶ表現が変わる。

| 定義 | 正解の集合 $G_q$ | 本トピックでの記号 | 学習の正例の作り方 |
|---|---|---|---|
| 記事単位 | query と同じ記事の、query の元の passage 以外の passage | R1 | query を抜いた窓とは別の、同じ記事の窓(文を 1 つ除いた形)。024 の independent cropping(Contriever [4])に相当する定義 |
| passage 単位 | query の文を抜いた元の passage ただ 1 つ | R2 | Inverse Cloze Task [1]: query を抜いた窓の、その文を除いた残り |

(表の R1・R2 は、本トピックで学習する 2 つの検索器の条件を表す記号である。)

- **記事単位** では、query の話題が合っている passage ならどれでも正解なので、話題の一致で解ける。同じ記事の他の passage は、すべて正解であり、互いを区別する信号は学習に現れない。
- **passage 単位** では、同じ記事の他の passage は話題が同じなのに正解ではない(困難な負例)。query の文の局所的な内容(固有名詞、数、文脈の語)が、元の passage の文脈と一致することが手がかりになる。

**入力の形式を揃える**: 024 の R1 に相当する学習は、固定長の窓を query と正例にするので、query が文である Inverse Cloze Task の入力と形式が違う。形式が違うと、各条件が学習で見た形式の評価で勝つだけで、関連性の定義の効果と形式の一致の効果を分けられない。
本トピックは、**R1 の query も、R2 と同じく「窓から抜いた 1 文」にし、正例の passage も、どちらの条件も文を 1 つ除いた形にする**。2 つの条件の違いは、(1)どの passage を正例にするか(query の元の窓 / 同じ記事の別の窓)と、(2)in-batch negatives の扱い(3.3 節)だけである。
評価も、同じ索引(すべての passage を 1 文を除いた形にしたもの)と同じ query(抜いた文)で、関連のラベルだけを変える(3.5 節)。

[025](./025_ann_search_and_reranking.ipynb) の実験 D・E は前提不成立だった。025 の事後的な解釈の候補の一つは、記事単位の関連性の定義が、query と passage の語どうしの相互作用を必要としていなかった
(話題の一致で解けるので、相互作用のある cross-encoder の利点が出なかった)可能性である。passage 単位の定義は局所的な内容の一致を要求する点でこの候補と関係するが、**本トピックの実験はその解釈を検証するものではない**。
本トピックが確かめるのは、定義の違いが検索器の指標にどう現れるか(実験 A・B)と、2 段階の検索で並べ替えを足す価値があるか(実験 C)である。

### 3.2 Inverse Cloze Task

passage $p$ が $m$ 個の文 $s_1, \dots, s_m$ からなるとする($m \ge 2$)。Inverse Cloze Task [1] は、次の手順で学習の組を作る(cloze は、文の穴埋め問題の意味で、その「逆」: 穴(抜いた文)から、穴の周りの文脈を探す)。

1. $s_i$ を、条件を満たす文の中から一様に選ぶ($i$ は抜く文の番号)。$q = s_i$ を query とする。
2. 確率 $1 - r$ で、文脈 $c = p \setminus s_i$(passage から $s_i$ を除いたもの)を正例にする。確率 $r$ で、文を抜かず、文脈 $c = p$(query の文を含む)を正例にする。

$r$ は抜いた文を文脈に残す確率で、原論文 [1] は $r = 0.1$ とする(文を抜く割合は 90%)。**文を抜かない組を混ぜる理由**: 常に文を抜くと、検索器は「語の一致」が強い手がかりになることを学べない。
文を残した組では、query の語が文脈にそのまま現れるので、語の一致を学べる。文を抜いた組では、語の一致に頼れず、話題・固有名詞・文脈の語などの抽象的な表現が要る。
原論文は、両方を学ばせるために、文を抜く割合を 90% にしている。本トピックも $r = 0.1$ を採る。

学習の損失は、[024](./024_text_embedding_and_retriever.ipynb) と同じ in-batch negatives の InfoNCE である。$N$ 組のバッチで、$u_i$ を query $q_i$ の、$v_j$ を文脈 $c_j$ の L2 正規化した埋め込み、
$\tau$ を温度として

$$
\mathcal{L}_{\mathrm{NCE}} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(u_i^\top v_i / \tau)}{\sum_{j=1}^{N} \exp(u_i^\top v_j / \tau)}
$$

である。本トピックの第 1 段の学習は、さらに Matryoshka Representation Learning [6] の入れ子の次元の和を使う(損失の形は 024 の条件 C5 と同じ。R1 と R2 で揃える量)。

```mermaid
flowchart LR
    subgraph B["同じ記事から 2 つの窓 p1・p2(重ならない)"]
        direction LR
        b1["窓 p1 から 1 文 s1 を抜く"] --> b2["query s1"]
        b1 --> b3["文脈 c1 = p1 から s1 を除いた残り"]
        b4["窓 p2 から 1 文 s2 を抜く"] --> b5["query s2"]
        b4 --> b6["文脈 c2 = p2 から s2 を除いた残り"]
    end
    b2 --> R2["R2: s1 の正例は c1、s2 の正例は c2"]
    b3 --> R2
    b2 --> R1["R1: s1 の正例は c2、s2 の正例は c1"]
    b6 --> R1
```

(図は補助的な視覚化であり、本文の数式と文章だけでも理解が成立する。)

**本トピックの学習の組**: 1 つのバッチは、32 個の記事 × 各記事 2 組の 64 組である。各記事から、互いに重ならない 2 つの窓 $p_1, p_2$ を引き、それぞれから抜く文 $s_1, s_2$ を選ぶ。
R2 は $(s_1, c_1)$・$(s_2, c_2)$ を組にし($c_i$ は $p_i$ から $s_i$ を除いた文脈)、R1 は $(s_1, c_2)$・$(s_2, c_1)$ を組にする(同じ記事のもう一方の窓の、文を除いた形が正例)。
したがって、同じバッチに入る query と passage のトークンは R1 と R2 で同じで、**正例の対応だけが違う**。文を文脈に残す確率 $r$ も共通である(文脈の組ごとに決める。R1 の正例にも同じ文脈をそのまま使う)。

### 3.3 in-batch negatives で、同じ記事の組が同じバッチに入ったときの扱い

in-batch negatives は、同じバッチの他の組の passage を負例とみなす。同じ記事から複数の組が同じバッチに入ったとき、2 つの定義で扱いが **逆** になる。

| 定義 | 同じバッチの、同じ記事の他の組の passage | 損失での扱い |
|---|---|---|
| 記事単位(R1) | 実際には正解(関連がある) | 負例から **除く**(損失の候補から取り除く)。負例として押しのけると、意味的に近いものを遠ざける(偽の負例、false negative) |
| passage 単位(R2) | 実際には負例(話題が同じで正解ではない) | 負例として **数える**。局所的な内容の一致を要求する困難な負例になる |

本トピックは、**同じ記事の組を意図的に同じバッチに入れ**(32 記事 × 2 組)、損失での扱いだけを定義に合わせる。R1 の候補は、query 1 つあたり、正例 1 つと、別の記事の passage 62 個(同じ記事の組の passage 1 つを除く)で、
R2 の候補は、正例 1 つと、63 個の負例(同じ記事の組の passage を含む)である。**関連性の定義を、正例と in-batch の負例の両方に一貫して当てはめること** が、実験 A・B の介入である。
InfoNCE の損失は、除く組を $\mathcal{E}_i$(query $i$ と同じ記事の他の組の集合。R2 では空集合)として

$$
\mathcal{L}_{\mathrm{NCE}} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(u_i^\top v_i / \tau)}{\sum_{j \notin \mathcal{E}_i} \exp(u_i^\top v_j / \tau)}
$$

である(和は、除く組を除いたすべての $j$ にわたり、正例 $j = i$ を含む)。

### 3.4 文の分割: トークン列の上の規則

passage は 008 のトークナイザ(バイトレベルの BPE)のトークン列である。文の区切りは、テキストにデコードせず、トークンごとの表(トークン ID を引く真偽の表)で次のように決める。このトークナイザは、空白と改行を直後の語の先頭に付けるので、
「次のトークンが空白または改行で始まる位置」が語の境目になる。位置 $t$ の直後で文が終わるのは、次のいずれかである。

1. 次のトークンが **改行** で始まる(行の終わり。箇条書き・見出し・段落の区切り)。
2. 次のトークンが **空白** で始まり、かつ位置 $t$ のトークンのテキストが`.`・`!`・`?`で終わる(直後に閉じ引用符・閉じ括弧が付いていてもよい)。位置 $t$ のトークンが閉じ引用符・閉じ括弧だけで、その直前のトークンが`.`・`!`・`?`で終わる場合も含める。

passage の端は文の端として数える。したがって、passage の先頭と末尾の文は、元の文の一部(断片)であることがある。**既知の誤り**: 略語・イニシャル(`U.S.`・`Dr.`など)の後の空白で分割してしまい、`3.14`のように空白を伴わないものは分割しない。略語の辞書は使わない。

**抜く文の条件**: 長さが $L_{\min} = 4$ トークン以上 $L_{\max} = 96$ トークン以下の文(短すぎる文は query にならず、長すぎる文を抜くと文脈が短くなりすぎる)。passage に文が 2 つ以上あり、この条件を満たす文が 1 つ以上ある passage だけが、Inverse Cloze Task の対象になる。
**切り詰めを減らす選び方**: 抜く文は、query の長さの上限 $L_q = 32$ トークン以内に収まる文があれば、その中から一様に選び、収まる文がない passage だけ、より長い文から選んで先頭を切り詰める。学習と評価の索引の両方に同じ規則を適用する。
**query の長さ**: 抜いた文の先頭の $L_q = 32$ トークン(024 と同じ上限。文がそれより長ければ先頭を切る。抜く範囲は文の全体)。文脈の長さは、窓の長さ $L_p = 128$ から抜いた文の長さを引いた値で、組ごとに違う。

### 3.5 評価の索引と、関連のラベル(passage 単位と記事単位)

評価では、索引のすべての passage について、抜く文を決定的に 1 つ選び、その文を **除いた形** を索引に入れる。抜いた文がその passage の query である。
索引の passage がすべて「1 文を除いた形」になるので、除いたかどうかが手がかりにならない(対象でない passage、すなわち文を除けない passage は、除いていない形で索引に入れず、索引から除く)。
**同じ索引・同じ query に、関連のラベルだけを変えて 2 回当てる**。

- **passage 単位のラベル**(実験 A): 正解は query の passage $p_i^+$ だけ。スコアを $s(q_i, p) = f(q_i)^\top f(p)$ として、正例の順位は

$$
r_i = 1 + \bigl\lvert \{\, p \ne p_i^+ : s(q_i, p) \ge s(q_i, p_i^+) \,\} \bigr\rvert
$$

  である(同点は正例に不利に数える: 正例と同じスコアの正例でない passage は、正例より上位とみなす)。
- **記事単位のラベル**(実験 B。024 と同じ定義): 正解は、query と同じ記事の、query の passage 以外の passage の集合 $G_i$。**query の passage 自身は候補から除く**(正解でも不正解でもない)。順位は、最もスコアの高い正解の順位

$$
r_i = 1 + \bigl\lvert \{\, p \notin G_i \cup \{p_i^+\} : s(q_i, p) \ge \max_{g \in G_i} s(q_i, g) \,\} \bigr\rvert
$$

  (同点は正解に不利)。正解が複数あるので、Recall@$k$ は **上位 $k$ 件に正解が 1 つ以上ある query の割合** である。

どちらのラベルも Recall@$k$ は $\frac{1}{Q} \sum_{i=1}^{Q} \mathbb{1}[r_i \le k]$($Q$ は query の数)。主指標は Recall@10 で、平均逆順位(Mean Reciprocal Rank)を診断量にする。
**query は、記事単位のラベルの正解(同じ記事の他の passage)が索引に 1 つ以上ある query に限る**(2 つのラベルで同じ query を使うため)。評価の対象の検索は、コーパス全体の索引(使う記事すべての passage)で行う。

診断量として、024 と同じ形式の記事単位の評価(024 の評価用の索引と query。固定長の passage と query、定義は 024 の 3.9 節)を、R1・R2・024 の Hub のモデルの 3 つで測る。

### 3.6 2 段階の検索(retrieve-then-rerank)と、並べ替えモデルの学習の入力の分布

全 passage との組を cross-encoder で符号化するのは費用が大きいので、第 1 段(dual encoder による全探索または近似最近傍探索)が上位 $K$ 件の候補を絞り込み、第 2 段(cross-encoder)がその $K$ 件だけを並べ替える。
1 query あたりの費用は

$$
C_{\mathrm{total}} = C_{\mathrm{first}} + K \cdot F_{\mathrm{pair}}
$$

である($C_{\mathrm{first}}$ は第 1 段の距離計算の回数(全探索なら索引の passage の数 $P$)、$K$ は並べ替える候補の数、$F_{\mathrm{pair}}$ は 1 組あたりの順伝播の費用)。cross-encoder の順伝播の回数は $K$ 回である。
並べ替えで到達できる Recall@10 の上限は、上位 $K$ 件に正例が入っている query の割合(coverage)である。

```mermaid
flowchart LR
    q["query q"] --> e1["dual encoder f(R2)"]
    e1 --> s1["第 1 段: 索引(全探索 / HNSW)の上位 K 件"]
    s1 --> x["cross-encoder: q と各候補を連結して符号化"]
    x --> r["スコアの高い順に並べ替え"]
```

**並べ替えモデルの学習の入力の分布を、推論時に見る入力の分布に揃える**。推論時の候補は、第 1 段の上位 $K$ 件である。学習で、正例を第 1 段の上位の外の passage から選ぶと、
「第 1 段のスコアが低い候補が正例」という、推論時には成り立たない手がかりで損失を下げられてしまう。そのため本トピックは、正例が第 1 段の上位 $K$ 件に入る query だけを使い、
負例も同じ上位 $K$ 件から(正例を除いて)選ぶ。学習の query は、索引に入れる形(文を除いた passage)と同じ作り方で、学習用の記事の passage から作る。

cross-encoder の構成([025](./025_ann_search_and_reranking.ipynb) と同じ): 列`[query] [区切り] [passage] [終端]`を因果マスクのもとで符号化し、**終端の位置の隠れ状態**(query 全体と passage 全体の両方を見ている唯一の位置)に線形の層を付けてスカラーのスコアにする。
query と passage の長さが組ごとに違うので、列は左から詰めて並べ、終端の位置は列の有効な長さの最後にする。

### 3.7 位置づけのみ(実装しない)

- **Body First Selection と Wiki Link Prediction**(Chang et al. [2]): Inverse Cloze Task と並ぶ、文書の構造を使う自己教師ありの検索の課題。Body First Selection は、記事の最初の節の中の文を query にして、同じ記事の別の passage を正例にする。
  Wiki Link Prediction は、ハイパーリンクのある文を query にして、リンク先の記事の passage を正例にする。[2] は、3 つの課題を組み合わせると、単独より検索の指標が高くなることを報告した。
- **DPR**(Karpukhin et al. [3]): 質問と、答えを含む passage の組による教師あり学習。BM25 による困難な負例と in-batch negatives を使う。質問と passage の組は、人手のラベルを要する。
- **合成の質問の生成**(Ma et al. [8]、Dai et al. [9]): 文章から質問を生成する生成モデル(小規模または大規模な言語モデル)で、質問と文章の組を自動で作り、検索器を学習する。Inverse Cloze Task は、生成モデルを使わず、文章自身の文を query にする最も単純な合成の形とみなせる。

### 3.8 データの流れとアルゴリズム(擬似コード)

```
第 1 段の学習(R1 と R2 で、記事・窓・抜く文・バッチ・温度・損失・ステップ数を揃える。違いは正例の対応と、同じ記事の組の扱い):
  各ステップで 32 個の記事を非復元抽出する
  各記事から、互いに重ならない長さ 128 の窓 p1・p2 を引く(文が 2 つ以上あり、長さ 4〜96 の文がある窓になるまで引き直す)
  各窓から、長さ 32 以内に収まる文があればその中から、なければ長い文から、文 s_k を 1 つ一様に選んで抜く。query = s_k の先頭 32 トークン
  文脈 c_k = p_k から s_k を除いた残り(確率 0.1 では文を抜かず、窓全体)
  R2: 組 = (s1, c1)・(s2, c2)。同じ記事の組の passage も負例として数える(63 個の負例)
  R1: 組 = (s1, c2)・(s2, c1)。同じ記事の組の passage は関連があるので負例から除く(62 個の負例)
  z_q = pool(encoder(query))、z_p = pool(encoder(passage))             # 右側のパディングを除いた平均プール
  Matryoshka Representation Learning の InfoNCE(R1 は除く組をマスク)-> gradient clipping -> AdamW(学習率は warmup + cosine)

評価の索引の構築(コーパス全体の passage について、決定的に):
  抜く文を、(シード、記事、窓の番号)だけで決まる乱数で 1 つ選ぶ(収まる文を優先)。対象でない passage は索引に入れない
  索引に入れる形 = passage から文を除いたもの。query = 抜いた文の先頭 32 トークン
  passage 単位のラベル: 正解 = その passage だけ。記事単位のラベル: 正解 = 同じ記事の他の passage(その passage 自身は候補から除く)

2 段階の検索:
  第 1 段(R2)で query の上位 K = 50 件を求める -> cross-encoder が q と各候補を連結して符号化し、スコアを出す -> スコア順に並べ替える

cross-encoder の学習:
  学習用の記事の passage の query(索引の形で作ったもの)のうち、正例が第 1 段(R2)の上位 50 件に入る query だけを使う
  各ステップで 8 個の query を引き、正例 1 個と、上位 50 件から正例を除いて選んだ負例 7 個の 8 個の候補の上の softmax 交差エントロピーを最小にする
```

## 4. 実装方針 / Implementation Policy

**`src/`に切り出す(スクラッチ実装、本トピックで新規作成)**:

- `src/data/inverse_cloze.py`: 文の分割の表`build_sentence_tables()`と文の境界`sentence_boundaries()`、抜く文の選択`choose_cloze_sentence()`、窓から文を抜く`remove_span()`・`extract_span()`、
  学習の組`sample_cloze_pair_batches()`(Inverse Cloze Task。1 つの記事から重ならない 2 つの窓を引き、同じバッチに入れる。右側をパディングしたトークンと長さの配列)、passage 単位の評価の索引`build_cloze_index()`(抜く文を決定的に選び、除いた形を索引にする)。
- `src/models/variable_length.py`: 可変長の入力の平均プール`pooled_with_lengths()`、可変長の符号化`encode_tokens_with_lengths()`、可変長の cross-encoder`VariableLengthCrossEncoder`(列を左から詰める)。
  既存の`TextEmbeddingModel`・`CrossEncoder`は変更しない。
- `src/training/contrastive_text_variable.py`: 学習の組を受け取る第 1 段の学習ループ`train_text_embedding_from_batches()`と、同じ組での損失の測定`compute_pair_losses_from_batches()`
  (024 の`train_text_embedding()`と同じ部品・同じ演算。すべての行が満長で、除く組がないときは同じ損失の系列になる)。同じ記事の組を負例から除くマスクつきの損失`masked_info_nce_loss()`・`masked_matryoshka_representation_loss()`を持つ。
- `src/training/reranking_variable.py`: cross-encoder の学習の組`sample_cloze_reranking_schedule()`と学習ループ`train_reranker_variable()`、同じ組での損失の測定`compute_reranking_losses_variable()`。
- `src/retrieval/passage_level.py`: 単一の正例の順位`rank_single_positive()`(同点は正例に不利)、同じ索引・同じ query の記事単位のラベルの順位`rank_article_label()`、並べ替えの順位`rerank_single_positive()`、可変長の cross-encoder による候補のスコア`score_candidates_variable()`。

**既存モジュールの扱い**: `src/`は新しいファイルの追加のみで、既存のファイルを変更しない(5.4 節で、変更前のコミットとの差分で確かめる)。
024 の`src/data/retrieval.py`・`src/retrieval/evaluation.py`・`src/models/text_embedding.py`・`src/training/contrastive_text.py`と、025 の`src/retrieval/`・`src/models/cross_encoder.py`・`src/data/reranking.py`・`src/training/reranking.py`は、そのまま再利用する。

**ノートブック内に直接書く(026 固有)**: 条件の定義、データの分割・コーパスの構成の確認、単体テストと不変条件の確認、スケーリングの計測、パイロット、実行計画の選択、学習率の較正、判定、可視化、Hub へのアップロード。

**生成物の置き場所**:

| 生成物 | 置き場所 |
|---|---|
| Hugging Face Hub から取得したコーパス・トークナイザ・008 のモデル・024 のモデル | `huggingface_hub`の既定のキャッシュ(Colab ではセッションの終了とともに破棄される) |
| 符号化した記事のトークン列 | `.cache/026_encoded/`(入力から決定的に再生成できるもの。同一セッション内の高速化のみを目的とする) |
| 文の分割の表・コーパスの passage の索引・Inverse Cloze Task の索引・学習の組 | メモリ上のみ(いずれも入力とシードから決定的に再生成できる) |
| 学習したモデル(R2 と cross-encoder のシード 0 を除く) | メモリ上のみ(評価の直後に破棄する) |
| R2 のシード 0・cross-encoder のシード 0 の学習後の重み、R1・R2 のシード 0 の索引の埋め込み | メモリ上に保持し、観察 D・E と 6.14 節のアップロードに使う(`.cache/`には置かない) |
| 学習の履歴・query ごとの順位・較正で決めた値・パイロットの結果 | メモリ上のみ(判定と図に使う) |
| 判定の記録 | 判定と前提条件を計算した各セルの出力(6.8〜6.13 節) |
| Hub へアップロードするモデル(R2 のシード 0、cross-encoder のシード 0) | `kojikojiprg/ai-theories-text-embedding-en`の`inverse-cloze-task`ブランチ、`kojikojiprg/ai-theories-cross-encoder-en`(公開の条件を満たし、`UPLOAD_ARTIFACTS`が`True`のときのみ) |

**外部からの取得**(Colab のセットアップセルのリポジトリの取得と依存関係のインストールを除く):

| 取得するもの | リポジトリ | 用途 |
|---|---|---|
| コーパス(`corpus.txt`・`metadata.json`) | `kojikojiprg/ai-theories-corpus-en`(Dataset、9,826 記事、約 546 MB) | 検索の対象のコーパスと記事の境界(`article_offsets`)。Wikipedia API にはフォールバックしない |
| 参照コーパス(`corpus.txt`) | `kojikojiprg/ai-theories-corpus-en-pretraining`(Dataset、356 記事、約 24 MB) | 008 の事前学習とトークナイザの学習に使った範囲の特定(先頭の 356 記事を使わないことの確認) |
| トークナイザ(`tokenizer.json`) | `kojikojiprg/ai-theories-tokenizer-en` | 008 の英語の BPE(Byte Pair Encoding)トークナイザ(語彙サイズ 8192) |
| 008 の学習済みモデル(`config.json`・`model_state.pt`、`main`) | `kojikojiprg/ai-theories-small-gpt-en` | R1・R2 と cross-encoder の起点 |
| 024 の学習済みモデル(`config.json`・`model_state.pt`、`main`) | `kojikojiprg/ai-theories-text-embedding-en` | R1 の再現の確認(診断量)だけに使う。学習の起点にも評価の対象にもしない |

**アップロード方針**: アップロードするのは R2 のシード 0 と cross-encoder のシード 0 のモデルだけで、結果を見て選ばない(後続のアプリ(`apps/`)の入力になるため)。
トークナイザは同梱せず、モデルカードで`kojikojiprg/ai-theories-tokenizer-en`を参照する。モデルカードの指標は、アップロードする重みそのものを読み込み直して評価した値とする。

## 5. 実装 / Implementation

### 5.1 環境セットアップ(Google Colab)

`SMOKE_TEST`(動作確認か本番か)はこのセルでのみ切り替える。実行環境はこのセルで 1 回だけ印字する。

**本番のコミットの確認**: `SMOKE_TEST = False`のときだけ、(1)追跡しているファイルに未コミットの変更がないこと、(2)`REQUIRED_ANCESTOR_COMMIT`が指定されていれば、
それが HEAD の祖先であることを確かめ、満たさなければ学習の前に停止する。追跡外のファイルは停止の条件にせず、あれば一覧を参考として印字する。

**精度と決定性**: 学習は、CUDA のときは FP16 の`torch.autocast`と動的損失スケーリング(011 の`DynamicLossScaler`)で行い、それ以外のデバイス(ローカルの MPS(Metal Performance Shaders、Apple Silicon の GPU 用のバックエンド)・CPU)では FP32 で行う。
埋め込みの正規化・類似度・損失・評価は常に FP32 で行う。`torch.use_deterministic_algorithms(True)`は使わない。同じシードの学習を繰り返しても結果が bit 単位では一致しないことがあるが、
条件間の対応付け(同じシードの条件どうしで記事の並びとバッチを揃えること)は乱数の生成器によって決まり、演算の決定性には依存しない(6.12 節で確かめる)。

In [1]:
# 環境セットアップ(Google Colab)
import os
import sys
import time

SMOKE_TEST = True  # True: 縮小スケールでの動作確認、False: 本番(Colab T4 の 1 セッション)
UPLOAD_ARTIFACTS = False  # 6.14 節。既定は False(Colab Secrets の HF_TOKEN を使ってアップロードするときだけ True)

NOTEBOOK_START_TIME = time.time()

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("/content/ai-theories"):  # 2 回目の実行では clone し直さない
        !git clone https://github.com/kojikojiprg/ai-theories.git /content/ai-theories
    %cd /content/ai-theories
    !pip install uv -q
    !uv pip install --system -r requirements.txt

# インストールで版が入れ替わったパッケージの古い版がメモリに残っていないかを、torch などを import する前に確かめる。
# Colab では食い違いがあればカーネルを終了する(再接続後、もう一度「すべてのセルを実行」する)
from src.utils.environment import check_preloaded_package_versions  # noqa: E402

check_preloaded_package_versions(on_mismatch="restart" if IN_COLAB else "raise")

import torch  # noqa: E402

from src.utils.environment import print_execution_environment  # noqa: E402

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
USE_FP16_AUTOCAST = device.type == "cuda"  # FP16 の autocast と動的損失スケーリングは CUDA のときのみ
execution_environment = print_execution_environment(device)

# 本番(SMOKE_TEST = False)は、追跡しているファイルに未コミットの変更がない状態で実行する。特定の修正を含むコミットで実行する場合は、
# そのコミットを REQUIRED_ANCESTOR_COMMIT に指定する(HEAD がその子孫であることを確かめる)
REQUIRED_ANCESTOR_COMMIT = None
if not SMOKE_TEST:
    import subprocess

    _head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
    _status = subprocess.run(
        ["git", "status", "--porcelain", "--untracked-files=no"], capture_output=True, text=True, check=True
    ).stdout.strip()
    _untracked = subprocess.run(
        ["git", "ls-files", "--others", "--exclude-standard"], capture_output=True, text=True, check=True
    ).stdout.split()
    _is_descendant = REQUIRED_ANCESTOR_COMMIT is None or (
        subprocess.run(["git", "merge-base", "--is-ancestor", REQUIRED_ANCESTOR_COMMIT, "HEAD"]).returncode == 0
    )
    if not _is_descendant or _status:
        raise RuntimeError(
            f"本番の実行条件を満たさないため、学習の前に停止する(結果の情報は何も得ていない)。HEAD {_head[:7]}、"
            f"{REQUIRED_ANCESTOR_COMMIT} の子孫か: {_is_descendant}、追跡しているファイルの未コミットの変更: {_status or 'なし'}。"
            "リポジトリを最新の main に更新し、変更をなくして再実行すること。"
        )
    print(
        f"本番の実行条件: HEAD {_head[:7]}、追跡しているファイルに未コミットの変更がない"
        f"、必要な祖先のコミット {REQUIRED_ANCESTOR_COMMIT}: OK(参考: 追跡外のファイル {_untracked or 'なし'})"
    )
print(
    f"SMOKE_TEST={SMOKE_TEST}、学習の精度 {'FP16 の autocast + 動的損失スケーリング' if USE_FP16_AUTOCAST else 'FP32'}"
    f"(埋め込みの正規化・損失・評価は FP32)、決定的な演算の強制: {torch.are_deterministic_algorithms_enabled()}"
)

読み込み済みのパッケージの版の検査(requirements.txt): 食い違いなし。検査した配布物 3 個(packaging==26.3, python-dateutil==2.9.0.post0, six==1.17.0)、未読み込みのため検査しなかった配布物 51 個、__version__ がないため比べなかった配布物 1 個(typing-extensions)、表記の違いで誤検出するため比べなかった配布物 0 個(なし)


実行環境 / Execution environment
  Python                                 : 3.12.11
  OS / platform                          : macOS-26.6.2-arm64-arm-64bit
  torch                                  : 2.13.0
  torch のビルド時の CUDA                : 該当なし
  cuDNN                                  : 該当なし
  デバイス / device                      : mps
  GPU 名 / GPU name                      : 該当なし
  compute capability                     : 該当なし
  GPU の総メモリ (GiB)                   : 該当なし
  コミット / git commit                  : 5c5bde465c70bbb3ab7a3e7a2c4d7930cfe1fa0d
  未コミットの変更 / uncommitted changes : あり
  実行日時 (UTC)                         : 2026-10-14T01:41:55+00:00
SMOKE_TEST=True、学習の精度 FP32(埋め込みの正規化・損失・評価は FP32)、決定的な演算の強制: False


In [2]:
import functools
import gc
import hashlib
import itertools
import json
import math
import resource
import subprocess
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.data.inverse_cloze import (
    build_cloze_index,
    build_sentence_tables,
    choose_cloze_sentence,
    extract_span,
    remove_span,
    sample_cloze_pair_batches,
    sentence_boundaries,
    sentence_structure,
    splitmix64_uniform,
)
from src.data.reranking import encode_articles_with_cache
from src.data.retrieval import (
    build_retrieval_set,
    concatenate_articles,
    compute_article_sampling_weights,
    find_articles_starting_at_or_after,
    sample_pair_schedule,
    select_evenly,
    split_articles,
)
from src.data.text import locate_wikipedia_article_spans, split_train_val_text
from src.data.tokenizer import load_bpe_id_tokenizer_from_hub, try_decode_byte_level_symbol
from src.layers.feedforward import SwiGLUFeedForwardNetwork
from src.layers.normalization import RMSNorm
from src.layers.positional_encoding import RotaryPositionEmbedding
from src.models.cross_encoder import CrossEncoder, build_cross_encoder_inputs
from src.models.gpt import GPTLanguageModel
from src.models.text_embedding import TextEmbeddingModel, compute_hidden_states, encode_tokens
from src.models.variable_length import (
    VariableLengthCrossEncoder,
    build_variable_cross_encoder_inputs,
    encode_tokens_with_lengths,
    pooled_with_lengths,
)
from src.retrieval.cost_curve import cluster_bootstrap_weights
from src.retrieval.evaluation import evaluate_embeddings, expected_random_recall
from src.retrieval.exact_search import exact_search, remove_excluded_and_truncate
from src.retrieval.hnsw import HierarchicalNavigableSmallWorld
from src.retrieval.passage_level import (
    rank_article_label,
    rank_single_positive,
    rerank_single_positive,
    score_candidates_variable,
)
from src.retrieval.reranking import rank_reranked_candidates
from src.training.contrastive_text import (
    compute_pair_losses,
    info_nce_loss,
    train_text_embedding,
)
from src.training.contrastive_text_variable import (
    VariableBatches,
    compute_pair_losses_from_batches,
    masked_info_nce_loss,
    train_text_embedding_from_batches,
)
from src.training.reranking import softmax_cross_entropy_over_candidates
from src.training.reranking_variable import (
    VariableTables,
    candidate_scores,
    compute_reranking_losses_variable,
    sample_cloze_reranking_schedule,
    train_reranker_variable,
    usable_cloze_queries,
)
from src.utils.reporting import dumps_compact_json
from src.utils.statistics import fit_power_law_exponent, paired_cluster_bootstrap_ratio_of_sums

ROOT = Path.cwd()
WIKIPEDIA_CACHE_DIR = ROOT / ".cache" / "wikipedia_en"  # locate_wikipedia_article_spans() の引数(記事の境界は metadata.json から得るので、Wikipedia API には接続しない)
ENCODED_CACHE_DIR = ROOT / ".cache" / "026_encoded"  # 符号化した記事のトークン列(入力から決定的に再生成できる。同一セッション内の高速化のみが目的)
CORPUS_REPO_ID = "kojikojiprg/ai-theories-corpus-en"  # 9,826 記事。使うのは先頭の 356 記事を除く 9,470 記事
REFERENCE_CORPUS_REPO_ID = "kojikojiprg/ai-theories-corpus-en-pretraining"  # 008 の事前学習と英語のトークナイザの学習に使ったコーパス
TOKENIZER_REPO_ID = "kojikojiprg/ai-theories-tokenizer-en"
REFERENCE_MODEL_REPO_ID = "kojikojiprg/ai-theories-small-gpt-en"  # 008 の学習済みモデル(R1・R2・cross-encoder の起点)
EMBEDDING_REPO_ID = "kojikojiprg/ai-theories-text-embedding-en"  # 024 の学習済みモデル(R1 の再現の確認だけに使う)/ R2 のアップロード先(ブランチ)
EMBEDDING_UPLOAD_BRANCH = "inverse-cloze-task"
CROSS_ENCODER_REPO_ID = "kojikojiprg/ai-theories-cross-encoder-en"
MANIFEST_PATH = ROOT / "src" / "data" / "wikipedia_manifests" / "en_009_scaling.json"
REFERENCE_MANIFEST_PATH = ROOT / "src" / "data" / "wikipedia_manifests" / "en_006_pretraining.json"
RUN_TAG = "[スモークテスト] " if SMOKE_TEST else ""
PLOT_TAG = "[smoke test] " if SMOKE_TEST else ""


def sync_device() -> None:
    # 時間計測の直前・直後に、非同期に実行される GPU の処理の完了を待つ。
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def empty_device_cache() -> None:
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()


def timed_call(function) -> float:
    sync_device()
    start = time.time()
    function()
    sync_device()
    return time.time() - start


def rounded(values, digits: int = 4):
    # 印字用: 入れ子のリスト・配列を丸めたリストにする
    return np.round(np.asarray(values, dtype=np.float64), digits).tolist()


precondition_status: dict[str, bool] = {}  # 前提条件の成否(6.1 節で宣言、各節で記録)

### 5.2 スケールの設定(`SMOKE_TEST`の配線)

水準の定義をこの 1 箇所に集約する。

**縮小の規則**: 動作確認(`SMOKE_TEST = True`)は、本番と **同じモデル・同じデータの分割・同じ条件・同じ実行計画の構造** で、次の量だけを縮小する。コードの経路が最後まで通ることの確認だけが目的で、数値には意味を持たせない
(学習は進まず、前提条件は成立しない見込みである)。

- **第 1 段の学習ステップ数 $T_1$**: 本番 1024、動作確認 16。
- **cross-encoder の学習ステップ数の候補 $T_2$**: 本番 $(2048, 4096, 8192)$、動作確認 $(16, 32, 64)$。どちらも隣どうしの比が 2 の等比という構造を保つ。
- **シード数**: 本番は $(5, 3, 2)$(削る前、中、低)、動作確認は $(4, 3, 2)$。「削る前 > 中 > 低 $\ge 2$」の順序関係を保つ。
- **コーパスに入れる学習用の記事の数**: 本番はすべて(9,070 記事)、動作確認は 80 記事(検証用 100 記事・評価用 300 記事は縮小しない)。コーパス全体の索引の大きさと、それを使う埋め込みの計算・探索の時間が縮む。
- **観察 D の索引の件数**: 本番 $(16384, 8192)$、動作確認 $(2048, 1024)$(隣どうしの比が 2 の等比)。観察 D の query の数は、本番 600、動作確認 60。
- **cross-encoder の評価に使う query の数**(検証用・評価用それぞれ): 本番はすべて(検証用 約 1,000 個、評価用 約 3,200 個)、動作確認は等間隔に選んだ 64 個(候補 50 件との組の順伝播の回数が query の数に比例して多いため)。
- **ブートストラップの反復回数**: 本番 10,000 回、動作確認 1,000 回。

**縮小しないもの**: モデルの構成、データの分割(024 と同じ)、バッチサイズ、query と passage の長さ、文の分割の規則、条件の対応表、学習率の較正の格子と選択の規則、学習率のスケジュールの形、
cross-encoder の負例の数と候補の数、実行計画の表の構造と優先順位、前提条件と判定の閾値、スケーリングの計測点。第 1 段の評価の query(評価用の約 3,200 個)は縮小しない。

**テスト専用の上書き**: 環境変数`AI_THEORIES_FORCE_STAGE`が設定されているときのみ、6.4 節で見積もりによる計画の選択の代わりにその段階を使う(動作確認で下位の計画の経路を確かめるため)。本番では受け付けず、このセルで停止する。

**実効水準の照合**: このセルで印字した水準が、実際の学習で使われた値と一致することを 6.12 節のアサーションで確かめる(`SMOKE_TEST`の配線漏れの検出)。

In [3]:
# --- 全水準で共通の定数(本番実行前に宣言し、SMOKE_TEST で変えない) ---
# モデル(008 と同じ小型 GPT。構成は Hub の config.json から読む)
EMBEDDING_DIMENSION = 256
MATRYOSHKA_DIMENSIONS = (16, 32, 64, 128, 256)  # 入れ子の次元の集合(024 の C5 と同じ。R1 と R2 で揃える)
# データ(5.3 節)
QUERY_LENGTH, PASSAGE_LENGTH = 32, 128  # L_q(上限)、L_p
MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS = 4, 96  # 抜く文の長さの範囲(文脈は 32 トークン以上残る)
KEEP_PROBABILITY = 0.1  # 抜いた文を文脈に残す確率 r(原論文は 0.1)
MAX_PASSAGES_PER_ARTICLE = 12  # 記事ごとの索引の passage 数の上限(024・025 と同じ)
MAX_QUERIES_PER_ARTICLE = 12
NUM_VALIDATION_ARTICLES, NUM_EVALUATION_ARTICLES = 100, 300
SPLIT_SEED, EVALUATION_SET_SEED, VALIDATION_SET_SEED = 24_001, 24_002, 24_003  # 024 と同じ分割と集合
SPLIT_DIGEST_024, EVALUATION_DIGEST_024, VALIDATION_DIGEST_024 = "716cfe33d05a2615", "da3e16b96714c0a4", "acc9ec62c2a091b0"  # 024 の本番の出力に記録された値
REFERENCE_RECALL_024 = 0.6477  # 024 の C5 のシード 0(Hub の重み)の、評価用の索引での Recall@10(全次元。モデルカード)
CLOZE_SEED = 26_001  # passage 単位の評価の索引で、抜く文を選ぶ乱数のシード
REFERENCE_VALIDATION_RATIO = 0.05  # 008 の事前学習と英語のトークナイザの学習は、参照コーパスの末尾 5% を使っていない
LIST_LIKE_TITLE_PREFIXES = ("List of", "Lists of", "Timeline of", "Outline of", "Index of", "Glossary of", "Bibliography of", "Comparison of", "Discography", "Filmography", "Deaths in", "Births in")
COLAB_RAM_GIB = 12.7  # Colab 無料枠の RAM(メモリ使用量の見込みの判定に使う)
RAM_FRACTION_LIMIT = 0.75  # ピークのメモリ使用量が RAM のこの割合以下であること
EMBED_BATCH = 512  # 埋め込みの計算の 1 回の順伝播の系列の数(結果には影響しない)
SEPARATOR_BYTE, TERMINAL_BYTE = 0x1F, 0x1E  # cross-encoder の区切りと終端に使う単一バイトのトークン(制御文字。語彙を増やさない)
# 第 1 段の学習(024 の条件 C5 に準じる)
BATCH_SIZE = 64  # N(1 バッチの組の数)
PAIRS_PER_ARTICLE = 2  # 1 つの記事から同じバッチに入れる組の数(窓は互いに重ならない)
ARTICLES_PER_BATCH = BATCH_SIZE // PAIRS_PER_ARTICLE  # 1 バッチの記事の数(32)
TRAINING_SAFETY_FACTOR = 1.1  # 見積もりで、学習の時間に掛ける安全係数
TEMPERATURE = 0.05  # tau(固定)
WEIGHT_DECAY = 0.1
WARMUP_RATIO = 0.1
MIN_LEARNING_RATE_RATIO = 0.01
GRADIENT_CLIP_THRESHOLD = 1.0
INIT_LOSS_SCALE = 2.0**16
LOSS_SCALE_GROWTH_INTERVAL = 2000
FIRST_STAGE_CONDITIONS = ("R1", "R2")  # R1: 記事単位の定義、R2: passage 単位の定義(Inverse Cloze Task)
FIRST_STAGE_LEARNING_RATE_CENTER = {"R1": 4.8e-4, "R2": 4.8e-4}  # 024 の C5 の較正で選ばれた値(T = 1024)を、どちらの条件の格子の中心にも使う(組の作り方が違うので、最適の保証はない)
LEARNING_RATE_GRID_MULTIPLIERS = (0.25, 1.0, 4.0)  # 格子 = 中心 x {1/4, 1, 4}(公比 4)
LEARNING_RATE_GRID_RATIO = 4.0
R1_RULE_LEARNING_RATE = 4.8e-4  # 削る段階 4 以降: R1 の学習率は較正せず、024 の C5(同じ構成・同じ T_1。組の作り方は違う)の較正で選ばれた値にする
# cross-encoder(025 と同じ構成)
BATCH_QUERIES, NUM_NEGATIVES = 8, 7  # B、n(1 ステップ = B 個の query x (正例 1 + 負例 n))
RERANK_CANDIDATES = 50  # K(並べ替える候補の数 = 採掘の深さ)
CROSS_ENCODER_LEARNING_RATE_CENTER = 2.4e-4  # 025 の D1 のパイロットで、最後の区間の訓練損失が最も低かった学習率
CROSS_ENCODER_RULE_LEARNING_RATE = CROSS_ENCODER_LEARNING_RATE_CENTER  # 削る段階 5 以降: cross-encoder の学習率は較正せず、中心にする
HEAD_INIT_STD = 0.02
SCORE_BATCH = 512  # 並べ替えの推論の 1 回の順伝播の系列の数(結果には影響しない)
# 前提条件と判定(6.1 節)
FINAL_LOSS_FRACTION = 0.05  # 「最後の区間」= 最後の 5% のステップ
P1_LOSS_RATIO = 0.9  # 第 1 段の P1(a): 最後の区間の訓練損失 <= 同じ組での学習前のモデルの損失 x この値
P1_VALIDATION_GAIN = 0.04  # 第 1 段の P1(b): 自分の定義の検証用の Recall@10 >= 学習前の 008 の値 + この値
P1_SIGMA_MULTIPLIER = 2.0  # cross-encoder の P1: 差の平均が標準誤差(標準偏差)のこの倍数以上
P2_FLOOR, P2_CEILING = 0.02, 0.95  # 実験 A・B の P2: 基準の条件の Recall@10 が [床, 天井] の内側
P2_ROOM = 0.05  # 実験 C の P2: R2 だけの Recall@10 <= coverage - この値
P3_MIN_COVERAGE, P3_MIN_HEADROOM = 0.35, 0.10  # 実験 C の P3: coverage の下限と、並べ替えの余地(coverage - R2 だけの Recall@10)の下限
SIGMA_MULTIPLIER = 2.0  # 判定の閾値は対比量の標準偏差の 2 倍
SELECTION_SIGMA_MULTIPLIER = 1.0  # 較正の選択: 最良との差がこの倍数の標準偏差以内の候補の中では、学習の成立の余裕が最も大きいものを選ぶ
# 乱数シード(学習のシード s: 第 1 段の組(記事の並びを含む。R1 と R2 で共通)26100 + s、cross-encoder の組 26500 + s、ヘッドの初期化 26200 + s)
CLOZE_BATCH_SEED_BASE, INIT_SEED_BASE, CROSS_ENCODER_SCHEDULE_SEED_BASE = 26_100, 26_200, 26_500
CALIBRATION_SEED_INDEX = 90  # 学習率の較正とパイロット専用のシード(実験のシード 0〜 と共有しない)
TIMING_SEED_INDEX = 91  # スケーリングの計測専用のシード
BOOTSTRAP_SEED = 26_600
SESSION_BUDGET_SECONDS = 120 * 60  # 1 セッションの予算(T4 で 120 分)
REPORTING_MARGIN_SECONDS = 300.0  # 判定・ブートストラップ・図・アップロードの準備の固定の余裕
RUN_OVERHEAD_SECONDS = 2.0  # 学習 1 回あたりのモデルの構築などの固定費の余裕
UPLOAD_SEED = 0  # Hub にアップロードする学習のシード(結果を見て選ばない)
UNIFORM_LOSS_CROSS_ENCODER = math.log(1 + NUM_NEGATIVES)  # cross-encoder の一様な予測の損失 ln(1 + n)
# 観察 D(2 段階の検索の費用)
HNSW_MAX_CONNECTIONS, HNSW_CONSTRUCTION_WIDTH = 5, 100  # M、efConstruction(025 と同じ)
OBSERVATION_RERANK_KS = (10, 20, 50, 100)  # 並べ替える候補の数 K
OBSERVATION_SEARCH_WIDTHS = (101, 127, 160, 202, 254)  # HNSW の探索の幅 ef の格子(公比 2^(1/3)、最小は max(K) + 1)
OBSERVATION_TARGET_RECALL = 0.9  # 第 1 段の近似の目標(全探索の上位 100 件との一致率)
OBSERVATION_SEED = 26_900

# --- 水準(SMOKE_TEST で変わるもの) ---
LEVELS = {
    "smoke": {
        "FIRST_STAGE_STEPS": 16, "CROSS_ENCODER_STEP_CANDIDATES": (16, 32, 64), "SEEDS": (4, 3, 2), "CORPUS_TRAIN_ARTICLES": 80,
        "OBSERVATION_N_CANDIDATES": (2048, 1024), "OBSERVATION_QUERIES": 60, "RERANK_QUERIES": 64, "BOOTSTRAP_RESAMPLES": 1_000, "HNSW_TIMING_SIZES": (128, 256, 512),
    },
    "prod": {
        "FIRST_STAGE_STEPS": 1024, "CROSS_ENCODER_STEP_CANDIDATES": (2048, 4096, 8192), "SEEDS": (5, 3, 2), "CORPUS_TRAIN_ARTICLES": None,
        "OBSERVATION_N_CANDIDATES": (16384, 8192), "OBSERVATION_QUERIES": 600, "RERANK_QUERIES": None, "BOOTSTRAP_RESAMPLES": 10_000, "HNSW_TIMING_SIZES": (1024, 2048, 4096),
    },
}
CURRENT_LEVEL_NAME = "smoke" if SMOKE_TEST else "prod"
LEVEL_SETTINGS = LEVELS[CURRENT_LEVEL_NAME]
FIRST_STAGE_STEPS = LEVEL_SETTINGS["FIRST_STAGE_STEPS"]  # T_1(第 1 段の学習ステップ数。全条件・全段階で共通)
CROSS_ENCODER_STEP_CANDIDATES = LEVEL_SETTINGS["CROSS_ENCODER_STEP_CANDIDATES"]  # T_2 の候補(パイロットで選ぶ)
CORPUS_TRAIN_ARTICLES = LEVEL_SETTINGS["CORPUS_TRAIN_ARTICLES"]
RERANK_QUERIES = LEVEL_SETTINGS["RERANK_QUERIES"]  # cross-encoder の評価(検証用・評価用)に使う query の数(None はすべて)
BOOTSTRAP_RESAMPLES = LEVEL_SETTINGS["BOOTSTRAP_RESAMPLES"]

# 削る段階(6.1 節)。各段階は前の段階に 1 つだけ変更を加える(累積)
STAGE_CHANGES = (
    "(削らない)",
    "観察 D の索引の件数を 1 段下げる",
    "cross-encoder のシード数を「中」にする",
    "第 1 段(R1・R2)のシード数を「中」にする",
    "R1 の学習率の較正をやめ、024 の値にする",
    "cross-encoder の学習率の較正をやめ、中心の値にする",
    "cross-encoder のシード数を「低」にする",
    "第 1 段(R1・R2)のシード数を「低」にする",
)
NUM_STAGES = len(STAGE_CHANGES)


def stage_settings(stage: int, level: str = CURRENT_LEVEL_NAME) -> dict:
    # 段階 stage の実効の設定(累積)
    full, mid, low = LEVELS[level]["SEEDS"]
    return {
        "observation_n": LEVELS[level]["OBSERVATION_N_CANDIDATES"][1 if stage >= 1 else 0],
        "cross_encoder_seeds": low if stage >= 6 else mid if stage >= 2 else full,
        "first_stage_seeds": low if stage >= 7 else mid if stage >= 3 else full,
        "calibrate_r1": stage < 4,
        "calibrate_ce": stage < 5,
    }


FORCED_STAGE_VALUE = os.environ.get("AI_THEORIES_FORCE_STAGE")
if FORCED_STAGE_VALUE is not None and not SMOKE_TEST:
    raise RuntimeError(
        f"本番(SMOKE_TEST=False)では段階の強制(AI_THEORIES_FORCE_STAGE={FORCED_STAGE_VALUE!r})を受け付けない。環境変数を削除して再実行すること。"
    )
FORCED_STAGE = None if FORCED_STAGE_VALUE is None else int(FORCED_STAGE_VALUE)
assert FORCED_STAGE is None or 0 <= FORCED_STAGE < NUM_STAGES, FORCED_STAGE


def warmup_steps_for(num_steps: int) -> int:
    return max(1, round(WARMUP_RATIO * num_steps))


def final_loss_window(num_steps: int) -> int:
    return max(1, round(FINAL_LOSS_FRACTION * num_steps))


def first_stage_grid(condition: str) -> tuple[float, ...]:
    return tuple(FIRST_STAGE_LEARNING_RATE_CENTER[condition] * m for m in LEARNING_RATE_GRID_MULTIPLIERS)


def cross_encoder_grid() -> tuple[float, ...]:
    return tuple(CROSS_ENCODER_LEARNING_RATE_CENTER * m for m in LEARNING_RATE_GRID_MULTIPLIERS)


# --- 縮小規則と水準の構造の確認 ---
assert set(LEVELS["smoke"]) == set(LEVELS["prod"])
for _name, _settings in LEVELS.items():
    _candidates = _settings["CROSS_ENCODER_STEP_CANDIDATES"]
    assert len(_candidates) == 3 and all(b == 2 * a for a, b in itertools.pairwise(_candidates))  # 等比(公比 2)
    _full, _mid, _low = _settings["SEEDS"]
    assert _full > _mid > _low >= 2  # 「削る前 > 中 > 低 >= 2」
    assert len(_settings["OBSERVATION_N_CANDIDATES"]) == 2 and _settings["OBSERVATION_N_CANDIDATES"][0] == 2 * _settings["OBSERVATION_N_CANDIDATES"][1]
    assert len(_settings["HNSW_TIMING_SIZES"]) == 3 and all(b == 2 * a for a, b in itertools.pairwise(_settings["HNSW_TIMING_SIZES"]))
    for _t in (_settings["FIRST_STAGE_STEPS"], *_candidates):
        assert warmup_steps_for(_t) < _t
assert LEVELS["smoke"]["BOOTSTRAP_RESAMPLES"] < LEVELS["prod"]["BOOTSTRAP_RESAMPLES"] and LEVELS["smoke"]["FIRST_STAGE_STEPS"] < LEVELS["prod"]["FIRST_STAGE_STEPS"]
assert LEVELS["smoke"]["OBSERVATION_QUERIES"] < LEVELS["prod"]["OBSERVATION_QUERIES"] and LEVELS["prod"]["CORPUS_TRAIN_ARTICLES"] is None and LEVELS["prod"]["RERANK_QUERIES"] is None
for _level in LEVELS:  # 段階ごとの変更が累積し、各段階で 1 つだけ変わる(非増加)。cross-encoder のシード数 <= 第 1 段のシード数
    for _k in range(1, NUM_STAGES):
        _before, _after = stage_settings(_k - 1, _level), stage_settings(_k, _level)
        _changed = [key for key in _before if _before[key] != _after[key]]
        assert len(_changed) == 1, (_level, _k, _changed)
        assert all(int(_after[key]) <= int(_before[key]) for key in _before)  # 数値は減る、真偽は True -> False
        assert _after["cross_encoder_seeds"] <= _after["first_stage_seeds"]
for _k in range(1, NUM_STAGES):  # 本番と動作確認で、各段階で何が変わるかが同じ
    assert [key for key in stage_settings(_k, "smoke") if stage_settings(_k, "smoke")[key] != stage_settings(_k - 1, "smoke")[key]] == [
        key for key in stage_settings(_k, "prod") if stage_settings(_k, "prod")[key] != stage_settings(_k - 1, "prod")[key]
    ]
assert all(math.isclose(b / a, LEARNING_RATE_GRID_RATIO) for a, b in itertools.pairwise(LEARNING_RATE_GRID_MULTIPLIERS))
assert R1_RULE_LEARNING_RATE == FIRST_STAGE_LEARNING_RATE_CENTER["R1"] and CROSS_ENCODER_RULE_LEARNING_RATE == CROSS_ENCODER_LEARNING_RATE_CENTER
assert BATCH_SIZE % PAIRS_PER_ARTICLE == 0 and PAIRS_PER_ARTICLE == 2 and TRAINING_SAFETY_FACTOR >= 1.0
assert QUERY_LENGTH <= PASSAGE_LENGTH and QUERY_LENGTH + 1 + PASSAGE_LENGTH + 1 <= 256  # cross-encoder の列は 008 の学習長(256)以内
assert MIN_QUERY_TOKENS <= QUERY_LENGTH and PASSAGE_LENGTH - MAX_REMOVED_TOKENS >= QUERY_LENGTH  # 文脈は query の長さ以上残る
assert max(OBSERVATION_RERANK_KS) + 1 <= OBSERVATION_SEARCH_WIDTHS[0] and 10 in OBSERVATION_RERANK_KS and RERANK_CANDIDATES in OBSERVATION_RERANK_KS
assert NUM_NEGATIVES + 1 <= RERANK_CANDIDATES
assert MATRYOSHKA_DIMENSIONS[-1] == EMBEDDING_DIMENSION and all(math.isclose(b / a, 2.0) for a, b in itertools.pairwise(MATRYOSHKA_DIMENSIONS))

print(f"水準 {CURRENT_LEVEL_NAME!r}: {json.dumps({k: list(v) if isinstance(v, tuple) else v for k, v in LEVEL_SETTINGS.items()})}")
print(
    f"データ: query {QUERY_LENGTH} トークン(上限)、passage {PASSAGE_LENGTH} トークン(窓)、記事ごとの passage 数の上限 {MAX_PASSAGES_PER_ARTICLE}、検証用 {NUM_VALIDATION_ARTICLES} 記事・評価用 {NUM_EVALUATION_ARTICLES} 記事(024 と同じ分割)。"
    f"Inverse Cloze Task: 抜く文の長さ {MIN_QUERY_TOKENS}〜{MAX_REMOVED_TOKENS} トークン、抜いた文を文脈に残す確率 {KEEP_PROBABILITY}"
)
print(
    f"第 1 段の学習: バッチ {BATCH_SIZE} 組({ARTICLES_PER_BATCH} 記事 x {PAIRS_PER_ARTICLE} 組、in-batch negatives)、温度 {TEMPERATURE}(固定)、Matryoshka Representation Learning {MATRYOSHKA_DIMENSIONS}、AdamW(重み減衰 {WEIGHT_DECAY}、foreach)、"
    f"warmup {WARMUP_RATIO:.0%} + cosine(下限 x{MIN_LEARNING_RATE_RATIO})、gradient clipping {GRADIENT_CLIP_THRESHOLD}、T_1 = {FIRST_STAGE_STEPS}"
)
print("第 1 段の学習率の格子: " + "、".join(f"{c} {tuple(float(f'{x:.3g}') for x in first_stage_grid(c))}" for c in FIRST_STAGE_CONDITIONS))
print(
    f"cross-encoder: 1 ステップ {BATCH_QUERIES} query x (正例 1 + 負例 {NUM_NEGATIVES})、並べ替える候補 K = {RERANK_CANDIDATES}、T_2 の候補 {CROSS_ENCODER_STEP_CANDIDATES}、"
    f"学習率の格子 {tuple(float(f'{x:.3g}') for x in cross_encoder_grid())}"
)
print(f"削る段階({CURRENT_LEVEL_NAME!r}): " + "、".join(f"段階 {k}({STAGE_CHANGES[k]}): {json.dumps(stage_settings(k))}" for k in range(NUM_STAGES)))
if FORCED_STAGE is not None:
    print(f"*** テスト専用の上書き: AI_THEORIES_FORCE_STAGE = {FORCED_STAGE}(6.4 節で見積もりの代わりにこの段階を使う) ***")

水準 'smoke': {"FIRST_STAGE_STEPS": 16, "CROSS_ENCODER_STEP_CANDIDATES": [16, 32, 64], "SEEDS": [4, 3, 2], "CORPUS_TRAIN_ARTICLES": 80, "OBSERVATION_N_CANDIDATES": [2048, 1024], "OBSERVATION_QUERIES": 60, "RERANK_QUERIES": 64, "BOOTSTRAP_RESAMPLES": 1000, "HNSW_TIMING_SIZES": [128, 256, 512]}
データ: query 32 トークン(上限)、passage 128 トークン(窓)、記事ごとの passage 数の上限 12、検証用 100 記事・評価用 300 記事(024 と同じ分割)。Inverse Cloze Task: 抜く文の長さ 4〜96 トークン、抜いた文を文脈に残す確率 0.1
第 1 段の学習: バッチ 64 組(32 記事 x 2 組、in-batch negatives)、温度 0.05(固定)、Matryoshka Representation Learning (16, 32, 64, 128, 256)、AdamW(重み減衰 0.1、foreach)、warmup 10% + cosine(下限 x0.01)、gradient clipping 1.0、T_1 = 16
第 1 段の学習率の格子: R1 (0.00012, 0.00048, 0.00192)、R2 (0.00012, 0.00048, 0.00192)
cross-encoder: 1 ステップ 8 query x (正例 1 + 負例 7)、並べ替える候補 K = 50、T_2 の候補 (16, 32, 64)、学習率の格子 (6e-05, 0.00024, 0.00096)
削る段階('smoke'): 段階 0((削らない)): {"observation_n": 2048, "cross_encoder_seeds": 4, "first_stage_seeds": 4, "calibrate_r1": true, "calibrate_ce": true}、段階 1(観察 D の索

### 5.3 データ: コーパス・記事の分割・Inverse Cloze Task の索引

**コーパスとトークナイザ**: 英語 Wikipedia のコーパス(`kojikojiprg/ai-theories-corpus-en`、9,826 記事)と、008 の BPE(Byte Pair Encoding)トークナイザ(語彙サイズ 8192)を使う。
記事の境界は`metadata.json`の`article_offsets`から得る。コーパスの先頭の 356 記事は 008 の事前学習とトークナイザの学習に使ったコーパスと同一なので使わず、使うのは、そのどちらも一度も見ていない 9,470 記事である。
記事の分割([024](./024_text_embedding_and_retriever.ipynb) と同じ。学習用 9,070 記事・検証用 100 記事・評価用 300 記事)と、評価用・検証用の記事単位の索引・query は、024 の本番の値(ダイジェスト)と一致することを確かめる。

**コーパスの passage の索引**: 使う記事すべてを、重ならない長さ $L_p = 128$ の窓に分けて passage にする(記事ごとに最大 12 個を等間隔に選ぶ。024・025 と同じ手順)。
評価用の記事の部分は、024 の評価用の索引と一致する。**評価の索引**(3.5 節)は、このコーパスの passage から、抜く文を決定的に 1 つ選んで「文を除いた形」にしたもので、
対象でない passage(文が 2 つ未満、または長さ $4 \le \ell \le 96$ の文がない passage)は索引に入れない。除いた passage の数を印字する。索引の passage の数が $P$ である。
抜く文は、長さ 32 トークン以内に収まる文を優先して選ぶ(収まる文がない passage だけ長い文を選んで query の先頭を切り詰める)。切り詰めた query の割合を、評価の索引と学習の組のそれぞれで印字する。

| 部分 | 記事 | 評価での役割 |
|---|---|---|
| 学習用(9,070 記事) | 索引に入る(検索の対象の一部)。cross-encoder の学習の query もここから作る | 検索の対象のコーパスを大きくする(困難な負例の候補になる) |
| 検証用(100 記事) | 索引に入る。較正は検証用の記事の passage だけを索引にした小さい索引で行う | 学習率の較正と前提条件に使う。判定には使わない |
| 評価用(300 記事) | 索引に入る。query はこの部分の passage から作る | 判定と診断量に使う |

**同じ索引・同じ query、2 つのラベル**: 実験 A・B と較正は、同じ索引・同じ query で、passage 単位のラベルと記事単位のラベルを当てる。query は、記事単位のラベルの正解(同じ記事の他の passage)が索引に 1 つ以上ある query に限る(除いた数を印字する)。
024 と同じ形式の記事単位の評価(024 の評価用の索引と query。固定長)は、診断量として別に測る。

**文の分割**: 3.4 節の規則を、トークンごとの表で実装する(トークンのテキストは、バイトレベルの BPE のシンボルを UTF-8 にデコードして得る。デコードできないトークンは、空白でも句点でもないものとして扱う)。

**第 1 段の学習の組**(5.5 節): 1 つのバッチは、32 記事 × 各記事 2 組の 64 組である。各記事から重ならない 2 つの窓を引き、それぞれから抜く文を選び、文脈を作る(3.2 節)。
R2 は各窓の文脈をその窓の query の正例にし、R1 は同じ記事のもう一方の窓の文脈を正例にする(バッチの中で隣り合う 2 組の文脈を入れ替える)。記事の並び・窓・抜く文・文を残す組は R1 と R2 で同一である(6.12 節のアサーションで確かめる)。
記事は、窓を重ならずに 2 つ取れる長さ($2 L_p = 256$ トークン以上)の記事から、窓の数(最大 12 で頭打ち)に比例する確率で非復元抽出する。

**cross-encoder の学習の query**: 索引の passage のうち学習用の記事のもの(索引の形の文を除いた passage と、抜いた文)から作る。正例は、その query の passage である。

In [4]:
_t0_data = time.time()
DATA_SECONDS_BY_STAGE: dict[str, float] = {}
PEAK_MEMORY_GIB_BY_STAGE: dict[str, float] = {}


def peak_memory_gib() -> float:
    # このプロセスのこれまでのピークのメモリ使用量(macOS は単位がバイト、Linux は KiB)
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return peak / (1024**3 if sys.platform == "darwin" else 1024**2)


def total_memory_gib() -> float | None:
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
    except (ValueError, OSError, AttributeError):
        return None


def end_stage(name: str, started: float) -> None:
    DATA_SECONDS_BY_STAGE[name] = time.time() - started
    PEAK_MEMORY_GIB_BY_STAGE[name] = peak_memory_gib()


from huggingface_hub import hf_hub_download  # noqa: E402

# --- トークナイザ: 区切りと終端に使う単一バイトのトークン(語彙を増やさない)と、文の分割の表 ---
_stage = time.time()
tokenizer, _tokenizer_from_hub = load_bpe_id_tokenizer_from_hub(TOKENIZER_REPO_ID)
assert _tokenizer_from_hub, "トークナイザを Hugging Face Hub から取得できなかった"
assert tokenizer.vocab_size == 8192
_bpe = tokenizer.bpe_tokenizer
assert tokenizer.vocab_size == 256 + len(_bpe.merges) and len(_bpe.vocab) == tokenizer.vocab_size  # 特殊トークンがない
_separator_ids, _terminal_ids = tokenizer.encode(chr(SEPARATOR_BYTE)), tokenizer.encode(chr(TERMINAL_BYTE))
assert len(_separator_ids) == len(_terminal_ids) == 1 and _separator_ids != _terminal_ids
SEPARATOR_TOKEN_ID, TERMINAL_TOKEN_ID = int(_separator_ids[0]), int(_terminal_ids[0])
TOKEN_TEXTS = [try_decode_byte_level_symbol(tokenizer.id_to_symbol[i]) for i in range(tokenizer.vocab_size)]
SENTENCE_TABLES = build_sentence_tables(TOKEN_TEXTS)
NUM_UNDECODABLE_TOKENS = sum(t is None for t in TOKEN_TEXTS)
end_stage("トークナイザの取得", _stage)

# --- 参照コーパス(008 の事前学習と英語のトークナイザの学習に使ったコーパス)---
_stage = time.time()
_reference_text = Path(hf_hub_download(REFERENCE_CORPUS_REPO_ID, "corpus.txt", repo_type="dataset")).read_text(encoding="utf-8")
REFERENCE_CORPUS_CHARACTERS = len(_reference_text)
_reference_train_text, _ = split_train_val_text(_reference_text, REFERENCE_VALIDATION_RATIO)
REFERENCE_PRETRAINING_END = len(_reference_train_text)  # 008 とトークナイザが使った範囲の終わり(文字位置)
del _reference_train_text
_manifest_items = list(json.loads(MANIFEST_PATH.read_text(encoding="utf-8")).items())
_reference_items = list(json.loads(REFERENCE_MANIFEST_PATH.read_text(encoding="utf-8")).items())
NUM_REFERENCE_ARTICLES = len(_reference_items)
assert _manifest_items[:NUM_REFERENCE_ARTICLES] == _reference_items, "先頭の記事のタイトルとリビジョン ID が参照コーパスのマニフェストと一致しない"
end_stage("参照コーパスの取得", _stage)

# --- コーパス(Hub から取得する。Wikipedia API にはフォールバックしない)---
_stage = time.time()
_corpus_path = hf_hub_download(CORPUS_REPO_ID, "corpus.txt", repo_type="dataset")
_metadata_path = hf_hub_download(CORPUS_REPO_ID, "metadata.json", repo_type="dataset")
end_stage("コーパスのダウンロード", _stage)
_stage = time.time()
corpus_metadata = json.loads(Path(_metadata_path).read_text(encoding="utf-8"))
assert corpus_metadata["manifest"] == MANIFEST_PATH.name and corpus_metadata["manifest_article_count"] == len(_manifest_items)
assert "article_offsets" in corpus_metadata, "metadata.json に article_offsets がない"
assert os.path.getsize(_corpus_path) == corpus_metadata["raw_bytes"], "コーパスの取得が破損している"
corpus_text = Path(_corpus_path).read_text(encoding="utf-8")
assert corpus_text.startswith(_reference_text) and corpus_text[REFERENCE_CORPUS_CHARACTERS] == "\n", "先頭が参照コーパスと一致しない"
del _reference_text
end_stage("コーパスの読み込み", _stage)

_stage = time.time()
article_spans, ARTICLE_SPANS_SOURCE = locate_wikipedia_article_spans(
    corpus_text, "en", WIKIPEDIA_CACHE_DIR, MANIFEST_PATH, metadata_path=_metadata_path, start_position=0
)
assert ARTICLE_SPANS_SOURCE == "metadata", "article_offsets を metadata.json から取得できなかった"
NUM_ARTICLES = len(article_spans)
assert NUM_ARTICLES == len(_manifest_items)
end_stage("記事の境界の検証", _stage)

# --- 使う記事: 参照コーパスの外に始まる記事だけ ---
CANDIDATE_ARTICLES = find_articles_starting_at_or_after(article_spans, REFERENCE_CORPUS_CHARACTERS)
assert CANDIDATE_ARTICLES == list(range(NUM_REFERENCE_ARTICLES, NUM_ARTICLES)), "使う記事が先頭の記事を除く全記事と一致しない"
assert article_spans[NUM_REFERENCE_ARTICLES - 1]["end"] == REFERENCE_CORPUS_CHARACTERS, "先頭の記事の末尾が参照コーパスの末尾と一致しない"
assert article_spans[CANDIDATE_ARTICLES[0]]["start"] == REFERENCE_CORPUS_CHARACTERS + 1 > REFERENCE_PRETRAINING_END

# --- 記事ごとの符号化(キャッシュあり。キャッシュを読んだ場合は一部を符号化し直して一致を確かめる)---
_stage = time.time()
_candidate_characters = article_spans[-1]["end"] - article_spans[CANDIDATE_ARTICLES[0]]["start"]
ARTICLE_TOKENS, ENCODED_FROM_CACHE = encode_articles_with_cache(
    tokenizer, corpus_text, article_spans, CANDIDATE_ARTICLES, ENCODED_CACHE_DIR / "tokens.npz"
)
ARTICLE_TOKEN_COUNTS = np.array([0 if t is None else len(t) for t in ARTICLE_TOKENS])  # 使わない記事は 0
CANDIDATE_TOKENS = int(ARTICLE_TOKEN_COUNTS.sum())
assert all(ARTICLE_TOKENS[a] is None for a in range(NUM_REFERENCE_ARTICLES)) and all(ARTICLE_TOKENS[a] is not None for a in CANDIDATE_ARTICLES)
TOKEN_BYTE_LENGTHS = np.array([len(tokenizer.id_to_symbol[i]) for i in range(tokenizer.vocab_size)], dtype=np.int64)
for _article in CANDIDATE_ARTICLES:  # 可逆性とバイト数の整合
    _tokens, _span = ARTICLE_TOKENS[_article], article_spans[_article]
    assert int(_tokens.max()) < tokenizer.vocab_size and _tokens.dtype == np.int32
    assert int(TOKEN_BYTE_LENGTHS[_tokens].sum()) == len(corpus_text[_span["start"] : _span["end"]].encode("utf-8"))
_first = article_spans[CANDIDATE_ARTICLES[0]]
_sample = corpus_text[_first["start"] : _first["start"] + 50_000]
assert tokenizer.decode(tokenizer.encode(_sample)) == _sample, "トークナイザのラウンドトリップが一致しない"
assert tokenizer.decode(ARTICLE_TOKENS[CANDIDATE_ARTICLES[-1]].tolist()) == corpus_text[article_spans[-1]["start"] : article_spans[-1]["end"]]
# 区切りと終端のトークンがコーパスに一度も現れない(cross-encoder の入力の区切りと終端が、コーパスの内容と衝突しない)
SEPARATOR_OCCURRENCES = sum(int((ARTICLE_TOKENS[a] == SEPARATOR_TOKEN_ID).sum()) for a in CANDIDATE_ARTICLES)
TERMINAL_OCCURRENCES = sum(int((ARTICLE_TOKENS[a] == TERMINAL_TOKEN_ID).sum()) for a in CANDIDATE_ARTICLES)
assert SEPARATOR_OCCURRENCES == 0 and TERMINAL_OCCURRENCES == 0, "区切り・終端のトークンがコーパスに現れる"
end_stage("記事の符号化と検証", _stage)

_titles = list(json.loads(MANIFEST_PATH.read_text(encoding="utf-8")))
LIST_LIKE_ARTICLES = frozenset(a for a in CANDIDATE_ARTICLES if _titles[a].startswith(LIST_LIKE_TITLE_PREFIXES))
CORPUS_CHARACTERS = len(corpus_text)
del corpus_text, _sample
gc.collect()

# --- 記事を単位とする分割(024 と同じ。ダイジェストが 024 の本番の値と一致する)---
_stage = time.time()
MIN_TOKENS_FOR_RETRIEVAL = 2 * PASSAGE_LENGTH
SPLIT = split_articles(ARTICLE_TOKEN_COUNTS, MIN_TOKENS_FOR_RETRIEVAL, CANDIDATE_ARTICLES, NUM_VALIDATION_ARTICLES, NUM_EVALUATION_ARTICLES, SPLIT_SEED)
assert SPLIT.digest() == SPLIT_DIGEST_024, f"分割が 024 の本番の分割と一致しない: {SPLIT.digest()}"
_parts = [set(SPLIT.train.tolist()), set(SPLIT.validation.tolist()), set(SPLIT.evaluation.tolist())]
assert not (_parts[0] & _parts[1]) and not (_parts[0] & _parts[2]) and not (_parts[1] & _parts[2]), "分割が重なっている"
assert _parts[0] | _parts[1] | _parts[2] == set(CANDIDATE_ARTICLES) and sum(len(p) for p in _parts) == len(CANDIDATE_ARTICLES)
assert len(SPLIT.train) == 9070 and len(SPLIT.validation) == NUM_VALIDATION_ARTICLES and len(SPLIT.evaluation) == NUM_EVALUATION_ARTICLES
_used_starts = np.array([article_spans[a]["start"] for a in sorted(_parts[0] | _parts[1] | _parts[2])])
assert _used_starts.min() >= REFERENCE_CORPUS_CHARACTERS > REFERENCE_PRETRAINING_END, "008 の事前学習の範囲にかかる記事を使っている"
end_stage("分割", _stage)

# --- 記事単位の評価用・検証用の索引と query(024 と同じ。ダイジェストが 024 の本番の値と一致する)---
_stage = time.time()
EVALUATION_SET = build_retrieval_set(
    ARTICLE_TOKENS, SPLIT.evaluation, PASSAGE_LENGTH, QUERY_LENGTH, MAX_PASSAGES_PER_ARTICLE, MAX_QUERIES_PER_ARTICLE, EVALUATION_SET_SEED
)
VALIDATION_SET = build_retrieval_set(
    ARTICLE_TOKENS, SPLIT.validation, PASSAGE_LENGTH, QUERY_LENGTH, MAX_PASSAGES_PER_ARTICLE, MAX_QUERIES_PER_ARTICLE, VALIDATION_SET_SEED
)
assert EVALUATION_SET.digest() == EVALUATION_DIGEST_024, f"評価用の索引と query が 024 の本番の集合と一致しない: {EVALUATION_SET.digest()}"
assert VALIDATION_SET.digest() == VALIDATION_DIGEST_024, f"検証用の索引と query が 024 の本番の集合と一致しない: {VALIDATION_SET.digest()}"
ARTICLE_LEVEL_SETS = {"evaluation": EVALUATION_SET, "validation": VALIDATION_SET}
# --- コーパスの passage(使う記事すべて。動作確認では、学習用の記事を一部に絞る)---
_pairable_train = [a for a in SPLIT.train.tolist() if ARTICLE_TOKEN_COUNTS[a] >= MIN_TOKENS_FOR_RETRIEVAL]
CORPUS_ARTICLES = (
    list(CANDIDATE_ARTICLES)
    if CORPUS_TRAIN_ARTICLES is None
    else sorted(set(SPLIT.evaluation.tolist()) | set(SPLIT.validation.tolist()) | {_pairable_train[i] for i in select_evenly(len(_pairable_train), CORPUS_TRAIN_ARTICLES)})
)
CORPUS = build_retrieval_set(
    ARTICLE_TOKENS, CORPUS_ARTICLES, PASSAGE_LENGTH, QUERY_LENGTH, MAX_PASSAGES_PER_ARTICLE, MAX_QUERIES_PER_ARTICLE, EVALUATION_SET_SEED
)
_corpus_evaluation = np.nonzero(np.isin(CORPUS.passage_articles, SPLIT.evaluation))[0]
assert np.array_equal(CORPUS.passages[_corpus_evaluation], EVALUATION_SET.passages), "コーパスの評価用の記事の部分が 024 の評価用の索引と一致しない"
NUM_CORPUS_PASSAGES = CORPUS.num_passages
# --- passage 単位の評価の索引(3.5 節): 抜く文を決定的に選び、文を除いた形にする ---
CLOZE = build_cloze_index(
    CORPUS.passages, CORPUS.passage_articles, CORPUS.passage_windows, SENTENCE_TABLES, QUERY_LENGTH, MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS, CLOZE_SEED
)
NUM_INDEX = CLOZE.num_passages  # P
ENTRY_ARTICLES = CORPUS.passage_articles[CLOZE.passage_ids]  # 索引の各 passage の記事
_entries_per_article = np.bincount(ENTRY_ARTICLES)
NUM_MATES = _entries_per_article[ENTRY_ARTICLES] - 1  # 索引の中の、同じ記事の他の passage の数(記事単位のラベルの正解の数)
HAS_MATE = NUM_MATES >= 1
EVALUATION_INDEX_ENTRIES = np.nonzero(np.isin(ENTRY_ARTICLES, SPLIT.evaluation))[0]
VALIDATION_INDEX_ENTRIES = np.nonzero(np.isin(ENTRY_ARTICLES, SPLIT.validation))[0]  # 検証用の記事の passage(較正の索引)
TRAIN_ENTRIES = np.nonzero(np.isin(ENTRY_ARTICLES, SPLIT.train))[0]
assert len(EVALUATION_INDEX_ENTRIES) + len(VALIDATION_INDEX_ENTRIES) + len(TRAIN_ENTRIES) == NUM_INDEX
# 実験 A・B の評価の query: 2 つのラベルで同じ query を使うため、記事単位のラベルの正解(同じ記事の他の passage)が 1 つ以上ある query に限る
EVALUATION_ENTRIES = EVALUATION_INDEX_ENTRIES[HAS_MATE[EVALUATION_INDEX_ENTRIES]]
VALIDATION_ENTRIES = VALIDATION_INDEX_ENTRIES[HAS_MATE[VALIDATION_INDEX_ENTRIES]]
CONTEXT_TOKENS = torch.from_numpy(CLOZE.context_tokens)  # 索引に入れる形(右側を 0 でパディング)
CONTEXT_LENGTHS = CLOZE.context_lengths
CLOZE_QUERY_TOKENS = torch.from_numpy(CLOZE.query_tokens)
CLOZE_QUERY_LENGTHS = CLOZE.query_lengths
EVALUATION_QUERY_ARTICLES = ENTRY_ARTICLES[EVALUATION_ENTRIES]  # ブートストラップのクラスタ(実験 A・C)
RANDOM_RECALL_PASSAGE = 10 / NUM_INDEX  # passage 単位のラベルの、ランダムな順位の Recall@10 の期待値(正例は 1 つ)
RANDOM_RECALL_ARTICLE_LABEL = expected_random_recall(NUM_MATES[EVALUATION_ENTRIES], NUM_INDEX - 1, 10)  # 記事単位のラベル(正解は同じ記事の他の passage)
end_stage("コーパスと索引の構成", _stage)

# --- 学習用の記事(第 1 段の組の切り出し): 連結した 1 本のトークン列(int32)---
_stage = time.time()
TRAIN_ARTICLES = SPLIT.train
TRAIN_TOKEN_COUNTS = ARTICLE_TOKEN_COUNTS[TRAIN_ARTICLES]
SAMPLING_WEIGHTS = compute_article_sampling_weights(TRAIN_TOKEN_COUNTS, QUERY_LENGTH, PASSAGE_LENGTH, MAX_PASSAGES_PER_ARTICLE)
SAMPLING_WEIGHTS[TRAIN_TOKEN_COUNTS < PAIRS_PER_ARTICLE * PASSAGE_LENGTH] = 0.0  # 窓を重ならずに 2 つ取れない短い記事は選ばない
_flat, TRAIN_STARTS = concatenate_articles(ARTICLE_TOKENS, TRAIN_ARTICLES)
TRAIN_STREAM = _flat.astype(np.int32)
del _flat
for _article in range(NUM_ARTICLES):  # 記事ごとのトークンは、passage・連結した列に取り出したので解放する
    ARTICLE_TOKENS[_article] = None
gc.collect()
NUM_PAIRABLE_TRAIN_ARTICLES = int((SAMPLING_WEIGHTS > 0).sum())
assert NUM_PAIRABLE_TRAIN_ARTICLES >= ARTICLES_PER_BATCH
TRAIN_TOKENS_PAIRABLE = int(TRAIN_TOKEN_COUNTS[SAMPLING_WEIGHTS > 0].sum())
UNIFORM_LOSS = math.log(BATCH_SIZE)  # ln(N): 類似度が一様なときの in-batch negatives の InfoNCE 損失
end_stage("学習用の記事の連結", _stage)
DATA_SECONDS = time.time() - _t0_data
DATA_PEAK_MEMORY_GIB = peak_memory_gib()
_total_memory = total_memory_gib()
_memory_limit = RAM_FRACTION_LIMIT * min(COLAB_RAM_GIB, _total_memory if _total_memory else COLAB_RAM_GIB)
assert DATA_PEAK_MEMORY_GIB <= _memory_limit, f"データの準備のピークのメモリ {DATA_PEAK_MEMORY_GIB:.1f} GiB が Colab の RAM に収まる見込みを超える({_memory_limit:.1f} GiB)"

print(
    f"コーパス: {CORPUS_CHARACTERS:,} 文字(取得元 Hugging Face Hub)、記事 {NUM_ARTICLES:,} 件。使う記事 {len(CANDIDATE_ARTICLES):,} 件({CANDIDATE_TOKENS:,} トークン)、"
    f"リスト系の記事の割合 {len(LIST_LIKE_ARTICLES) / len(CANDIDATE_ARTICLES):.1%}。符号化のキャッシュ: {'読み込み' if ENCODED_FROM_CACHE else '新規に作成'}"
)
print(
    f"トークナイザ: 語彙サイズ {tokenizer.vocab_size}(特殊トークンなし)、UTF-8 として不正なトークン {NUM_UNDECODABLE_TOKENS} 個。文の分割の表: 空白で始まる {int(SENTENCE_TABLES.starts_with_whitespace.sum())}、"
    f"改行で始まる {int(SENTENCE_TABLES.starts_with_newline.sum())}、文の終わりで終わる {int(SENTENCE_TABLES.ends_sentence.sum())}、閉じ記号だけ {int(SENTENCE_TABLES.closer_only.sum())} トークン。"
    f"区切り: 単一バイト 0x1F(ID {SEPARATOR_TOKEN_ID})、終端: 0x1E(ID {TERMINAL_TOKEN_ID})。コーパスでの出現回数は 区切り {SEPARATOR_OCCURRENCES}・終端 {TERMINAL_OCCURRENCES}"
)
print(
    f"分割(ダイジェスト {SPLIT.digest()}、024 の本番の値と一致): 学習用 {len(SPLIT.train):,} 記事(うち組を作れる記事 {NUM_PAIRABLE_TRAIN_ARTICLES:,} 件・{TRAIN_TOKENS_PAIRABLE:,} トークン)、"
    f"検証用 {len(SPLIT.validation)} 記事、評価用 {len(SPLIT.evaluation)} 記事。使う記事の開始位置の最小 {int(_used_starts.min()):,} は参照コーパスの全体({REFERENCE_CORPUS_CHARACTERS:,} 文字)の外"
)
for _name, _set in ARTICLE_LEVEL_SETS.items():
    print(f"  記事単位の{_name}: 索引 {_set.num_passages} passage・query {_set.num_queries} 個、ダイジェスト {_set.digest()}(024 の本番の値と一致)")
print(
    f"コーパス: passage {NUM_CORPUS_PASSAGES:,} 個(記事 {len(set(CORPUS.passage_articles.tolist())):,} 本、動作確認では学習用の記事を絞る: {CORPUS_TRAIN_ARTICLES})。"
    f"passage 単位の評価の索引(Inverse Cloze Task の対象): {NUM_INDEX:,} 個。除いた passage {CLOZE.num_excluded:,} 個"
    f"(うち文が 2 つ未満の passage {CLOZE.num_single_sentence:,} 個。残りは長さ {MIN_QUERY_TOKENS}〜{MAX_REMOVED_TOKENS} トークンの文がない passage)"
)
print(
    f"索引の内訳: 学習用 {len(TRAIN_ENTRIES):,}・検証用 {len(VALIDATION_INDEX_ENTRIES):,}・評価用 {len(EVALUATION_INDEX_ENTRIES):,}。評価・較正の query は、記事単位のラベルの正解が 1 つ以上ある query に限る: "
    f"評価用 {len(EVALUATION_ENTRIES):,} 個(正解の数の中央値 {int(np.median(NUM_MATES[EVALUATION_ENTRIES]))}、除いた {len(EVALUATION_INDEX_ENTRIES) - len(EVALUATION_ENTRIES)} 個)、検証用 {len(VALIDATION_ENTRIES):,} 個。"
    f"ランダムな順位の Recall@10 の期待値: passage 単位のラベル {RANDOM_RECALL_PASSAGE:.6f}、記事単位のラベル {RANDOM_RECALL_ARTICLE_LABEL:.5f}。文脈の長さ: 平均 {CONTEXT_LENGTHS.mean():.1f}、10・50・90 パーセンタイル {np.percentile(CONTEXT_LENGTHS, [10, 50, 90]).tolist()}。"
    f"query の長さ(抜いた文の先頭 {QUERY_LENGTH} トークンまで): 平均 {CLOZE_QUERY_LENGTHS.mean():.1f}、50 パーセンタイル {float(np.median(CLOZE_QUERY_LENGTHS)):.0f}。"
    f"評価の索引で、抜いた文が {QUERY_LENGTH} トークンを超えて切られた query の割合 {float((CLOZE.removed_length > QUERY_LENGTH).mean()):.1%}"
)
print(
    f"学習: 1 ステップ {ARTICLES_PER_BATCH} 記事 x {PAIRS_PER_ARTICLE} 組 = {BATCH_SIZE} 組(窓 {BATCH_SIZE} 個 = {BATCH_SIZE * PASSAGE_LENGTH:,} トークン)。組を作れる学習用の記事のトークン数に対する、窓が見るトークン数のエポック数: "
    f"T_1 = {FIRST_STAGE_STEPS}: {FIRST_STAGE_STEPS * BATCH_SIZE * PASSAGE_LENGTH / TRAIN_TOKENS_PAIRABLE:.3f}。ln(N) = {UNIFORM_LOSS:.4f}(同じ記事の組を負例から除く R1 では ln(N - 1) = {math.log(BATCH_SIZE - 1):.4f})"
)
print("データの準備の時間(段階ごと): " + "、".join(f"{k} {v:.1f} 秒" for k, v in DATA_SECONDS_BY_STAGE.items()) + f"。合計 {DATA_SECONDS:.1f} 秒")
print(
    f"データの準備のピークのメモリ: {DATA_PEAK_MEMORY_GIB:.2f} GiB(Colab 無料枠 {COLAB_RAM_GIB} GiB の {RAM_FRACTION_LIMIT:.0%} = {_memory_limit:.1f} GiB 以下: OK)"
)

/Users/koji/projects/ai-theories/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


コーパス: 542,892,375 文字(取得元 Hugging Face Hub)、記事 9,826 件。使う記事 9,470 件(147,610,040 トークン)、リスト系の記事の割合 30.2%。符号化のキャッシュ: 読み込み
トークナイザ: 語彙サイズ 8192(特殊トークンなし)、UTF-8 として不正なトークン 137 個。文の分割の表: 空白で始まる 4952、改行で始まる 352、文の終わりで終わる 223、閉じ記号だけ 7 トークン。区切り: 単一バイト 0x1F(ID 3551)、終端: 0x1E(ID 3550)。コーパスでの出現回数は 区切り 0・終端 0
分割(ダイジェスト 716cfe33d05a2615、024 の本番の値と一致): 学習用 9,070 記事(うち組を作れる記事 8,338 件・141,064,791 トークン)、検証用 100 記事、評価用 300 記事。使う記事の開始位置の最小 24,214,547 は参照コーパスの全体(24,214,546 文字)の外
  記事単位のevaluation: 索引 3244 passage・query 3244 個、ダイジェスト da3e16b96714c0a4(024 の本番の値と一致)
  記事単位のvalidation: 索引 1019 passage・query 1019 個、ダイジェスト acc9ec62c2a091b0(024 の本番の値と一致)
コーパス: passage 5,185 個(記事 480 本、動作確認では学習用の記事を絞る: 80)。passage 単位の評価の索引(Inverse Cloze Task の対象): 5,162 個。除いた passage 23 個(うち文が 2 つ未満の passage 19 個。残りは長さ 4〜96 トークンの文がない passage)
索引の内訳: 学習用 919・検証用 1,016・評価用 3,227。評価・較正の query は、記事単位のラベルの正解が 1 つ以上ある query に限る: 評価用 3,227 個(正解の数の中央値 11、除いた 0 個)、検証用 1,016 個。ランダムな順位の Recall@10 の期待値: passage 単位のラベル 0.001937、記事単位のラベル 0.02019。文

### 5.4 モデルの構築・単体テスト・不変条件の確認

**モデルの構築**:

1. 008 と同じ構成の小型 GPT(4 層、$d_{\mathrm{model}} = 256$、ヘッド数 8、RoPE、RMSNorm、SwiGLU、正規化前置、埋め込みと出力層の重み共有、系列長 256、語彙サイズ 8192。構成は Hub の`config.json`から読む)を作り、
   008 の学習済みの重み(`kojikojiprg/ai-theories-small-gpt-en`の`main`)を読み込む。R1・R2・cross-encoder のすべてがこの重みから始まる。
2. R1・R2 は、平均プール・因果マスクの`TextEmbeddingModel`で包む(024 の条件 C5 と同じ)。cross-encoder は、`VariableLengthCrossEncoder`(区切り・終端のトークンと、標準偏差 0.02 で初期化したスカラーのヘッド)で包む。
   ヘッドの初期化はシード(`torch.manual_seed(26200 + s)`)で決まる。R1・R2 の初期値は、シードによらず 008 の重みで同一である。

**単体テスト**(CPU、FP32。許容の誤差は、実際に計算される型の丸めの単位 × 値の大きさから導き、固定の絶対値は使わない。比べる値がその型で計算されていることもアサーションで確かめる):

- 文の分割: 手で作った文章で、文の区切りが規則どおり(改行、`.`・`!`・`?`、閉じ引用符・閉じ括弧、略語での既知の誤り、小数点で分割しないこと)。窓の端は文の端。
- 抜く文の選択: 抜く文が範囲の長さで文の先頭から始まる、2 つの対象の文があるとき一様に選ぶ、文を抜いた文脈に抜いた文を戻すと元の窓に一致する、パディングが 0。
- passage 単位の評価の索引: 抜く文の選択が passage の並びに依らず決定的、すべての索引の passage が「1 文を除いた形」(文脈の長さ + 抜いた文の長さ = 窓の長さ)、抜いた文を戻すと元の passage に一致する。
- 学習の組(Inverse Cloze Task): シードで決定的、32 記事 × 2 組で同じ記事の 2 組が隣り合い窓が重ならない、1 つのバッチに同じ記事が 1 回だけ、文を文脈に残す割合が約 0.1、長さの範囲、パディング、ストリームの窓と文脈・抜いた文の整合、query の長さ以内に収まる文の優先(切り詰める割合が下がること)。
- 可変長のモデル: 右側のパディングが出力に影響しない(パディングなしの出力と一致)、満長のとき`TextEmbeddingModel.pooled()`と bit 単位で一致、バッチの大きさに依らない、cross-encoder の列の組み立てと終端の位置、
  パディングの内容がスコアに影響しない、パディングなしの 025 の`CrossEncoder`と同じスコア。
- 指標: 単一の正例の順位(passage 単位のラベル)と、最もスコアの高い正解の順位(記事単位のラベル。query の passage 自身は候補から除く)が素朴な実装と一致、**同点は正例・正解に不利**(全候補に同じスコアを返すモデルの Recall@10 が 0 になる)。
- マスクつきの in-batch negatives: 同じ記事の組の passage を負例から除いた損失が閉形式の値と一致、除かないとき通常の InfoNCE と一致。
- 学習ループ: 満長の組で 024 の`train_text_embedding()`と同じ損失の系列になる、学習率 0 の学習(更新がない)の訓練損失が同じ組での学習前のモデルの損失と一致し、モデルが変化しない(第 1 段の R1・R2 の組と cross-encoder)。R1 の組では各行で同じ記事のもう一方の組が負例から除かれ、負例の数が R2 より 1 つ少ない。
- cross-encoder の学習の組: 正例が先頭で上位の候補の中にある、負例が正例を含まず重複しない、決定的。

**既存モジュールの後方互換性**: 本トピックの`src/`は新しいファイルの追加だけである。変更前のコミット(`8ca43d1`)と比べて、`src/`・`scripts/`の既存のファイルに変更も削除もないことを`git diff`で確かめる。

In [5]:
_t0_checks = time.time()
from huggingface_hub import hf_hub_download  # noqa: E402, F811

EPSILON_FP16, EPSILON_FP32, EPSILON_FP64 = (float(torch.finfo(t).eps) for t in (torch.float16, torch.float32, torch.float64))


def max_abs_difference(a, b) -> float:
    return float((torch.as_tensor(a).detach().cpu().double() - torch.as_tensor(b).detach().cpu().double()).abs().max())  # MPS は FP64 を扱えないので CPU で計算する


# --- 008 の構成と重み(Hub の main) ---
REFERENCE_CONFIG = json.loads(Path(hf_hub_download(REFERENCE_MODEL_REPO_ID, "config.json")).read_text(encoding="utf-8"))
REFERENCE_STATE = {k: v.clone() for k, v in torch.load(hf_hub_download(REFERENCE_MODEL_REPO_ID, "model_state.pt"), map_location="cpu").items()}
assert REFERENCE_CONFIG["vocabulary_size"] == tokenizer.vocab_size == 8192 and REFERENCE_CONFIG["d_model"] == EMBEDDING_DIMENSION
assert REFERENCE_CONFIG["sequence_length"] >= QUERY_LENGTH + 1 + PASSAGE_LENGTH + 1 and REFERENCE_CONFIG["tie_embeddings"]


def build_gpt(config: dict) -> GPTLanguageModel:
    # 006・008・024 と同じ構成(RoPE・RMSNorm・SwiGLU・正規化前置・重み共有)
    return GPTLanguageModel(
        vocabulary_size=config["vocabulary_size"],
        d_model=config["d_model"],
        num_layers=config["num_layers"],
        num_heads=config["num_heads"],
        d_ff=config["d_ff"],
        max_sequence_length=config["sequence_length"],
        positional_transform=RotaryPositionEmbedding(config["d_model"] // config["num_heads"], max_position=config["sequence_length"]),
        normalization_factory=RMSNorm,
        feed_forward_factory=functools.partial(SwiGLUFeedForwardNetwork, config["d_model"], config["swiglu_d_ff"]),
        tie_embeddings=config["tie_embeddings"],
        dropout=config["dropout"],
    )


def state_hash(model_or_state) -> str:
    state = model_or_state.state_dict() if hasattr(model_or_state, "state_dict") else model_or_state
    digest = hashlib.sha256()
    for _, value in sorted(state.items()):
        digest.update(value.detach().contiguous().cpu().numpy().tobytes())
    return digest.hexdigest()[:16]


def build_embedding_model(seed_index: int = 0) -> TextEmbeddingModel:
    # R1・R2: 008 の重みから始める平均プール・因果マスクのモデル(初期値はシードによらない。024 の C5 と同じ)
    torch.manual_seed(INIT_SEED_BASE + seed_index)
    backbone = build_gpt(REFERENCE_CONFIG)
    backbone.load_state_dict(REFERENCE_STATE)
    return TextEmbeddingModel(backbone, pooling="mean", attention="causal")


def build_cross_encoder(seed_index: int) -> VariableLengthCrossEncoder:
    # cross-encoder: 008 の重み + 区切り・終端 + シードで初期化したヘッド
    torch.manual_seed(INIT_SEED_BASE + seed_index)
    backbone = build_gpt(REFERENCE_CONFIG)
    backbone.load_state_dict(REFERENCE_STATE)
    return VariableLengthCrossEncoder(backbone, SEPARATOR_TOKEN_ID, TERMINAL_TOKEN_ID, HEAD_INIT_STD)


def build_format024_batches(schedule: dict[str, np.ndarray]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    # 024 の学習の組(記事の連結から切り出した固定長の区間)。すべての行が満長。学習ループの等価性の確認だけに使う
    starts = np.asarray(TRAIN_STARTS)[schedule["article"]]
    query_index = (starts + schedule["query_start"])[:, :, None] + np.arange(QUERY_LENGTH)[None, None, :]
    passage_index = (starts + schedule["passage_start"])[:, :, None] + np.arange(PASSAGE_LENGTH)[None, None, :]
    shape = schedule["article"].shape
    return (
        TRAIN_STREAM[query_index].astype(np.int64),
        np.full(shape, QUERY_LENGTH, dtype=np.int64),
        TRAIN_STREAM[passage_index].astype(np.int64),
        np.full(shape, PASSAGE_LENGTH, dtype=np.int64),
    )


def small_pair_batches(num_steps: int, articles_per_batch: int, seed: int):
    return sample_cloze_pair_batches(
        TRAIN_STREAM, np.asarray(TRAIN_STARTS), TRAIN_TOKEN_COUNTS, SAMPLING_WEIGHTS, SENTENCE_TABLES, num_steps, articles_per_batch, PAIRS_PER_ARTICLE,
        PASSAGE_LENGTH, QUERY_LENGTH, MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS, KEEP_PROBABILITY, seed=seed,
    )


def segment_texts(token_ids: np.ndarray) -> list[str]:
    # 文の分割の確認用: 1 本のトークン列を文ごとのテキストに戻す
    boundary = sentence_boundaries(token_ids[None, :], SENTENCE_TABLES)[0]
    pieces, current = [], []
    for token, end in zip(token_ids.tolist(), boundary.tolist(), strict=True):
        current.append(token)
        if end:
            pieces.append(tokenizer.decode(current))
            current = []
    return pieces


# --- 文の分割 ---
_document = "The cat sat on the mat. It was a very nice mat indeed! Is it? \"Yes.\" He said (ok). New line here\n\nNext para is long enough to count as a sentence.\n* item one is here\n* item two is there"
_expected = ["The cat sat on the mat.", " It was a very nice mat indeed!", " Is it?", ' "Yes."', " He said (ok).", " New line here", "\n\nNext para is long enough to count as a sentence.", "\n* item one is here", "\n* item two is there"]
assert segment_texts(np.array(tokenizer.encode(_document))) == _expected, segment_texts(np.array(tokenizer.encode(_document)))
_known_error = segment_texts(np.array(tokenizer.encode("He met Dr. Smith. Pi is 3.14 exactly.")))
assert _known_error == ["He met Dr.", " Smith.", " Pi is 3.14 exactly."], _known_error  # 略語の後の空白で分割する(既知の誤り)。小数点では分割しない
_no_period = segment_texts(np.array(tokenizer.encode("alpha beta gamma delta")))
assert _no_period == ["alpha beta gamma delta"]  # 区切りがなければ窓全体が 1 つの文(窓の端は文の端)
print("文の分割: 手で作った文章で規則どおり(改行・文末記号・閉じ引用符/括弧)、略語の後の空白では分割する(既知の誤り)、小数点では分割しない、区切りがなければ窓全体が 1 文: OK")

# --- 抜く文の選択・文を抜く操作(コーパスの実際の窓で) ---
_windows = CORPUS.passages[:2000]
_uniform = np.random.default_rng(0).random(len(_windows))
_valid, _start, _length = choose_cloze_sentence(_windows, SENTENCE_TABLES, _uniform, MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS)
_is_start, _sentence_length, _num_sentences, _boundary = sentence_structure(_windows, SENTENCE_TABLES)
_rows = np.arange(len(_windows))
assert _valid.mean() > 0.9 and (_boundary[:, -1]).all()
assert _is_start[_rows[_valid], _start[_valid]].all() and (_num_sentences[_valid] >= 2).all(), "抜く文が文の先頭から始まっていない、または文が 2 つ未満"
assert ((_length[_valid] >= MIN_QUERY_TOKENS) & (_length[_valid] <= MAX_REMOVED_TOKENS)).all()
assert np.array_equal(_length[_valid], _sentence_length[_rows[_valid], _start[_valid]])
_again = choose_cloze_sentence(_windows, SENTENCE_TABLES, _uniform, MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS)
assert all(np.array_equal(a, b) for a, b in zip((_valid, _start, _length), _again, strict=True)), "抜く文の選択が決定的でない"
_context, _context_length = remove_span(_windows, _start, _length)
_query, _query_length = extract_span(_windows, _start, _length, QUERY_LENGTH)
for _i in np.nonzero(_valid)[0][:300]:  # 文脈に抜いた文を戻すと元の窓に一致する(文が query の長さ以内のとき、query が文そのもの)
    _restored = np.concatenate([_context[_i, : _start[_i]], _windows[_i, _start[_i] : _start[_i] + _length[_i]], _context[_i, _start[_i] : _context_length[_i]]])
    assert np.array_equal(_restored, _windows[_i]) and _context_length[_i] == PASSAGE_LENGTH - _length[_i]
    assert (_context[_i, _context_length[_i] :] == 0).all() and (_query[_i, _query_length[_i] :] == 0).all()
    assert np.array_equal(_query[_i, : _query_length[_i]], _windows[_i, _start[_i] : _start[_i] + _query_length[_i]]) and _query_length[_i] == min(_length[_i], QUERY_LENGTH)
# 対象の文が 2 つある窓では、先頭の文を選ぶ確率が約 1/2(二項分布の標準誤差の 4 倍以内)
_eligible_count = (_is_start & (_sentence_length >= MIN_QUERY_TOKENS) & (_sentence_length <= MAX_REMOVED_TOKENS) & (_num_sentences >= 2)[:, None]).sum(axis=1)
_two = _eligible_count == 2
_first_eligible = np.array([np.nonzero(_is_start[i] & (_sentence_length[i] >= MIN_QUERY_TOKENS) & (_sentence_length[i] <= MAX_REMOVED_TOKENS))[0][0] for i in np.nonzero(_two)[0]])
_picked_first = float(np.mean(_start[_two] == _first_eligible))
assert abs(_picked_first - 0.5) < 4 * math.sqrt(0.25 / _two.sum()), (_picked_first, int(_two.sum()))
print(f"抜く文の選択: 範囲の長さの文の先頭から始まる、決定的、抜いた文を戻すと元の窓に一致する、パディングが 0、対象の文が 2 つの窓({int(_two.sum())} 個)で先頭の文を選ぶ割合 {_picked_first:.3f}: OK")

# --- passage 単位の評価の索引 ---
assert CLOZE.passage_ids.shape == (NUM_INDEX,) and (np.diff(CLOZE.passage_ids) > 0).all()
assert (CLOZE.context_lengths + CLOZE.removed_length == PASSAGE_LENGTH).all(), "索引の passage が「1 文を除いた形」でない"
assert (CLOZE.context_lengths >= PASSAGE_LENGTH - MAX_REMOVED_TOKENS).all() and (CLOZE.removed_length >= MIN_QUERY_TOKENS).all()
_original = CORPUS.passages[CLOZE.passage_ids]
for _i in np.linspace(0, NUM_INDEX - 1, 400).astype(int):
    _s, _n = int(CLOZE.removed_start[_i]), int(CLOZE.removed_length[_i])
    _ctx = CLOZE.context_tokens[_i, : CLOZE.context_lengths[_i]]
    assert np.array_equal(np.concatenate([_ctx[:_s], _original[_i, _s : _s + _n], _ctx[_s:]]), _original[_i]), "抜いた文を戻しても元の passage に一致しない"
    assert (CLOZE.context_tokens[_i, CLOZE.context_lengths[_i] :] == 0).all() and CLOZE.query_lengths[_i] == min(_n, QUERY_LENGTH)
    assert np.array_equal(CLOZE.query_tokens[_i, : CLOZE.query_lengths[_i]], _original[_i, _s : _s + CLOZE.query_lengths[_i]])
_permutation = np.random.default_rng(1).permutation(CORPUS.num_passages)
_permuted = build_cloze_index(
    CORPUS.passages[_permutation], CORPUS.passage_articles[_permutation], CORPUS.passage_windows[_permutation], SENTENCE_TABLES, QUERY_LENGTH, MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS, CLOZE_SEED
)
_by_passage = dict(zip(_permutation[_permuted.passage_ids].tolist(), _permuted.removed_start.tolist(), strict=True))
assert set(_by_passage) == set(CLOZE.passage_ids.tolist()) and all(_by_passage[p] == s for p, s in zip(CLOZE.passage_ids.tolist(), CLOZE.removed_start.tolist(), strict=True)), "抜く文の選択が passage の並びに依る"
_other_seed = build_cloze_index(CORPUS.passages, CORPUS.passage_articles, CORPUS.passage_windows, SENTENCE_TABLES, QUERY_LENGTH, MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS, CLOZE_SEED + 1)
assert not np.array_equal(_other_seed.removed_start, CLOZE.removed_start), "異なるシードで同じ選択(確認の検出力がない)"
_unpreferred = build_cloze_index(CORPUS.passages, CORPUS.passage_articles, CORPUS.passage_windows, SENTENCE_TABLES, 10**6, MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS, CLOZE_SEED)
assert float((CLOZE.removed_length > QUERY_LENGTH).mean()) < float((_unpreferred.removed_length > QUERY_LENGTH).mean()), "評価の索引で、収まる文を優先していない"
print(f"評価の索引で切り詰めた query の割合: 優先なし {float((_unpreferred.removed_length > QUERY_LENGTH).mean()):.1%}、収まる文を優先 {float((CLOZE.removed_length > QUERY_LENGTH).mean()):.1%}")
print(f"passage 単位の評価の索引: すべて「1 文を除いた形」(文脈の長さ + 抜いた文の長さ = {PASSAGE_LENGTH})、抜いた文を戻すと元の passage に一致する、抜く文の選択が passage の並びに依らず決定的、シードを変えると変わる: OK")

# --- 学習の組(Inverse Cloze Task。32 記事 x 2 組) ---
_pairs_a = small_pair_batches(200, ARTICLES_PER_BATCH, 5)
_pairs_b = small_pair_batches(200, ARTICLES_PER_BATCH, 5)
_fields = ("query_tokens", "query_lengths", "context_tokens", "context_lengths", "articles", "kept", "truncated", "window_starts")
assert all(np.array_equal(getattr(_pairs_a, k), getattr(_pairs_b, k)) for k in _fields), "学習の組がシードで決定的でない"
assert not np.array_equal(_pairs_a.context_tokens, small_pair_batches(200, ARTICLES_PER_BATCH, 6).context_tokens), "異なるシードで同じ組(確認の検出力がない)"
_articles = _pairs_a.articles
assert _articles.shape == (200, BATCH_SIZE) and (_articles[:, 0::2] == _articles[:, 1::2]).all(), "同じ記事の 2 組がバッチの中で隣り合っていない"
assert all(len(set(row[0::2].tolist())) == ARTICLES_PER_BATCH for row in _articles), "1 つのバッチに同じ記事が複数入っている"
assert (SAMPLING_WEIGHTS[_articles] > 0).all()
_window_starts = _pairs_a.window_starts
assert (np.abs(_window_starts[:, 0::2] - _window_starts[:, 1::2]) >= PASSAGE_LENGTH).all(), "同じ記事の 2 組の窓が重なっている"
assert (_window_starts >= 0).all() and (_window_starts + PASSAGE_LENGTH <= TRAIN_TOKEN_COUNTS[_articles]).all(), "窓が記事に収まっていない"
_kept = float(_pairs_a.kept.mean())
assert abs(_kept - KEEP_PROBABILITY) < 4 * math.sqrt(KEEP_PROBABILITY * (1 - KEEP_PROBABILITY) / _pairs_a.kept.size), _kept
assert (_pairs_a.context_lengths[_pairs_a.kept] == PASSAGE_LENGTH).all() and (_pairs_a.context_lengths[~_pairs_a.kept] < PASSAGE_LENGTH).all()
assert (_pairs_a.context_lengths >= PASSAGE_LENGTH - MAX_REMOVED_TOKENS).all() and (_pairs_a.query_lengths >= MIN_QUERY_TOKENS).all() and (_pairs_a.query_lengths <= QUERY_LENGTH).all()
assert (_pairs_a.context_tokens * (np.arange(PASSAGE_LENGTH)[None, None, :] >= _pairs_a.context_lengths[:, :, None])).sum() == 0
assert (_pairs_a.query_tokens * (np.arange(QUERY_LENGTH)[None, None, :] >= _pairs_a.query_lengths[:, :, None])).sum() == 0
# 窓の切り出しと、抜いた文・文脈の整合(ストリームから窓を取り直し、文脈に抜いた文を戻すと窓に一致する)
for _step, _slot in itertools.product((0, 57, 199), (0, 1, 30, 63)):
    _window = TRAIN_STREAM[TRAIN_STARTS[_articles[_step, _slot]] + _window_starts[_step, _slot] :][:PASSAGE_LENGTH].astype(np.int64)
    _context = _pairs_a.context_tokens[_step, _slot, : _pairs_a.context_lengths[_step, _slot]]
    _query = _pairs_a.query_tokens[_step, _slot, : _pairs_a.query_lengths[_step, _slot]]
    if _pairs_a.kept[_step, _slot]:
        assert np.array_equal(_context, _window)
    else:
        _removed = PASSAGE_LENGTH - len(_context)
        _starts = [s for s in range(len(_context) + 1) if np.array_equal(np.concatenate([_context[:s], _window[s : s + _removed], _context[s:]]), _window) and np.array_equal(_window[s : s + len(_query)], _query)]
        assert _starts and (_pairs_a.truncated[_step, _slot] == (_removed > QUERY_LENGTH)), (_step, _slot)
# 抜く文の選び方: query の長さの上限以内に収まる文を優先する。収まる文がある窓では、切り詰めない
_short_windows = CORPUS.passages[:3000]
_valid_s, _start_s, _length_s = choose_cloze_sentence(_short_windows, SENTENCE_TABLES, np.random.default_rng(4).random(len(_short_windows)), MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS, QUERY_LENGTH)
_valid_n, _start_n, _length_n = choose_cloze_sentence(_short_windows, SENTENCE_TABLES, np.random.default_rng(4).random(len(_short_windows)), MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS)
_is_start_s, _sentence_length_s, _num_sentences_s, _ = sentence_structure(_short_windows, SENTENCE_TABLES)
_has_short = (_is_start_s & (_sentence_length_s >= MIN_QUERY_TOKENS) & (_sentence_length_s <= QUERY_LENGTH) & (_num_sentences_s >= 2)[:, None]).any(axis=1)
assert np.array_equal(_valid_s, _valid_n), "優先しても、対象の窓の集合が変わっている"
assert (_length_s[_valid_s & _has_short] <= QUERY_LENGTH).all(), "収まる文がある窓で、長い文を選んでいる"
assert (_length_s[_valid_s & ~_has_short] > QUERY_LENGTH).all() and (_length_s[_valid_s] <= MAX_REMOVED_TOKENS).all()
_truncated_before, _truncated_after = float((_length_n[_valid_n] > QUERY_LENGTH).mean()), float((_length_s[_valid_s] > QUERY_LENGTH).mean())
assert _truncated_after < _truncated_before
print(
    f"学習の組(Inverse Cloze Task、{ARTICLES_PER_BATCH} 記事 x {PAIRS_PER_ARTICLE} 組): 200 ステップ x {BATCH_SIZE} 組で、シードで決定的、同じ記事の 2 組がバッチの中で隣り合い窓が重ならない、1 つのバッチに同じ記事が 1 回だけ、"
    f"文を文脈に残す割合 {_kept:.4f}(期待 {KEEP_PROBABILITY})、文脈・query の長さの範囲、パディングが 0、ストリームの窓と文脈・抜いた文の整合、窓を引き直した回数 {_pairs_a.num_attempts - _pairs_a.kept.size // 2}、"
    f"記事の全窓から選んだ組 {_pairs_a.num_exhaustive}: OK。切り詰めた query の割合(学習の組) {float(_pairs_a.truncated.mean()):.1%}"
    f"(窓の抜く文を一様に選ぶと {_truncated_before:.1%}、収まる文を優先すると {_truncated_after:.1%}): OK"
)

# --- 可変長のモデル(008 の重み、CPU、FP32) ---
_model = build_embedding_model().eval()
_torch_generator = torch.Generator().manual_seed(1)
_tokens = torch.randint(0, 8192, (6, 50), generator=_torch_generator)
_lengths = torch.tensor([50, 31, 10, 2, 1, 40])
with torch.no_grad():
    _padded = pooled_with_lengths(_model, _tokens, _lengths)
    assert _padded.dtype == torch.float32
    for _i in range(len(_lengths)):  # パディングなし(その行の長さだけ)の出力と一致する
        _reference = _model.pooled(_tokens[_i : _i + 1, : _lengths[_i]])[0]
        assert max_abs_difference(_padded[_i], _reference) <= 64 * EPSILON_FP32 * float(_reference.abs().max()), (_i, max_abs_difference(_padded[_i], _reference))
    _garbage = _tokens.clone()
    for _i in range(len(_lengths)):
        _garbage[_i, _lengths[_i] :] = torch.randint(0, 8192, (50 - int(_lengths[_i]),), generator=_torch_generator)
    assert torch.equal(pooled_with_lengths(_model, _garbage, _lengths), _padded), "パディングの内容が出力に影響している"
    assert torch.equal(pooled_with_lengths(_model, _tokens, torch.full((6,), 50), full_length=True), _model.pooled(_tokens)), "満長の経路が TextEmbeddingModel.pooled() と一致しない"
    _wide = encode_tokens_with_lengths(_model, _tokens, _lengths.numpy(), 6, (None, 16))
    _narrow = encode_tokens_with_lengths(_model, _tokens, _lengths.numpy(), 2, (None, 16))
    _repeat = max_abs_difference(_wide[256], encode_tokens_with_lengths(_model, _tokens, _lengths.numpy(), 6)[256])
    _unit = EPSILON_FP32 * float(_wide[256].abs().max())
    assert max_abs_difference(_wide[256], _narrow[256]) <= 16 * max(_repeat, _unit) and max_abs_difference(_wide[16], _narrow[16]) <= 16 * max(_repeat, _unit)
    _expected_unit = torch.nn.functional.normalize(_padded, dim=-1)
    assert max_abs_difference(_wide[256], _expected_unit) <= 16 * max(_repeat, _unit), "符号化の結果が元の並びに戻っていない、または正規化が違う"
print(f"可変長の平均プール: パディングなしの出力と一致(許容は丸めの単位 x 値の大きさの 64 倍)、パディングの内容が出力に影響しない(bit 単位)、満長の経路が pooled() と bit 単位で一致、バッチの大きさに依らず元の並びに戻る: OK")

_cross = build_cross_encoder(0).eval()
_reference_cross = CrossEncoder(_cross.backbone, SEPARATOR_TOKEN_ID, TERMINAL_TOKEN_ID, HEAD_INIT_STD).eval()
_reference_cross.head.load_state_dict(_cross.head.state_dict())
_query_tokens = torch.randint(0, 8192, (5, 12), generator=_torch_generator)
_passage_tokens = torch.randint(0, 8192, (5, 20), generator=_torch_generator)
_query_lengths = torch.tensor([12, 7, 1, 12, 4])
_passage_lengths = torch.tensor([20, 20, 3, 9, 15])
_inputs, _total = build_variable_cross_encoder_inputs(_query_tokens, _query_lengths, _passage_tokens, _passage_lengths, SEPARATOR_TOKEN_ID, TERMINAL_TOKEN_ID)
for _i in range(5):  # 列は左から詰まっている: [query][区切り][passage][終端]、終端の後は 0
    _lq, _lp = int(_query_lengths[_i]), int(_passage_lengths[_i])
    _row = _inputs[_i].tolist()
    assert _row[:_lq] == _query_tokens[_i, :_lq].tolist() and _row[_lq] == SEPARATOR_TOKEN_ID and _row[_lq + 1 : _lq + 1 + _lp] == _passage_tokens[_i, :_lp].tolist()
    assert _row[_lq + 1 + _lp] == TERMINAL_TOKEN_ID and sum(abs(v) for v in _row[_lq + 2 + _lp :]) == 0 and int(_total[_i]) == _lq + _lp + 2
_full_inputs = build_cross_encoder_inputs(_query_tokens, _passage_tokens, SEPARATOR_TOKEN_ID, TERMINAL_TOKEN_ID)
_inputs_full, _ = build_variable_cross_encoder_inputs(_query_tokens, torch.full((5,), 12), _passage_tokens, torch.full((5,), 20), SEPARATOR_TOKEN_ID, TERMINAL_TOKEN_ID)
assert torch.equal(_inputs_full, _full_inputs), "満長の列が 025 の build_cross_encoder_inputs() と一致しない"
with torch.no_grad():
    _scores = _cross(_query_tokens, _passage_tokens, _query_lengths, _passage_lengths)
    for _i in range(5):  # パディングなしの 025 の CrossEncoder と同じスコア
        _single = _reference_cross(_query_tokens[_i : _i + 1, : _query_lengths[_i]], _passage_tokens[_i : _i + 1, : _passage_lengths[_i]])[0]
        assert abs(float(_scores[_i]) - float(_single)) <= 64 * EPSILON_FP32 * max(abs(float(_single)), 1.0), (_i, float(_scores[_i]), float(_single))
    _garbage_q, _garbage_p = _query_tokens.clone(), _passage_tokens.clone()
    for _i in range(5):
        _garbage_q[_i, _query_lengths[_i] :] = 7
        _garbage_p[_i, _passage_lengths[_i] :] = 11
    assert torch.equal(_cross(_garbage_q, _garbage_p, _query_lengths, _passage_lengths), _scores), "パディングの内容がスコアに影響している"
    _changed_passage = _passage_tokens.clone()
    _changed_passage[0, 19] = (_changed_passage[0, 19] + 1) % 8192  # 満長の行 0 の passage の最後のトークン
    assert float(_cross(_query_tokens, _changed_passage, _query_lengths, _passage_lengths)[0]) != float(_scores[0]), "終端の位置の出力が passage の最後のトークンに依存しない"
    _changed_query = _query_tokens.clone()
    _changed_query[1, 0] = (_changed_query[1, 0] + 1) % 8192
    assert float(_cross(_changed_query, _passage_tokens, _query_lengths, _passage_lengths)[1]) != float(_scores[1]), "終端の位置の出力が query に依存しない"
print("可変長の cross-encoder: 列が左から詰まっている([query][区切り][passage][終端])、満長のとき 025 の入力と bit 単位で一致、パディングなしの 025 の CrossEncoder と同じスコア、パディングの内容がスコアに影響しない(bit 単位)、終端の位置が query と passage の両方に依存する: OK")
del _model, _cross, _reference_cross

# --- 指標: 単一の正例の順位と、同点の扱い ---
_generator = torch.Generator().manual_seed(2)
_passage_embeddings = torch.nn.functional.normalize(torch.randn(60, 8, generator=_generator, dtype=torch.float64), dim=-1)
_query_embeddings = torch.nn.functional.normalize(torch.randn(10, 8, generator=_generator, dtype=torch.float64), dim=-1)
_positive = np.random.default_rng(3).integers(0, 60, size=10)
_ranks = rank_single_positive(_query_embeddings, _passage_embeddings, _positive, chunk_size=4)
for _i in range(10):
    _scores_i = (_passage_embeddings @ _query_embeddings[_i]).numpy()
    _naive = 1 + sum(1 for p in range(60) if p != _positive[_i] and _scores_i[p] >= _scores_i[_positive[_i]])
    assert _ranks[_i] == _naive
_constant = torch.nn.functional.normalize(torch.ones(1, 8), dim=-1)
_constant_ranks = rank_single_positive(_constant.repeat(5, 1), _constant.repeat(30, 1), np.array([0, 5, 10, 20, 29]))
assert (_constant_ranks == 30).all() and not (_constant_ranks <= 10).any(), "同点を正例に不利に数えていない(全件が同じスコアなら、順位は最下位)"
_tie = rerank_single_positive(np.array([[1.0, 1.0, 0.0]]), np.array([[3, 7, 2]]), np.array([7]), 10)
assert _tie.rank.tolist() == [2], "同点の負例が正例より上位でない"
assert rerank_single_positive(np.array([[0.5, 0.9, 0.1]]), np.array([[4, 7, 2]]), np.array([7]), 10).rank.tolist() == [1]
_constant_rerank = rerank_single_positive(np.zeros((20, 50)), np.tile(np.arange(50), (20, 1)), np.arange(20), 100)
assert _constant_rerank.coverage() == 1.0 and _constant_rerank.recall_at(10) == 0.0 and (_constant_rerank.rank == 50).all(), "全候補に同じスコアを返すモデルの Recall@10 が 0 でない"
assert rerank_single_positive(np.zeros((1, 5)), np.array([[0, 1, 2, 3, 4]]), np.array([99]), 100).rank.tolist() == [6], "候補に正例がないときの順位が K + 1 でない"
print("指標: 単一の正例の順位が素朴な実装と一致、同点は正例に不利(全件同じスコアで順位が最下位、全候補に同じスコアを返すモデルの Recall@10 が 0)、候補に正例がなければ順位 K + 1: OK")

# --- 指標: 記事単位のラベル(同じ索引・同じ query。正解は同じ記事の他の passage、query の passage 自身は候補から除く)---
_articles_small = np.repeat(np.arange(15), 4)  # 15 記事 x 4 passage
_label_passages = torch.nn.functional.normalize(torch.randn(60, 8, generator=_generator, dtype=torch.float64), dim=-1)
_label_positions = np.array([0, 5, 6, 13, 22, 40, 41, 59])
_label_queries = torch.nn.functional.normalize(torch.randn(8, 8, generator=_generator, dtype=torch.float64), dim=-1)
_article_result = rank_article_label(_label_queries, _label_passages, _label_positions, _articles_small, chunk_size=3)
for _i, _p in enumerate(_label_positions):
    _scores_i = (_label_passages @ _label_queries[_i]).numpy()
    _mates = [m for m in range(60) if _articles_small[m] == _articles_small[_p] and m != _p]
    _best = max(_scores_i[m] for m in _mates)
    _naive = 1 + sum(1 for c in range(60) if c != _p and c not in _mates and _scores_i[c] >= _best)
    assert _article_result.rank[_i] == _naive, (_i, _article_result.rank[_i], _naive)
assert (_article_result.num_positives == 3).all()
_constant_article = rank_article_label(_constant.repeat(3, 1), _constant.repeat(30, 1), np.array([0, 9, 17]), np.repeat(np.arange(10), 3))
assert (_constant_article.rank == 28).all() and not (_constant_article.rank <= 10).any(), "同点を正解に不利に数えていない(記事単位のラベル)"
print("指標(記事単位のラベル): 素朴な実装と一致(query の passage 自身は候補から除く、最もスコアの高い正解の順位、同点は正解に不利)、全件同じスコアで Recall@10 が 0: OK")

# --- マスクつきの in-batch negatives(同じ記事の組の扱い)---
_u = torch.nn.functional.normalize(torch.randn(6, 16, generator=_generator, dtype=torch.float64), dim=-1)
_v = torch.nn.functional.normalize(torch.randn(6, 16, generator=_generator, dtype=torch.float64), dim=-1)
_ids = np.array([0, 0, 1, 1, 2, 2])
_ignore = torch.from_numpy((_ids[:, None] == _ids[None, :]) & ~np.eye(6, dtype=bool))
_masked = masked_info_nce_loss(_u, _v, TEMPERATURE, _ignore)
_logits = (_u @ _v.t() / TEMPERATURE).numpy()
_expected_terms = []
for _i in range(6):
    _candidates = [j for j in range(6) if j == _i or _ids[j] != _ids[_i]]  # 正例と、別の記事の passage だけ
    _expected_terms.append(-(_logits[_i, _i] - math.log(sum(math.exp(_logits[_i, j]) for j in _candidates))))
assert abs(float(_masked) - float(np.mean(_expected_terms))) <= 64 * EPSILON_FP64 * abs(float(np.mean(_expected_terms)))
_unmasked = masked_info_nce_loss(_u, _v, TEMPERATURE, torch.zeros(6, 6, dtype=torch.bool))
assert abs(float(_unmasked) - float(info_nce_loss(_u, _v, TEMPERATURE))) <= 64 * EPSILON_FP64 * float(_unmasked), "除かないとき、通常の InfoNCE と一致しない"
assert float(_masked) < float(_unmasked), "候補を除いたのに損失が下がっていない(負例が減ると下がる)"
print(f"マスクつきの in-batch negatives: 同じ記事の組の passage を負例から除いた損失が閉形式の値と一致(FP64)、除かないとき通常の InfoNCE と一致、除くと損失は下がる({float(_masked):.4f} < {float(_unmasked):.4f}): OK")

# --- 学習ループ(008 の重み、CPU、FP32、小さい組) ---
_schedule_small = sample_pair_schedule(TRAIN_TOKEN_COUNTS, SAMPLING_WEIGHTS, QUERY_LENGTH, PASSAGE_LENGTH, 4, BATCH_SIZE, seed=5)  # 024 の学習の組(満長の区間)。学習ループの等価性の確認だけに使う
_small = {k: v[:2, :8] for k, v in _schedule_small.items()}
_small_format024 = build_format024_batches(_small)
_stream_tensor = torch.from_numpy(TRAIN_STREAM.astype(np.int64))
for _case in ("r1_equivalence", "zero_learning_rate"):
    _lr = 1e-4 if _case == "r1_equivalence" else 0.0
    _reference_model = build_embedding_model(0)
    _new_model = build_embedding_model(0)
    _reference_history = train_text_embedding(
        _reference_model, _stream_tensor, TRAIN_STARTS, _small, QUERY_LENGTH, PASSAGE_LENGTH, _lr, 1, 0.0 if _lr == 0 else 1e-6, WEIGHT_DECAY,
        GRADIENT_CLIP_THRESHOLD, TEMPERATURE, matryoshka_dimensions=MATRYOSHKA_DIMENSIONS,
    )
    _new_batches = VariableBatches(*_small_format024, torch.device("cpu"))
    _initial_hash = state_hash(_new_model)
    _new_history = train_text_embedding_from_batches(
        _new_model, _new_batches, _lr, 1, 0.0 if _lr == 0 else 1e-6, WEIGHT_DECAY, GRADIENT_CLIP_THRESHOLD, TEMPERATURE, matryoshka_dimensions=MATRYOSHKA_DIMENSIONS
    )
    # 同じ演算・同じ順序なので、損失の系列は一致する(許容: 丸めの単位 x 損失の大きさ x 16)
    assert np.allclose(_new_history["loss"], _reference_history["loss"], rtol=16 * EPSILON_FP32, atol=0.0), (_new_history["loss"], _reference_history["loss"])
    if _case == "zero_learning_rate":
        assert state_hash(_new_model) == _initial_hash, "学習率 0 の学習でモデルが変化した"
        _measured = compute_pair_losses_from_batches(_new_model, _new_batches, range(2), TEMPERATURE, matryoshka_dimensions=MATRYOSHKA_DIMENSIONS)
        assert state_hash(_new_model) == _initial_hash and _new_model.training, "compute_pair_losses_from_batches() がモデルまたはモードを変更した"
        assert np.allclose(_new_history["loss"], _measured, rtol=16 * EPSILON_FP32, atol=0.0), "学習率 0 の訓練損失が、同じ組での学習前のモデルの損失と一致しない"
        _reference_pair = compute_pair_losses(_reference_model, _stream_tensor, TRAIN_STARTS, _small, range(2), QUERY_LENGTH, PASSAGE_LENGTH, TEMPERATURE, matryoshka_dimensions=MATRYOSHKA_DIMENSIONS)
        assert np.allclose(_measured, _reference_pair, rtol=16 * EPSILON_FP32, atol=0.0)
        # 可変長の組(Inverse Cloze Task の組)でも、R2(通常の InfoNCE の負例)と R1(同じ記事の組を負例から除く)の両方で、
        # 学習率 0 の訓練損失が同じ組での学習前のモデルの損失と一致し、モデルが変化しない
        _tiny_pairs = small_pair_batches(2, 4, 3)  # 4 記事 x 2 組 = 8 組
        _swap = np.arange(8) ^ 1
        for _label, _context_tokens, _context_lengths, _mask in (
            ("R2", _tiny_pairs.context_tokens, _tiny_pairs.context_lengths, False),
            ("R1", _tiny_pairs.context_tokens[:, _swap], _tiny_pairs.context_lengths[:, _swap], True),
        ):
            _variable = VariableBatches(
                _tiny_pairs.query_tokens, _tiny_pairs.query_lengths, _context_tokens, _context_lengths, torch.device("cpu"),
                article_ids=_tiny_pairs.articles, mask_same_article=_mask,
            )
            _variable_model = build_embedding_model(0)
            _variable_before = compute_pair_losses_from_batches(_variable_model, _variable, range(2), TEMPERATURE, matryoshka_dimensions=MATRYOSHKA_DIMENSIONS)
            _variable_history = train_text_embedding_from_batches(
                _variable_model, _variable, 0.0, 1, 0.0, WEIGHT_DECAY, GRADIENT_CLIP_THRESHOLD, TEMPERATURE, matryoshka_dimensions=MATRYOSHKA_DIMENSIONS
            )
            assert state_hash(_variable_model) == _initial_hash and np.allclose(_variable_history["loss"], _variable_before, rtol=16 * EPSILON_FP32, atol=0.0), _label
            _ratio = float(np.mean(_variable_history["loss"]) / np.mean(_variable_before))
            assert abs(_ratio - 1.0) <= 16 * EPSILON_FP32 and _ratio > P1_LOSS_RATIO, "更新なしの訓練損失 / 学習前のモデルの損失が 1 でない(P1(a) が学習率 0 で不成立にならない)"
            assert _variable.ignore(0) is None if not _mask else int(_variable.ignore(0).sum()) == 8  # R1: 各行で同じ記事のもう一方の組が 1 つ除かれる
            assert (_variable.negatives_per_row() == (7 if not _mask else 6)).all()
        assert _variable_history["data_stream_hash"] != _new_history["data_stream_hash"]
print("第 1 段の学習ループ: 満長の組で 024 の train_text_embedding() と同じ損失の系列、学習率 0 の学習(R1・R2 の組、R1 は同じ記事の組を負例から除く)で訓練損失が同じ組での学習前のモデルの損失と一致しモデルが変化しない(比が 1 で P1(a) は不成立): OK")

# --- cross-encoder の学習の組と学習ループ ---
_mined = np.stack([np.concatenate([[q], np.random.default_rng(q).choice(NUM_INDEX, size=RERANK_CANDIDATES - 1, replace=False)]) for q in TRAIN_ENTRIES[:60]])
_mined[::3, 0] = (_mined[::3, 0] + 1) % NUM_INDEX  # 3 つに 1 つは、先頭を別の passage に置き換える(正例が入らない、または別の位置)
_mined[::3] = np.where(_mined[::3] == TRAIN_ENTRIES[:60][::3, None], (_mined[::3] + 7) % NUM_INDEX, _mined[::3])  # 正例を上位から外す
_usable = usable_cloze_queries(_mined, TRAIN_ENTRIES[:60])
assert 0 < _usable.sum() < 60 and not _usable[::3].any()
_schedule_ce = sample_cloze_reranking_schedule(TRAIN_ENTRIES[:60], _mined, 6, BATCH_QUERIES, NUM_NEGATIVES, seed=7)
_again_ce = sample_cloze_reranking_schedule(TRAIN_ENTRIES[:60], _mined, 6, BATCH_QUERIES, NUM_NEGATIVES, seed=7)
assert _schedule_ce.digest() == _again_ce.digest() != sample_cloze_reranking_schedule(TRAIN_ENTRIES[:60], _mined, 6, BATCH_QUERIES, NUM_NEGATIVES, seed=8).digest()
for _step in range(6):
    for _b in range(BATCH_QUERIES):
        _row = _schedule_ce.candidate_ids[_step, _b]
        _q = int(_schedule_ce.query_ids[_step, _b])
        _members = _mined[np.nonzero(TRAIN_ENTRIES[:60] == _q)[0][0]]
        assert _row[0] == _q and _q in _members, "正例が先頭でない、または上位の候補に入っていない"
        assert len(set(_row.tolist())) == 1 + NUM_NEGATIVES and set(_row[1:].tolist()) <= set(_members.tolist()) and _q not in _row[1:].tolist()
assert _usable[np.searchsorted(TRAIN_ENTRIES[:60], _schedule_ce.query_ids.reshape(-1))].all(), "学習に使えない query が学習の組に入っている"
_tables = VariableTables(CLOZE.query_tokens, CLOZE.query_lengths, CLOZE.context_tokens, CLOZE.context_lengths, torch.device("cpu"))
_cross = build_cross_encoder(0)
_initial_hash = state_hash(_cross)
_before = compute_reranking_losses_variable(_cross, _tables, _schedule_ce, range(3))
assert state_hash(_cross) == _initial_hash and _before.shape == (3,) and _before.dtype == np.float64
_history = train_reranker_variable(_cross, _tables, _schedule_ce, 0.0, 1, 0.0, WEIGHT_DECAY, GRADIENT_CLIP_THRESHOLD)
assert state_hash(_cross) == _initial_hash, "学習率 0 の学習でモデルが変化した"
assert np.allclose(_history["loss"][:3], _before, rtol=16 * EPSILON_FP32, atol=0.0), "学習率 0 の訓練損失が、同じ組での学習前のモデルの損失と一致しない"
assert abs(float(np.mean(_history["loss"][:3])) - UNIFORM_LOSS_CROSS_ENCODER) < 0.2, "学習前の cross-encoder の損失が一様な予測の損失 ln(1 + n) の付近にない"
print(f"cross-encoder の学習: 学習の組の正例が先頭で上位の候補に入り、負例が正例を含まず重複しない、決定的。学習率 0 の訓練損失が同じ組での学習前のモデルの損失と一致しモデルが変化しない(学習前の損失 {float(np.mean(_before)):.3f}、一様な予測 {UNIFORM_LOSS_CROSS_ENCODER:.3f}): OK")
del _cross, _tables

# --- 既存モジュールの後方互換性: 変更前のコミットと比べて、src/・scripts/ の既存のファイルに変更も削除もない ---
_REFERENCE_COMMIT = "8ca43d1"  # 026 の変更を始める前の最後のコミット
_diff = subprocess.run(["git", "diff", "--name-only", "--diff-filter=MD", _REFERENCE_COMMIT, "--", "src", "scripts"], capture_output=True, text=True)
assert _diff.returncode == 0, _diff.stderr
assert _diff.stdout.strip() == "", f"既存のファイルが変更または削除されている: {_diff.stdout}"
_existing = subprocess.run(["git", "ls-tree", "-r", "--name-only", _REFERENCE_COMMIT, "src", "scripts"], capture_output=True, text=True).stdout.split()
print(f"既存モジュールの後方互換性: 参照コミット {_REFERENCE_COMMIT} の src/・scripts/ の既存のファイル {len(_existing)} 個に、変更も削除もない: OK")
CHECK_SECONDS = time.time() - _t0_checks
print(f"単体テストと不変条件の確認 {CHECK_SECONDS:.1f} 秒")

文の分割: 手で作った文章で規則どおり(改行・文末記号・閉じ引用符/括弧)、略語の後の空白では分割する(既知の誤り)、小数点では分割しない、区切りがなければ窓全体が 1 文: OK
抜く文の選択: 範囲の長さの文の先頭から始まる、決定的、抜いた文を戻すと元の窓に一致する、パディングが 0、対象の文が 2 つの窓(68 個)で先頭の文を選ぶ割合 0.397: OK


### 5.5 学習と評価のヘルパー

- `evaluate_calibration()`: 検証用の記事の passage だけを索引にし、同じ索引・同じ query に、passage 単位のラベルと記事単位のラベルを当てて Recall@10 を測る。較正の指標とパイロットに使う。
- `evaluate_main()`: 評価用の集合で、コーパス全体の索引と評価用の記事の query に、passage 単位のラベル(実験 A)と記事単位のラベル(実験 B)を当てて、query ごとの順位を記録する。診断量(上位 10 件のうち同じ記事の passage の割合、
  1 位が同じ記事の別の passage である query の割合、024 と同じ形式の記事単位の評価)も測る。評価は FP32 で、全件の内積。
- `train_first_stage()`: 条件(R1 または R2)・シード・学習率・ステップ数で第 1 段を学習する。組は両条件で共通(32 記事 × 2 組、シード $26100 + s$)で、R2 は各窓の文脈を正例にし、
  R1 は同じ記事のもう一方の窓の文脈を正例にして、同じ記事の組を負例から除く。学習前のモデルの損失(最後の区間の組で)、最後の区間の訓練損失、検証用の Recall@10 を記録し、本番の学習(`main=True`)は評価用の集合の順位も記録する。
  R2 は、cross-encoder の学習と評価に使う第 1 段の上位 $K$ 件(学習用・検証用・評価用の query)も求める。
- `train_cross_encoder()`: 学習の組(`sample_cloze_reranking_schedule()`)で cross-encoder を学習する。学習前のモデルと学習後のモデルで検証用の query の並べ替えを評価し(学習の成立の前提条件)、
  本番の学習は評価用の query の並べ替えの順位も記録する。

In [ ]:
# ---------- 埋め込みの計算と評価 ----------
EVALUATION_BATCH = EMBED_BATCH


def embed_contexts(model, entries: np.ndarray | None = None) -> torch.Tensor:
    # 索引の passage(文を除いた形)の埋め込み(FP32、デバイス上)。entries が None なら索引全体
    if entries is None:
        tokens, lengths = CONTEXT_TOKENS, CONTEXT_LENGTHS
    else:
        tokens, lengths = CONTEXT_TOKENS[torch.as_tensor(entries)], CONTEXT_LENGTHS[entries]
    return encode_tokens_with_lengths(model, tokens, lengths, EVALUATION_BATCH, (None,))[EMBEDDING_DIMENSION]


def embed_cloze_queries(model, entries: np.ndarray) -> torch.Tensor:
    return encode_tokens_with_lengths(model, CLOZE_QUERY_TOKENS[torch.as_tensor(entries)], CLOZE_QUERY_LENGTHS[entries], EVALUATION_BATCH, (None,))[EMBEDDING_DIMENSION]


def embed_article_set(model, set_name: str) -> tuple[torch.Tensor, torch.Tensor]:
    retrieval_set = ARTICLE_LEVEL_SETS[set_name]
    queries = encode_tokens(model, torch.from_numpy(retrieval_set.queries), EVALUATION_BATCH)["by_dimension"][EMBEDDING_DIMENSION]
    passages = encode_tokens(model, torch.from_numpy(retrieval_set.passages), EVALUATION_BATCH)["by_dimension"][EMBEDDING_DIMENSION]
    return queries, passages


def evaluate_calibration(model) -> dict:
    # 検証用の集合の Recall@10(query ごとの 0/1)。索引は検証用の記事の passage だけ(文を除いた形)で、同じ索引・同じ query に 2 つのラベルを当てる:
    # passage 単位(正解は query の passage だけ)と、記事単位(正解は同じ記事の他の passage。query の passage は候補から除く)
    contexts = embed_contexts(model, VALIDATION_INDEX_ENTRIES)
    queries = embed_cloze_queries(model, VALIDATION_ENTRIES)
    positions = np.searchsorted(VALIDATION_INDEX_ENTRIES, VALIDATION_ENTRIES)
    passage_ranks = rank_single_positive(queries, contexts, positions)
    article = rank_article_label(queries, contexts, positions, ENTRY_ARTICLES[VALIDATION_INDEX_ENTRIES])
    return {"passage_hits": passage_ranks <= 10, "article_hits": article.rank <= 10}


def evaluate_main(model, keep_embeddings: bool = False) -> dict:
    # 評価用の集合: コーパス全体の索引と評価用の記事の query に、2 つのラベルを当てる(実験 A: passage 単位、実験 B: 記事単位)。
    # 診断量として、024 と同じ形式の記事単位の評価(024 の評価用の索引と query。固定長)も測る
    contexts = embed_contexts(model)
    queries = embed_cloze_queries(model, EVALUATION_ENTRIES)
    ranks = rank_single_positive(queries, contexts, EVALUATION_ENTRIES)
    article = rank_article_label(queries, contexts, EVALUATION_ENTRIES, ENTRY_ARTICLES)
    top_ids, _ = exact_search(queries, contexts, 10)
    same_article = ENTRY_ARTICLES[top_ids] == EVALUATION_QUERY_ARTICLES[:, None]
    result = {
        "passage_rank": ranks,
        "passage_hits": ranks <= 10,
        "article_rank": article.rank,
        "article_hits": article.rank <= 10,
        "same_article_top10_share": float(same_article.mean()),
        "top1_same_article_wrong": float((same_article[:, 0] & (top_ids[:, 0] != EVALUATION_ENTRIES)).mean()),
        "passage_mean_reciprocal_rank": float((1.0 / ranks).mean()),
        "article_mean_reciprocal_rank": article.mean_reciprocal_rank(),
        "article_normalized_discounted_cumulative_gain": article.ndcg_at_10,
    }
    format_queries, format_passages = embed_article_set(model, "evaluation")
    format_024 = evaluate_embeddings(format_queries, format_passages, EVALUATION_SET)
    result.update({"format024_hits": format_024.rank <= 10, "format024_mean_reciprocal_rank": format_024.mean_reciprocal_rank()})
    if keep_embeddings:
        result["contexts"], result["queries"] = contexts, queries
    return result


# ---------- 第 1 段の学習 ----------
PAIR_BATCHES: dict[tuple[int, int], object] = {}


def pair_batches(seed_index: int, num_steps: int):
    # Inverse Cloze Task の組(32 記事 x 2 組、同じ記事の組は窓が重ならない)。R1 と R2 で共通(シードとステップ数だけで決まる)
    key = (seed_index, num_steps)
    if key not in PAIR_BATCHES:
        PAIR_BATCHES.clear()  # 同時に保持するのは 1 つだけ(メモリ)
        PAIR_BATCHES[key] = sample_cloze_pair_batches(
            TRAIN_STREAM, np.asarray(TRAIN_STARTS), TRAIN_TOKEN_COUNTS, SAMPLING_WEIGHTS, SENTENCE_TABLES, num_steps, ARTICLES_PER_BATCH, PAIRS_PER_ARTICLE,
            PASSAGE_LENGTH, QUERY_LENGTH, MIN_QUERY_TOKENS, MAX_REMOVED_TOKENS, KEEP_PROBABILITY, seed=CLOZE_BATCH_SEED_BASE + seed_index,
        )
    return PAIR_BATCHES[key]


def pair_arrays(condition: str, seed_index: int, num_steps: int):
    # R2: 組の文脈をそのまま正例にする。R1: 同じ記事のもう一方の組の窓(文を除いた形)を正例にする(バッチの中で隣り合う 2 組の文脈を入れ替える)。
    # 同じバッチの query と passage のトークンの集合は R1 と R2 で同じで、正例の対応だけが違う
    batches = pair_batches(seed_index, num_steps)
    context_tokens, context_lengths = batches.context_tokens, batches.context_lengths
    if condition == "R1":
        swap = np.arange(BATCH_SIZE) ^ 1
        context_tokens, context_lengths = context_tokens[:, swap], context_lengths[:, swap]
    return batches.query_tokens, batches.query_lengths, context_tokens, context_lengths


def first_stage_batches(condition: str, seed_index: int, num_steps: int) -> VariableBatches:
    batches = pair_batches(seed_index, num_steps)
    return VariableBatches(*pair_arrays(condition, seed_index, num_steps), device, article_ids=batches.articles, mask_same_article=condition == "R1")


SAVED_STATES: dict[tuple[str, int], dict] = {}  # アップロード用(R2 と cross-encoder のシード 0)
KEPT_EMBEDDINGS: dict[str, dict] = {}  # 観察 D・E 用(R1・R2 のシード 0 の、索引と評価用の query の埋め込み。CPU)
MINED: dict[int, dict] = {}  # シード -> R2 の上位 K 件(学習用・検証用・評価用の query)と、R2 だけの Recall@10 の hits


def mine_candidates(model, contexts: torch.Tensor) -> dict:
    # R2 の上位 K 件(索引全体の全探索)。学習用・検証用・評価用の query のそれぞれについて
    mined = {}
    for name, entries in (("train", TRAIN_ENTRIES), ("validation", VALIDATION_ENTRIES), ("evaluation", EVALUATION_ENTRIES)):
        queries = embed_cloze_queries(model, entries)
        ids, _ = exact_search(queries, contexts, RERANK_CANDIDATES, chunk_size=2048)
        mined[name] = ids
        if name != "train":
            mined[name + "_hits"] = rank_single_positive(queries, contexts, entries) <= 10  # R2 だけ(並べ替えなし)の Recall@10 の hits
        del queries
    return mined


def train_first_stage(condition: str, seed_index: int, learning_rate: float, num_steps: int, main: bool, mine: bool | None = None) -> dict:
    start = time.time()
    model = build_embedding_model(seed_index).to(device)
    batches = first_stage_batches(condition, seed_index, num_steps)
    window = final_loss_window(num_steps)
    initial_losses = compute_pair_losses_from_batches(model, batches, range(num_steps - window, num_steps), TEMPERATURE, MATRYOSHKA_DIMENSIONS, USE_FP16_AUTOCAST)
    history = train_text_embedding_from_batches(
        model, batches, peak_learning_rate=learning_rate, warmup_steps=warmup_steps_for(num_steps), min_learning_rate=learning_rate * MIN_LEARNING_RATE_RATIO,
        weight_decay=WEIGHT_DECAY, gradient_clip_threshold=GRADIENT_CLIP_THRESHOLD, temperature=TEMPERATURE, matryoshka_dimensions=MATRYOSHKA_DIMENSIONS,
        use_fp16_autocast=USE_FP16_AUTOCAST, init_loss_scale=INIT_LOSS_SCALE, loss_scale_growth_interval=LOSS_SCALE_GROWTH_INTERVAL,
    )
    assert len(history["loss"]) == num_steps
    train_loss = np.array(history["loss"], dtype=np.float64)
    record = {
        "condition": condition, "seed": seed_index, "learning_rate": learning_rate, "num_steps": num_steps, "schedule_hash": history["data_stream_hash"],
        "article_hash": hashlib.sha256(np.ascontiguousarray(pair_batches(seed_index, num_steps).articles).tobytes()).hexdigest(),
        "negatives_per_row": float(batches.negatives_per_row().mean()), "truncated_query_share": float(pair_batches(seed_index, num_steps).truncated.mean()),
        "kept_share": float(pair_batches(seed_index, num_steps).kept.mean()),
        "train_loss": train_loss.astype(np.float32), "initial_train_loss": float(train_loss[:window].mean()), "final_train_loss": float(train_loss[-window:].mean()),
        "initial_model_loss_on_final_window": float(initial_losses.mean()), "finite": bool(np.isfinite(train_loss).all()),
        "skipped_steps": int(sum(history["step_skipped"])), "clip_rate": float(np.mean(history["gradient_clip_triggered"])), "final_loss_scale": float(history["loss_scale"][-1]),
    }
    record["validation"] = evaluate_calibration(model)
    do_mine = (main and condition == "R2") if mine is None else mine
    if main:
        final = evaluate_main(model, keep_embeddings=do_mine or seed_index == UPLOAD_SEED)
        contexts, eval_queries = final.pop("contexts", None), final.pop("queries", None)
        record.update(final)
        if seed_index == UPLOAD_SEED:
            KEPT_EMBEDDINGS[condition] = {"contexts": contexts.cpu(), "queries": eval_queries.cpu()}
            if condition == "R2":
                SAVED_STATES[("R2", UPLOAD_SEED)] = {k: v.detach().cpu().clone() for k, v in model.backbone.state_dict().items()}
        if do_mine:
            MINED[seed_index] = mine_candidates(model, contexts)
        del contexts, eval_queries
    elif do_mine:
        MINED[seed_index] = mine_candidates(model, embed_contexts(model))
    record["seconds"] = time.time() - start
    del model, batches
    empty_device_cache()
    return record


def first_stage_failure_reasons(record: dict) -> list[str]:
    # 第 1 段の P1(a)(6.1 節): 訓練損失がすべてのステップで有限で、最後の区間の訓練損失が、同じ組での学習前のモデルの損失の P1_LOSS_RATIO 倍以下
    reasons = []
    if not record["finite"]:
        reasons.append("訓練損失が有限でない")
    ratio = record["final_train_loss"] / record["initial_model_loss_on_final_window"]
    if not ratio <= P1_LOSS_RATIO:
        reasons.append(f"最後の区間の訓練損失 / 学習前のモデルの損失 {ratio:.3f} が {P1_LOSS_RATIO} 以下でない")
    return reasons


# ---------- cross-encoder の学習と並べ替えの評価 ----------
CROSS_ENCODER_TABLES = VariableTables(CLOZE.query_tokens, CLOZE.query_lengths, CLOZE.context_tokens, CLOZE.context_lengths, device)
CROSS_ENCODER_SCHEDULES: dict[tuple[int, int], object] = {}
UNTRAINED_VALIDATION: dict[int, np.ndarray] = {}  # シード -> 学習前の cross-encoder の検証用の query ごとの hits


def median_score_std(scores: np.ndarray) -> float:
    # 診断量: query ごとの、候補 K 件のスコアの標準偏差の中央値(定数に近い出力に崩れたモデルでは 0 に近づく。判定には使わない)
    return float(np.median(np.asarray(scores, dtype=np.float64).std(axis=1)))


def evaluate_rerank(model, entries: np.ndarray, candidate_ids: np.ndarray):
    scores = score_candidates_variable(
        model, CLOZE_QUERY_TOKENS, CLOZE_QUERY_LENGTHS, CONTEXT_TOKENS, CONTEXT_LENGTHS, entries, candidate_ids, SCORE_BATCH, USE_FP16_AUTOCAST
    )
    return rerank_single_positive(scores, candidate_ids, entries, NUM_INDEX), scores


def cross_encoder_schedule(seed_index: int, num_steps: int):
    key = (seed_index, num_steps)
    if key not in CROSS_ENCODER_SCHEDULES:
        CROSS_ENCODER_SCHEDULES[key] = sample_cloze_reranking_schedule(TRAIN_ENTRIES, MINED[seed_index]["train"], num_steps, BATCH_QUERIES, NUM_NEGATIVES, seed=CROSS_ENCODER_SCHEDULE_SEED_BASE + seed_index)
    return CROSS_ENCODER_SCHEDULES[key]


def _rerank_positions(count: int) -> np.ndarray:
    return np.arange(count) if RERANK_QUERIES is None else select_evenly(count, RERANK_QUERIES)


RERANK_EVALUATION_POSITIONS = _rerank_positions(len(EVALUATION_ENTRIES))  # cross-encoder の評価に使う query(評価用の query の中の位置。本番はすべて)
RERANK_VALIDATION_POSITIONS = _rerank_positions(len(VALIDATION_ENTRIES))
RERANK_EVALUATION_ENTRIES = EVALUATION_ENTRIES[RERANK_EVALUATION_POSITIONS]
RERANK_VALIDATION_ENTRIES = VALIDATION_ENTRIES[RERANK_VALIDATION_POSITIONS]
RERANK_EVALUATION_ARTICLES = ENTRY_ARTICLES[RERANK_EVALUATION_ENTRIES]  # ブートストラップのクラスタ(実験 C)


def train_cross_encoder(seed_index: int, learning_rate: float, num_steps: int, main: bool) -> dict:
    start = time.time()
    model = build_cross_encoder(seed_index).to(device)
    schedule = cross_encoder_schedule(seed_index, num_steps)
    window = final_loss_window(num_steps)
    initial_losses = compute_reranking_losses_variable(model, CROSS_ENCODER_TABLES, schedule, range(num_steps - window, num_steps), USE_FP16_AUTOCAST)
    if seed_index not in UNTRAINED_VALIDATION:  # 学習前の cross-encoder(ヘッドの初期化がシードで決まる)の検証用の query の hits
        UNTRAINED_VALIDATION[seed_index] = evaluate_rerank(model, RERANK_VALIDATION_ENTRIES, MINED[seed_index]["validation"][RERANK_VALIDATION_POSITIONS])[0].hits(10)
    train_start = time.time()
    history = train_reranker_variable(
        model, CROSS_ENCODER_TABLES, schedule, peak_learning_rate=learning_rate, warmup_steps=warmup_steps_for(num_steps), min_learning_rate=learning_rate * MIN_LEARNING_RATE_RATIO,
        weight_decay=WEIGHT_DECAY, gradient_clip_threshold=GRADIENT_CLIP_THRESHOLD, use_fp16_autocast=USE_FP16_AUTOCAST,
        init_loss_scale=INIT_LOSS_SCALE, loss_scale_growth_interval=LOSS_SCALE_GROWTH_INTERVAL,
    )
    train_seconds = time.time() - train_start
    assert len(history["loss"]) == num_steps
    train_loss = np.array(history["loss"], dtype=np.float64)
    record = {
        "seed": seed_index, "learning_rate": learning_rate, "num_steps": num_steps, "schedule_hash": history["data_stream_hash"],
        "train_loss": train_loss.astype(np.float32), "initial_train_loss": float(train_loss[:window].mean()), "final_train_loss": float(train_loss[-window:].mean()),
        "initial_model_loss_on_final_window": float(initial_losses.mean()), "initial_model_losses_by_step": initial_losses, "finite": bool(np.isfinite(train_loss).all()),
        "skipped_steps": int(sum(history["step_skipped"])), "clip_rate": float(np.mean(history["gradient_clip_triggered"])), "final_loss_scale": float(history["loss_scale"][-1]),
        "untrained_validation_hits": UNTRAINED_VALIDATION[seed_index], "train_seconds": train_seconds,
    }
    validation, validation_scores = evaluate_rerank(model, RERANK_VALIDATION_ENTRIES, MINED[seed_index]["validation"][RERANK_VALIDATION_POSITIONS])
    record["validation_hits"] = validation.hits(10)
    record["validation_recall"] = validation.recall_at(10)
    record["validation_coverage"] = validation.coverage()
    record["validation_score_std_median"] = median_score_std(validation_scores)
    record["validation_no_rerank_hits"] = MINED[seed_index]["validation_hits"][RERANK_VALIDATION_POSITIONS]
    if main:
        final, final_scores = evaluate_rerank(model, RERANK_EVALUATION_ENTRIES, MINED[seed_index]["evaluation"][RERANK_EVALUATION_POSITIONS])
        record["hits"], record["rank"] = final.hits(10), final.rank
        record["coverage"] = final.coverage()
        record["mean_reciprocal_rank"], record["normalized_discounted_cumulative_gain"] = final.mean_reciprocal_rank(), final.mean_normalized_discounted_cumulative_gain_at_10()
        record["score_std_median"] = median_score_std(final_scores)
        record["no_rerank_hits"] = MINED[seed_index]["evaluation_hits"][RERANK_EVALUATION_POSITIONS]
        if seed_index == UPLOAD_SEED:
            SAVED_STATES[("cross_encoder", UPLOAD_SEED)] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    record["seconds"] = time.time() - start
    del model
    empty_device_cache()
    return record


def learning_loss_statistics(record: dict) -> dict:
    # cross-encoder の P1(a)(6.1 節)の量: 最後の区間の各ステップの、訓練損失と、同じ組での学習前のモデルの損失の差 d_t(学習前より下がれば負)。
    # 差の平均 mean と標準誤差 standard_error = (d_t の標準偏差) / sqrt(区間のステップ数)。更新がなければ d_t = 0
    window = len(record["initial_model_losses_by_step"])
    differences = record["train_loss"][-window:].astype(np.float64) - record["initial_model_losses_by_step"]
    mean = float(differences.mean())
    standard_error = float(differences.std(ddof=1) / math.sqrt(window)) if window >= 2 else float("nan")
    return {"mean": mean, "standard_error": standard_error, "ratio": record["final_train_loss"] / record["initial_model_loss_on_final_window"], "final_loss": record["final_train_loss"]}


def cross_encoder_failure_reasons(record: dict) -> list[str]:
    # P1(a) を満たさない理由(空なら成立): 訓練損失が有限で、(i)差の平均 <= -max(2 x 標準誤差、数値の丸めの床)、かつ(ii)最後の区間の訓練損失 < 一様な予測の損失 ln(1 + n)。
    # 数値の丸めの床 = 16 x 実際に計算される型(FP16 の autocast なら FP16、そうでなければ FP32)の丸めの単位 x 学習前のモデルの損失の大きさ。更新がなければ差は 0 なので (i) は成り立たない
    statistics = learning_loss_statistics(record)
    rounding = 16 * (EPSILON_FP16 if USE_FP16_AUTOCAST else EPSILON_FP32) * record["initial_model_loss_on_final_window"]
    threshold = max(P1_SIGMA_MULTIPLIER * statistics["standard_error"], rounding)
    reasons = []
    if not record["finite"]:
        reasons.append("訓練損失が有限でない")
    if not statistics["mean"] <= -threshold:
        reasons.append(f"損失の差の平均 {statistics['mean']:+.3f} が -{P1_SIGMA_MULTIPLIER:g} x 標準誤差({threshold:.3f})以下でない")
    if not record["final_train_loss"] < UNIFORM_LOSS_CROSS_ENCODER:
        reasons.append(f"最後の区間の訓練損失 {record['final_train_loss']:.4f} が ln(1 + n) = {UNIFORM_LOSS_CROSS_ENCODER:.4f} 未満でない")
    return reasons


# ---------- 判定と統計 ----------
def judge(delta: float, sigma: float) -> str:
    threshold = SIGMA_MULTIPLIER * sigma
    if delta > threshold:
        return "支持"
    if delta < -threshold:
        return "反証"
    return "判定不能"


def combined_sigma(per_seed: np.ndarray, bootstrap_contrast: np.ndarray) -> dict:
    # シードが 1 つのときは、シード間の項を 0 とする
    seed_variance = float(np.var(per_seed, ddof=1)) / len(per_seed) if len(per_seed) >= 2 else 0.0
    bootstrap_variance = float(np.var(bootstrap_contrast, ddof=1))
    return {"sigma": math.sqrt(seed_variance + bootstrap_variance), "seed_term": math.sqrt(seed_variance), "bootstrap_term": math.sqrt(bootstrap_variance)}


VALIDATION_WEIGHTS = cluster_bootstrap_weights(ENTRY_ARTICLES[VALIDATION_ENTRIES], BOOTSTRAP_RESAMPLES, BOOTSTRAP_SEED + 1)  # 検証用の query(2 つのラベルで共通)
RERANK_VALIDATION_WEIGHTS = cluster_bootstrap_weights(ENTRY_ARTICLES[RERANK_VALIDATION_ENTRIES], BOOTSTRAP_RESAMPLES, BOOTSTRAP_SEED + 3)  # cross-encoder の検証用の query


def bootstrap_mean(weights: np.ndarray, values: np.ndarray) -> np.ndarray:
    # query ごとの値の平均の、記事を単位とするクラスタブートストラップの再標本の分布 (反復,)
    return (weights @ np.asarray(values, dtype=np.float64)) / weights.sum(axis=1)


def calibration_metric(validation: dict) -> float:
    # 第 1 段の較正の指標 M: 検証用の passage 単位と記事単位の Recall@10 の平均(どちらの定義にも偏らない)
    return 0.5 * (float(validation["passage_hits"].mean()) + float(validation["article_hits"].mean()))


def calibration_metric_deviation(validation: dict) -> float:
    boot = 0.5 * (bootstrap_mean(VALIDATION_WEIGHTS, validation["passage_hits"]) + bootstrap_mean(VALIDATION_WEIGHTS, validation["article_hits"]))
    return float(boot.std(ddof=1))


# ---------- 較正の選択(6.1 節) ----------
def select_calibrated_index(points: list[float], values: list[float], deviations: list[float], margins: list[float], eligible: list[bool]) -> int | None:
    # 指標が最大の点から SELECTION_SIGMA_MULTIPLIER x 標準偏差以内の候補(P1 を満たす点だけ)のうち、学習の成立の余裕(margins。大きいほど余裕がある)が最大の点。
    # 同点なら小さい学習率。候補がなければ None
    candidates = [i for i, ok in enumerate(eligible) if ok]
    if not candidates:
        return None
    best = max(candidates, key=lambda i: (values[i], -points[i]))
    near = [i for i in candidates if values[best] - values[i] <= SELECTION_SIGMA_MULTIPLIER * deviations[best]]
    return max(near, key=lambda i: (margins[i], -points[i]))


def bootstrap_rerank_gain(records: list[dict]) -> dict:
    # cross-encoder の P1(b): 検証用の query での Recall@10 の、学習前の cross-encoder からの増分(シードごと g_s)のシード平均 gain と、
    # 判定と同じ式の標準偏差 sigma = sqrt(Var_s(g_s) / n + sigma_boot^2)
    difference = np.stack([r["validation_hits"].astype(np.float64) - r["untrained_validation_hits"].astype(np.float64) for r in records])
    per_seed = difference.mean(axis=1)
    resampled = bootstrap_mean(RERANK_VALIDATION_WEIGHTS, difference.mean(axis=0))
    sigma = combined_sigma(per_seed, resampled)
    return {"gain": float(per_seed.mean()), "per_seed_gain": per_seed, **sigma}

## 6. 実験 / Experiments

### 6.1 実験宣言セル: 共通の設定・検証すること・判定基準・前提条件

**この節の内容は本番実行の前に確定させ、結果を見た後に変更しない。**

#### 共通の設定

- **モデル**: 008 と同じ小型 GPT(4 層、$d_{\mathrm{model}} = 256$、ヘッド数 8、RoPE、RMSNorm、SwiGLU の順伝播ネットワーク(中間次元 683)、正規化前置、埋め込みと出力層の重み共有、系列長 256、語彙サイズ 8192)。
  第 1 段(R1・R2)は encoder として使い、query と passage で重みを共有する。最終正規化層の後の隠れ状態の平均プール(右側のパディングを除く)を L2 正規化したものを埋め込みとする(射影層なし、次元 $d = 256$)。**全パラメータを更新する。**
- **データ**: 5.3 節のとおり。コーパスの先頭の 356 記事を除く 9,470 記事を、記事を単位に学習用(9,070 記事)・検証用(100 記事)・評価用(300 記事)に分ける([024](./024_text_embedding_and_retriever.ipynb) と同じ分割)。
  **学習用・検証用・評価用のすべてが、008 の事前学習とトークナイザの学習の範囲(参照コーパス)の外にある**(5.3 節のセルで、文字位置の範囲のアサーションにより確かめる)。
- **第 1 段の学習**: 損失は in-batch negatives の InfoNCE(query から passage への方向、温度 $\tau = 0.05$ で固定)と Matryoshka Representation Learning($\mathcal{M} = \{16, 32, 64, 128, 256\}$、重み $c_m = 1 / \lvert \mathcal{M} \rvert$)で、
  024 の条件 C5 と同じ損失の形。AdamW(重み減衰 0.1、行列形の重みにのみ掛ける)、バッチ 64 組(**32 記事 × 各記事 2 組**。同じ記事の組を意図的に同じバッチに入れる。損失での扱いだけを定義に合わせる: R1 は同じ記事の組の passage を負例から除き(1 行あたりの負例 62 個)、R2 は負例として数える(63 個))、最初の 10% のステップで線形 warmup の後 cosine で最大値の 0.01 倍まで減衰、gradient clipping の閾値 1.0。
  CUDA では FP16 の autocast と動的損失スケーリング。埋め込みの正規化・類似度・損失・評価は FP32。**学習ステップ数は $T_1 = 1024$**(全条件・全段階で共通。024 の C5 と同じ)。**query は、どちらの条件も「窓から抜いた 1 文」で、文を文脈に残す確率 $r = 0.1$ も共通。** 正例は、R2 が query を抜いた窓の文脈、R1 が同じ記事のもう一方の窓の文脈(3.2 節)。
- **cross-encoder**: [025](./025_ann_search_and_reranking.ipynb) と同じ構成(008 の重みから始め、列`[query] [区切り] [passage] [終端]`を因果マスクのもとで符号化して、終端の位置の出力に線形のヘッドを付ける。全パラメータを更新する)。
  損失は 1 個の正例と 7 個の負例の 8 個の候補の上の softmax 交差エントロピー(in-batch negatives は使わない)。1 ステップ 8 個の query。学習率の形と gradient clipping は第 1 段と同じ。
  **学習ステップ数 $T_2$** は、候補 $(2048, 4096, 8192)$ から 6.3 節のパイロットで自動で選ぶ。
- **シード**: シード $s$ が動かすのは、記事の並び・窓・抜く文・文を文脈に残す組(第 1 段は $26100 + s$。R1 と R2 で同一)と、cross-encoder の組($26500 + s$)・ヘッドの初期化($26200 + s$)である。R1・R2 の初期値はシードによらず 008 の重みで同一。
  同じシードの R1 と R2 は同じ記事・窓・抜く文を見る(6.12 節で確かめる)。cross-encoder のシード $s$ は、同じシード $s$ の R2 の第 1 段の上位 $K$ 件で学習し、評価する。
- **評価**: Recall@10(3.5 節)。学習の最終ステップの重みで、FP32、全件の内積で順位をつける。$R_s(c)$ を条件 $c$・シード $s$ の Recall@10 とする。
  **実験 A・B は同じ索引・同じ query** で、関連のラベルだけを変える。索引はコーパス全体(すべての passage を、抜く文を 1 つ固定して除いた形にしたもの。対象の passage 約 9.4 万個、動作確認では約 5 千個)で、
  query は評価用の記事の passage から取り除いた文(約 3,200 個。記事単位のラベルの正解が 1 つ以上ある query に限る)。**実験 A** は passage 単位のラベル(正解は query の passage だけ)、
  **実験 B** は記事単位のラベル(正解は同じ記事の他の passage。query の passage 自身は候補から除く。Recall@10 は上位 10 件に正解が 1 つ以上ある query の割合。024 と同じ定義)。
  診断量として、024 と同じ形式の記事単位の評価(024 の評価用の索引と query、3,244 個。固定長)を、R1・R2・024 の Hub のモデルで測る(判定には使わない)。
- **008 の重みは 1 つのチェックポイントである。** 実験 A・B の結論は、その 1 つのチェックポイント(`kojikojiprg/ai-theories-small-gpt-en`の`main`)から始めた場合に限る。

#### 条件の対応表

| 記号 | 正例の定義 | 学習の組 | 並べ替え | 使う実験 |
|---|---|---|---|---|
| R1 | 記事単位 | query = 窓から抜いた文。正例 = 同じ記事のもう一方の窓の、文を除いた形。同じ記事の組を負例から除く | なし | 実験 A の基準、実験 B の条件 1 |
| R2 | passage 単位 | Inverse Cloze Task: query = 窓から抜いた文。正例 = その窓の文脈(10% は文を残す)。同じ記事の組を負例として数える | なし | 実験 A の条件 2、実験 B の条件 2、実験 C の基準 |
| R3 | passage 単位 | R2 の上位 50 件を cross-encoder で並べ替える | cross-encoder | 実験 C の条件 |

数式の添字の R1・R2 はこの表の記号である。

#### 揃えた量と揃えなかった量

| 量 | 扱い |
|---|---|
| 起点(008 の`main`)・モデルの構成・総パラメータ数・$T_1$・バッチサイズ・温度・損失の形・重み減衰・学習率スケジュールの形・gradient clipping の閾値 | **揃える**(R1 と R2 で同じ) |
| 記事の並び・窓・抜く文・文を残す組・バッチ(32 記事 × 2 組。同じシードの R1 と R2) | **揃える**(同じバッチの query と passage のトークンは R1 と R2 で同じ) |
| 入力の形式(query = 抜いた文、passage = 文を除いた形)・評価の索引・query・評価の手順 | **揃える**(R1 と R2 は同じ索引・同じ query で評価し、関連のラベルだけを変える) |
| 学習率 | **揃えない**(条件ごとに較正する。下の「学習率の較正」) |
| 正例の対応と、同じ記事の組の扱い(正例 + in-batch の負例) | **揃えない**(関連性の定義を、正例と in-batch の負例の両方に一貫して当てはめる。これが実験 A・B の介入) |
| 1 行あたりの負例の数 | **揃えない**(R1 は 62 個、R2 は 63 個。同じ記事の組を除くことの結果で、介入に含まれる) |
| cross-encoder の追加の学習量 | **実験 C では揃えない**(実用の問い。実験 C の「解釈の注意」) |

#### 学習量についての注意

1. **エポック数**: 組を作れる学習用の記事は 8,639 本(141,126,568 トークン)で、1 ステップは 64 個の窓($64 \times 128 = 8{,}192$ トークン)を見る(query は窓の一部)。$T_1 = 1024$ は **約 0.059 エポック分** で、学習用の記事のトークンの大半は一度も見ない
   (5.3 節の出力)。窓は記事のランダムな位置から引くので、同じ passage を繰り返し見ることは稀である。
2. **学習が飽和していない状態での比較になりうる**: 024 の C5 は $T_1 = 1024$ でも検証用の Recall@10 が学習の最後まで伸びていた。実験 A・B の結果は、この学習量での比較である。
3. **cross-encoder の学習量**: $T_2 = 2048, 4096, 8192$ は、8 query/ステップで見る query が 16,384・32,768・65,536 個で、学習に使える query(正例が上位 50 件に入る query)の数に対するエポック数は 5.3 節・6.3 節の出力に印字する。
   学習量を下げる段階(「実行計画」)が選ばれうるのは第 1 段のシード数・cross-encoder のシード数・学習率の較正の方式などで、$T_1$ と $T_2$ は段階では下げない。
   ただし、$T_2$ の候補のうち小さいものが選ばれる(パイロットで早く立ち上がる)場合は、学習が飽和していない状態での比較になる。

#### 学習率の較正(本番の冒頭、6.5 節)

- **格子**: 第 1 段は R1・R2 とも、中心 $4.8 \times 10^{-4}$(024 の C5 の較正で選ばれた値)の $\{1/4, 1, 4\}$ 倍(公比 4 の 3 点: $1.2 \times 10^{-4}$、$4.8 \times 10^{-4}$、$1.92 \times 10^{-3}$)。
  cross-encoder は、中心 $2.4 \times 10^{-4}$(025 の学習のパイロットで最後の区間の訓練損失が最も低かった学習率)の $\{1/4, 1, 4\}$ 倍($6 \times 10^{-5}$、$2.4 \times 10^{-4}$、$9.6 \times 10^{-4}$)。
  公比を 4 にした理由は、指標が学習率に対して緩やかにしか変わらず、公比 2 の格子では隣の点との差が評価の変動より小さくなって、最良が観測のノイズで端になりやすいためである(024 と同じ。観測の結果の向きに依らない、格子の刻みについての一般論による選択)。
- **較正のスケール**: 較正は **本番と同じスケール**(第 1 段は $T_1 = 1024$、cross-encoder は選ばれた $T_2$)で、**較正専用のシード(番号 90。実験のシードと共有しない)** の 1 シードで各格子点を学習して行う。評価用の集合は較正に使わない。
- **指標**: 第 1 段は、検証用の記事の passage だけを索引にした **同じ索引・同じ query** での、passage 単位のラベルと記事単位のラベルの Recall@10 の **平均** $M$。R1 と R2 の両方を同じ指標で較正する(どちらのラベルにも偏らない指標。一方だけに自分のラベルの指標を使うと、実験 A・B の対比量が、自分のラベルで較正した側に有利に偏るため)。
  cross-encoder は、検証用の query の並べ替えの Recall@10(R2 のシード 90 の上位 50 件を並べ替える)。
- **最良と次点の差が評価の変動より小さい場合の選び方(本番の前に宣言)**: 指標が最大の点から、**指標の標準偏差(記事を単位とするブートストラップ)の 1 倍以内** の候補(学習の成立の前提条件 P1(a) を満たす点だけ)のうち、
  **学習の成立の余裕が最大の点** を選ぶ。余裕は、第 1 段では「1 − 最後の区間の訓練損失 ÷ 同じ組での学習前のモデルの損失」、cross-encoder では「最後の区間の損失の差の平均(学習後 - 学習前)を標準誤差で割った値の符号を反転したもの」(どちらも大きいほど学習前より損失が下がっている)。
  同点なら小さい学習率。P1(a) を満たす点が 1 つもないときは、余裕が最大の点を選び、P1 の不成立として進める(その条件が入る実験は「前提不成立」になる)。
- **拡張の規則**: 選ばれた点が格子の端に来た場合は、その方向に公比 4 で 1 点だけ格子を拡張して学習し、改めて同じ規則で選ぶ(1 回のみ)。
- **P0(較正)**: その実験の対比量に入る条件のうち、自分自身を較正した条件で、選んだ学習率が格子の **内点** であること(拡張後も端なら不成立)。規則で決めた条件(下の段階 4・5)は対象外。
- **規則で学習率を決める段階(削る段階 4・5)**: 段階 4 以降は、R1 の較正をやめ、**024 の C5(同じ構成・同じ $T_1$・同じ起点。組の作り方は違う)の較正で選ばれた $4.8 \times 10^{-4}$** を使う。
  段階 5 以降は、cross-encoder の較正をやめ、中心の $2.4 \times 10^{-4}$ を使う。規則で決めた条件は P0 の対象外になる。**最適からずれたときの対比量の偏りの向き**:
  - R1(段階 4 以降): $4.8 \times 10^{-4}$ は 024 の組の作り方で較正した値で、本トピックの R1 の組の作り方(query が抜いた文、同じ記事の組を負例から除く)での最適とは限らない。R1 の学習率が最適からずれていると、R1 の Recall@10 が下がる。
    **実験 A の対比量は正の向き(仮説に有利)、実験 B の対比量は負の向き(仮説に不利)に偏る**。したがって実験 A の **支持** と実験 B の **反証・判定不能** は、偏りの向きを考慮して読む(A の支持は偏りで過大になりうる。B の支持は偏りがあっても成り立つ結果として読む)。
  - cross-encoder(段階 5 以降): 学習率が最適からずれていると、並べ替えの Recall@10 が下がり、**実験 C の対比量は負の向き(仮説に不利)に偏る**。支持は偏りがあっても成り立つ結果として読み、判定不能は隠れた正の効果を排除しない。反証は、並べ替えが劣ることと学習率の不足を区別できない。
  段階 4・5 を使わないとき(段階 0〜3)は、この制限はない。
- **較正の段階で、学習の成立の余裕が全格子点で小さい条件を検出した場合**(cross-encoder で、すべての格子点で P1(a) が不成立): その条件が入る実験(実験 C)は、P1 の不成立として「前提不成立」で進める。本番は 1 回で完結させるので、較正の出力を見てから設計を変えない。

#### 共通の前提条件

前提条件は、検証したい仮説(対比量の向き)とは独立な量で定義する。成立しなかった実験は、支持・反証・判定不能のいずれとも判定せず「前提不成立」として記録する。

- **P0(較正)**: 上の「学習率の較正」のとおり。対象は、実験 A: R1・R2、実験 B: R1・R2、実験 C: R2・cross-encoder(規則で決めた条件は対象外)。
- **P1(学習の成立)**:
  - **第 1 段(実験 A・B)**: (a)対象の条件の全シードで、訓練損失がすべてのステップで有限であり、**最後の 5% のステップの訓練損失が、同じステップの組での学習前のモデルの損失の 0.9 倍以下**。
    (b)対象の条件の全シードで、**自分の定義のラベルの検証用の Recall@10(R1: 記事単位のラベル、R2: passage 単位のラベル。検証用の記事の passage だけの索引・同じ query)が、学習前の 008(同じ構成)の同じ集合での値より 0.04 以上高い**。
  - **cross-encoder(実験 C)**: (a)全シードで、訓練損失が有限で、最後の 5% のステップの訓練損失が、同じ組での学習前のモデルの損失より **標準誤差の 2 倍以上低く**、かつ一様な予測の損失 $\ln(1 + n) = \ln 8$ 未満。
    (b)検証用の query の並べ替えの Recall@10 の、学習前の cross-encoder からの増分のシード平均が、**判定と同じ式の標準偏差 $\sigma_g$ の 2 倍以上**。
  すべて、**更新なしの状態では成立しない量** で定義する: (a)の比は更新がなければ厳密に 1(差は 0)、(b)の増分は更新がなければ 0 である(学習率 0 の学習で不成立になることは 5.4 節の単体テストで確かめる)。
- **P2(改善の余地・天井と床)**: 実験 A・B は、基準となる条件の Recall@10 が **床 $0.02$ 以上、天井 $0.95$ 以下**(実験 A: $R(\mathrm{R1})$ の passage 単位のラベル、実験 B: $R(\mathrm{R1})$ の記事単位のラベル)。差ではなく基準となる条件の値で定義する。
  床の 0.02 は、passage 単位の評価でランダムな順位の期待値(約 0.0001)の約 200 倍で、第 1 段が検索として働いていることの最低限の確認である。実験 C は、$R(\mathrm{R2})$ が coverage より 0.05 以上低いこと。
- **P3(実験 C の第 1 段の候補)**: 第 1 段(R2)の上位 $K = 50$ 件に正例が入る割合(coverage)が **0.35 以上**、かつ coverage − $R(\mathrm{R2})$(並べ替えの余地)が **0.10 以上**(全シードの平均)。

**閾値の根拠**(観測の結果の向きに依らない一般論による):

- **P1(a)の 0.9**: 024 の C5($T_1 = 1024$、中心の 1/4・1・4 倍の学習率)の最後の区間の訓練損失 ÷ 学習前のモデルの損失は 0.538〜0.613 で、学習が進んでいない場合は 1(更新がない場合は厳密に 1)。0.9 は、その間にあり、
  024 の C2 の $T = 256$ の 5 シードの標準偏差 0.011(ばらつきが最大の見込みの値)に対して、正常な最大の 0.613 から約 26 標準偏差、学習が進んでいない場合の 1 から約 9 標準偏差離れている。**測っていない範囲**: R1・R2 の比(組の作り方が 024 の C5 と違う)と、R1・R2 の格子の外側(拡張した点)での比。比が 0.9 と 1 の間に来る場合は、学習が弱いと判定され、前提不成立になる。R1 は同じ記事の組を負例から除くので、比の分母(学習前のモデルの損失)も違う。
- **P1(b)の 0.04**: 024 の C2 の検証用の Recall@10 の学習前からの増分(+0.1373〜+0.1599、$T = 1024$)と、増分が 0 の学習が進んでいない場合の間に置いた値(024 の閾値と同じ)。**測っていない範囲**: R1・R2 の、自分のラベルでの増分(組の作り方と評価のラベルが 024 と違う)。
- **cross-encoder の P1(a)の標準誤差 2 倍**: 025 と同じ(最後の区間の差の平均の標準誤差)。更新がなければ差は 0 で、(a)は成り立たない。
- **P2 の床 0.02・天井 0.95、P3 の 0.35・0.10**: 025 の第 1 段(記事単位の定義の 024 のモデル、コーパス全体 94,486 passage)で、上位 50 件の coverage が 0.4094、並べ替えなしの Recall@10 が 0.2199 だった値を、P3 の下限(0.35)と余地(0.10)の目安にした
  (passage 単位の R2 の値は事前に分からない。測っていない範囲)。

#### 前提条件の閾値の下限と、想定される増分(評価の変動の項から)

cross-encoder の P1(b)は「増分のシード平均 $\ge 2\sigma_g$」の形で、$\sigma_g = \sqrt{\mathrm{Var}_s(g_s) / n + \sigma_{\mathrm{boot}}^2}$ のブートストラップの項 $\sigma_{\mathrm{boot}}$ は、シード数 $n$ を増やしても小さくならない。
閾値は常に $2\sigma_{\mathrm{boot}}$ 以上である。評価の変動の項の見積もりは、025 の同じ大きさの検証用の query(877 個、100 記事)での値を使う: 学習した dual encoder の増分(5 シード平均)で 0.0070、ランダムな負例で学習した cross-encoder で 0.0137、パイロット(1 シード)で 0.0176。
本トピックの検証用の query は passage 単位で約 1,000 個(100 記事)で、同程度の大きさなので、$\sigma_{\mathrm{boot}} \approx 0.007$〜$0.018$、閾値の下限は **約 0.014〜0.035** である(学習は不要な計算)。

**想定される増分との比較**: 025 で、学習した dual encoder の増分は +0.018(閾値 0.014 に対して成立)、ランダムな負例で学習した cross-encoder の増分は +0.0176 で、ブートストラップの項 0.0137 の 2 倍(0.0274)に届かず不成立だった。この下限は、増分が小さい状態を検出するために置いている。
並べ替えの余地は、coverage − 学習前の cross-encoder の Recall@10 で、025 では 0.47 − 0.18 = 約 0.29 だった。passage 単位の R2 の coverage は事前に分からないが、cross-encoder が学習の余地の 12% 以上
(0.035 ÷ 0.29)を実現すれば、閾値の上端でも成立する。学習が成立すれば超える見込みで、025 のように学習が立ち上がらなければ超えない。**パイロットの規則(6.3 節)は、この増分が閾値を超えるかを、本番の $T_2$ の選択に使う。**

#### 判定の形と標準偏差の単位

すべての実験で、対比量 $\Delta$ は **同じシードどうしで対応をとった差 $d_s$ のシード平均** である。標準偏差は

$$
\sigma = \sqrt{\frac{\mathrm{Var}_s(d_s)}{n} + \sigma_{\mathrm{boot}}^2}
$$

とする($n$ はシード数、$\mathrm{Var}_s$ はシード間の不偏分散)。**導出**: $d_s = \delta + a_s + e$ と書く。$\delta$ は真の効果、$a_s$ は学習の乱数(記事の並びと組、cross-encoder ではヘッドの初期化)による変動、$e$ は評価用の query が有限であることによる変動である。
評価用の query は全シードで共通なので、$e$ はシードによらず同じ値をとり、シード間の分散には現れない。したがってシード平均の分散は $\mathrm{Var}(a_s)/n + \mathrm{Var}(e)$ で、第 1 項をシード間の分散から、第 2 項を評価用の query の再標本化から見積もる。
$\sigma_{\mathrm{boot}}$ は、**記事を単位とする対応付きのクラスタブートストラップ**(015・024 と同じ方式。同じ記事から作った query どうしは独立でないので、記事を復元抽出する。比べる全条件・全シードに同じ再標本を使う)で、
対比量(シード平均)を反復ごとに作り直したときの標準偏差である。Recall@10 は query ごとの 0 か 1 なので、各反復の値は選ばれた記事の query の hit の割合である。

判定は 3 値で行う(閾値は $2\sigma$): **支持** は $\Delta > 2\sigma$、**反証** は $\Delta < -2\sigma$(期待と逆の向きの差が閾値を超えて現れた)、**判定不能** は $\lvert \Delta \rvert \le 2\sigma$(ノイズ床の 2 倍以内)。

#### 実行計画(削る段階)と、その自動選択

本番は Colab の 1 セッション(T4 で 120 分)で完結させる。本番の冒頭で測った時間(6.2 節)と、cross-encoder のパイロット(6.3 節。その時間も予算に含める)の後の経過時間から、次の 8 通りの計画の残りの実行時間を見積もり(6.4 節)、
**予算に収まる計画のうち番号の最も小さいもの** を選ぶ。選択は見積もりと経過時間のみに基づき、どの実験の結果も参照しない。最も下位の計画でも収まらなければ、本番の学習の前に停止する。

**見積もりの安全係数**: 学習(第 1 段・cross-encoder)の時間の見積もりには、本番の冒頭で測った 1 ステップの時間に **1.1 倍の安全係数** を掛ける(パイロットの学習は、実際に使った時間を経過時間に入れるので、見積もりに入れない)。
理由: 025 の再実行に向けたパイロットで、T4 の実際の 1 ステップ(80〜81 ms)が本番の冒頭の計測値(75 ms)より約 7% 長かったため、計測値の外れを吸収する。学習以外の時間(評価・採掘・HNSW の構築・固定の余裕)には掛けない。

**削る段階**(番号が大きいほど多く削る。各段階は前の段階に 1 つだけ変更を加える。シード数は、削る前 $(5)$、中 $(3)$、低 $(2)$ の 3 水準):

| 段階 | 変更 | 観察 D の索引の件数 | cross-encoder のシード数 | 第 1 段のシード数 | R1 の較正 | cross-encoder の較正 |
|---|---|---|---|---|---|---|
| 0 | (削らない) | 16384 | 5 | 5 | する | する |
| 1 | 観察 D の索引の件数を 1 段下げる | 8192 | 5 | 5 | する | する |
| 2 | cross-encoder のシード数を中にする | 8192 | 3 | 5 | する | する |
| 3 | 第 1 段のシード数を中にする | 8192 | 3 | 3 | する | する |
| 4 | R1 の較正をやめ、024 の値にする | 8192 | 3 | 3 | しない | する |
| 5 | cross-encoder の較正をやめ、中心の値にする | 8192 | 3 | 3 | しない | しない |
| 6 | cross-encoder のシード数を低にする | 8192 | 2 | 3 | しない | しない |
| 7 | 第 1 段のシード数を低にする | 8192 | 2 | 2 | しない | しない |

**この順にした理由**:

1. **観察 D を最初に削る**: 判定を設けない観察で、削っても判定に影響しない。
2. **cross-encoder のシード数を次に削る**(段階 2): cross-encoder の 1 シードあたりの学習が最も長い($T_2$ ステップ)ので、削ったときの時間の節約が大きい。第 1 段のシード数以下に保つ。
3. **第 1 段のシード数を削る**(段階 3): 実験 A・B・C のシード数が減るが、対比量の標準偏差は記事を単位とするブートストラップの項が支配的な見込み(024 の実験 A・B・D で、ブートストラップの項がシード間の項の約 1.7〜2.3 倍)で、5 から 3 に減らしたときの閾値の増え方が小さい。
4. **学習率の較正をやめる**(段階 4・5): 較正は学習を条件ごとに最大 4 回行うので、時間の節約が大きいが、規則で決めた学習率が最適からずれたときの対比量の偏りの向きが条件によって逆になる(上の「学習率の較正」)。**較正の範囲を削る段階** として、(1)規則で決めた条件は P0 の対象外になること、(2)偏りの向きを上に記した。
   支持を誤らせうる偏り(実験 A の R1)を持つ段階 4 を、仮説に不利な向きの偏り(実験 C の cross-encoder)を持つ段階 5 より先にした。
5. **シード数を最後に「低」にする**(段階 6・7): シード数 2 は、シード間の分散を推定できる最小の数である。

**$T_1$ と $T_2$ は段階では下げない**: $T_1 = 1024$ は 024 の C5 の再現(R1)の前提で、下げると R1 の再現の確認が意味を持たなくなる。$T_2$ は、パイロットが決める学習の成立の量であり、予算のために下げると、前提条件が成立しない状態を作る。

**パイロットの分岐の規則(本番の前に宣言。6.3 節)**: 学習を伴うパイロットは、本番と同じセッションの冒頭で、単一の条件の量だけを測って行う。

1. **第 1 段のパイロット**: R2 を、較正専用のシード 90・$T_1 = 1024$・中心の学習率 $4.8 \times 10^{-4}$ で学習し、第 1 段の上位 $K = 50$ 件を求める(学習用・検証用・評価用の query)。この学習は、R2 の較正の格子の中心の点として **使い回す**(同じ設定であることをアサーションで確かめる)。
2. **cross-encoder のパイロット**: シード 90 の上位 50 件で、cross-encoder を、中心の学習率 $2.4 \times 10^{-4}$・較正専用のシード 90 で、$T_2 = 2048, 4096, 8192$ の順に学習する。各 $T_2$ の後に、検証用の query で次の 3 つを調べ、**3 つがすべて成立した最初の $T_2$ で止める**。
   - (i)学習の成立 P1(a)(最後の区間の訓練損失が、同じ組での学習前のモデルの損失より標準誤差の 2 倍以上低く、かつ $\ln 8$ 未満)。
   - (ii)検証用の query の並べ替えの Recall@10 の、学習前の cross-encoder からの増分が、記事を単位とするブートストラップの標準偏差(増分の再標本の標準偏差)の 2 倍以上(P1(b) の閾値の式の、シード間の項がない 1 シード版)。
   - (iii)検証用の query の並べ替えの Recall@10 が、**R2 だけ(並べ替えなし)の値を上回る**(点推定)。
3. **本番の $T_2$** は、(i)〜(iii)が成立した最小の候補。3 つの候補のどれでも成立しなければ、**最大の候補(8192)を使い**、実験 C は事前の規則のとおり進める(前提条件の成否は本番の結果で決まる。パイロットが失敗しても実験 C を中止しない: パイロットは 1 シード・1 つの学習率の量で、中止すると実験 C の情報が得られないため)。
4. パイロットの時間は予算に含め、パイロットの後の経過時間で、削る段階を選び直す。パイロットで学習した cross-encoder の最後の 1 回は、同じ設定(シード 90、選ばれた $T_2$、中心の学習率)の較正の格子の中心の点として使い回す。

**パイロットの分岐に下流の指標((iii)の Recall@10)を含める理由と、その影響**: 025 では、訓練損失が下がっても、並べ替えの Recall@10 は何もしない場合を上回らなかった(損失が下がることと、下流の指標が基準を上回ることは別)。
訓練損失だけで $T_2$ を決めると、損失が下がっても役に立たないモデルに予算を使う。(iii)を含めると、選ばれた $T_2$ は「検証用の query で、1 シードの cross-encoder が R2 だけを上回った」最小の学習量になる。
影響: $T_2$ の選択に、検証用の query(判定には使わない集合)の下流の指標を使うので、本番の cross-encoder が、実験 C に有利な向きに $T_2$ を選ばれている(選択の偏り)。偏りは $T_2$ の 3 通りの選択に限られ、評価用の query と別のシード・別の集合での判定なので、
実験 C の支持は「このセッションのパイロットで R2 を上回る見込みのある学習量で、評価用の query でも上回ったか」として読む。(i)〜(iii)がどれも成立しなかった場合は、選択の偏りがなく、前提不成立または反証・判定不能の見込みが高い。

#### 実験 A: passage 単位のラベルの評価での、関連性の定義の効果

**検証すること**: 同じ索引・同じ query の passage 単位のラベルの評価(正解は query の passage だけ)で、passage 単位の定義(Inverse Cloze Task)で学習した検索器(R2)の Recall@10 が、記事単位の定義で学習した検索器(R1)より高いか。

**条件**: R1 と R2。どちらも 008 の重みから始め、平均プール・因果マスク・Matryoshka Representation Learning の損失・同じ記事・同じ窓・同じ抜く文・同じ入力の形式・同じ $T_1$ で、
**違いは関連性の定義だけ** である(正例の対応と、in-batch の同じ記事の組の扱い。3.2・3.3 節)。

**対比量**:

$$
\Delta_A = \frac{1}{n} \sum_{s} \left( R_s(\mathrm{R2}) - R_s(\mathrm{R1}) \right)
$$

(正を期待。$n$ は実験 A のシード数、$R_s(c)$ はシード $s$ の条件 $c$ の passage 単位のラベルでの Recall@10。)標準偏差 $\sigma_A$ は「判定の形」のとおり、$d_s = R_s(\mathrm{R2}) - R_s(\mathrm{R1})$ のシード間の分散と、記事を単位とするクラスタブートストラップの分散を合成する。

**判定基準**: $\Delta_A > 2\sigma_A$ なら支持、$\Delta_A < -2\sigma_A$ なら反証、それ以外は判定不能。

**介入の直接の作用点と、対比量までの距離**: 介入(関連性の定義)が直接作用するのは、**学習の信号の中身** である。正例の対応と、in-batch の負例のうち同じ記事の組をどう扱うかが、その結果として「同じ記事の中の passage どうしを区別できるか」に現れる。
Recall@10 は、その埋め込みの内積による順位づけと上位 10 件での二値化という 2 段階だけ下流の量で、作用点に近い。作用点に近い診断量として、(1)上位 10 件のうち、**query の正例と同じ記事の passage の割合**、
(2)1 位が **同じ記事の別の passage** である query の割合(話題は合っているのに passage を取り違えた割合)、(3)閾値を使わない順位の量として平均逆順位(Mean Reciprocal Rank)を併記する(判定には使わない)。

**対比量が閾値を超えうる状態と、仮説との整合**: $\Delta_A > 2\sigma_A$ となりうる状態は次のとおり。(i)R2 が同じ記事の passage を区別する表現を学んだ(仮説どおり)。(ii)R1 の学習が進んでいない(P1 で検出して前提不成立にする)。
(iii)R1 の学習率が最適からずれている(段階 4 以降。上の「学習率の較正」のとおり、仮説に有利な向きの偏り)。(iv)R1 の値が上限に近く差が出ない(P2 で検出する。これは支持ではなく判定不能・前提不成立の側)。
入力の形式・索引・query は両条件で同じなので、「各条件が学習で見た形式の評価で勝つ」経路はない。ただし、評価の passage 単位のラベルは、R2 の学習の目標(正例が 1 つの passage)と一致し、R1 の目標(同じ記事のどの passage でも正解)とは一致しない。
この一致は **関連性の定義そのものの帰結** であり、介入の効果に含まれる(実験 A は、各定義で学習した検索器が、その定義のラベルで評価されたときに優れるかを見る)。

**解釈の注意(R2 が同じ記事の困難な負例を見ること)**: 本トピックは、in-batch negatives に同じ記事の組を意図的に入れ、R2 はそれを負例として数える。R2 が同じ記事の passage を区別する学習の信号を得ることは、
**実験 A の対比量を正の向きに大きくしうる**。これは「関連性の定義を、正例と in-batch の負例の両方に一貫して当てはめる」という介入の一部で、設計どおりの効果である。
一方、R1 は同じ記事の組を負例から除くので、負例の数が 1 行あたり 62 個と R2(63 個)より 1 つ少ない。この差は小さく、介入に含まれる(6.6 節で数を印字する)。

**前提条件**: P0(R1・R2)、P1(R1・R2)、P2($R(\mathrm{R1})$ の passage 単位のラベルが $[0.02, 0.95]$)。

**診断量(判定なし)**: 上の(1)〜(3)、R1・R2 の最後の区間の訓練損失、学習率とその出どころ、clipping の発動、1 行あたりの負例の数、切り詰めた query の割合、
024 と同じ形式の記事単位の評価(R1・R2・024 の Hub のモデル。下の「診断量: 024 と同じ形式の評価と R1 の再現」)。

#### 実験 B: 記事単位のラベルの評価での、関連性の定義の効果

**検証すること**: 実験 A と同じ索引・同じ query の記事単位のラベルの評価(正解は同じ記事の他の passage。query の passage 自身は候補から除く)で、R1 の Recall@10 が R2 より高いか。

**条件**: 実験 A と同じ R1 と R2(同じ学習。再学習しない)。評価の関連のラベルだけが違う。

**対比量**:

$$
\Delta_B = \frac{1}{n} \sum_{s} \left( R_s(\mathrm{R1}) - R_s(\mathrm{R2}) \right)
$$

(正を期待。$R_s(c)$ は記事単位のラベルでの Recall@10。上位 10 件に正解が 1 つ以上ある query の割合。)標準偏差 $\sigma_B$ は実験 A と同じ構成で、ブートストラップのクラスタも実験 A と同じ(同じ query)。

**判定基準**: $\Delta_B > 2\sigma_B$ なら支持、$\Delta_B < -2\sigma_B$ なら反証、それ以外は判定不能。

**介入の直接の作用点と、対比量までの距離**: 実験 A と同じ。作用点に近い診断量として、記事単位のラベルの平均逆順位と正規化割引累積利得(上位 10 件)を併記する。

**対比量が閾値を超えうる状態と、仮説との整合**: $\Delta_B > 2\sigma_B$ となりうる状態は、(i)記事単位の定義の学習が、記事単位のラベルの評価で優れる(仮説どおり)、(ii)R2 の学習が進んでいない(P1 で検出)、
(iii)R1 の値が上限に近い(P2 で検出)。入力の形式は両条件で同じで、形式の不一致による不利は、どちらの条件にもない。

**規則で決めた学習率による偏り**: 実験 A の「規則で学習率を決める段階」のとおり、段階 4 以降は R1 の学習率が最適からずれうる。実験 B では、R1 の値が下がり、対比量は **負の向き(仮説に不利)** に偏る。

**前提条件**: P0(R1・R2)、P1(R1・R2)、P2($R(\mathrm{R1})$ の記事単位のラベルが $[0.02, 0.95]$)。

**実験 A と B の判定の組み合わせの読み方**(事前の宣言。部分ごとの判定は別々に記録し、組み合わせはここで宣言した読み方だけに使う。どちらかが前提不成立のときは、組み合わせの読み方を適用しない)。
入力の形式・索引・query が同じなので、両方が支持のとき、それは入力の形式の一致の効果ではなく、関連性の定義(正例と in-batch の負例への一貫した当てはめ)の効果と読む:

| A \ B | 支持(R1 > R2) | 判定不能 | 反証(R2 > R1) |
|---|---|---|---|
| 支持(R2 > R1 で passage 単位のラベル) | **交差**: 各定義の検索器が、その定義のラベルの評価で優れる。定義が学ぶものを決める | R2 の利点は passage 単位のラベルで示されたが、R1 の記事単位のラベルでの優位は示されない。交差とは言えない | R2 が両方のラベルで優れる。定義の交差ではなく、passage 単位の定義の全面的な優位 |
| 判定不能 | R1 の利点は記事単位のラベルで示されたが、R2 の passage 単位のラベルでの優位は示されない。交差とは言えない | どちらのラベルでも差はノイズ床以内。この学習量・この設定で、定義の違いは検出されない | R2 は記事単位のラベルでも優れ、passage 単位のラベルでは差がない |
| 反証(R1 > R2) | R1 が両方のラベルで優れる。定義の交差ではなく、記事単位の定義の全面的な優位 | R1 は passage 単位のラベルで優れ、記事単位のラベルでは差がない | **逆の交差**: 各定義の検索器が、その定義と反対のラベルの評価で優れる |

(段階 4 以降に R1 の学習率を規則で決めた場合は、実験 A の「R1 の学習率が最適からずれて R2 が有利に見える」偏りと、実験 B の「R1 が不利に見える」偏りを考慮して読む。)

#### 実験 C: 2 段階の検索の効果(アプリの構成の判断)

**検証すること**: passage 単位の評価で、R2 の上位 $K = 50$ 件を cross-encoder で並べ替えた Recall@10 が、R2 だけ(並べ替えなし)より高いか。

**条件**: R2(並べ替えなし)と、R2 + cross-encoder(同じシードの R2 の上位 50 件を、同じシードの cross-encoder で並べ替える)。cross-encoder は、上の学習の組(第 1 段 R2 の上位 50 件から正例・負例を選ぶ)で学習する。

**対比量**:

$$
\Delta_C = \frac{1}{n} \sum_{s} \left( R_s(\mathrm{R3}) - R_s(\mathrm{R2}) \right)
$$

(正を期待。$n$ は実験 C のシード数、$R_s(\mathrm{R3})$ は R2 の上位 $K$ 件を cross-encoder で並べ替えた構成 R3 の Recall@10。)標準偏差 $\sigma_C$ は実験 A と同じ構成で、評価に使う query は 6.1 節の「cross-encoder の評価に使う query」(本番は評価用のすべて)。

**判定基準**: $\Delta_C > 2\sigma_C$ なら支持、$\Delta_C < -2\sigma_C$ なら反証、それ以外は判定不能。

**解釈の注意(学習量との交絡。実用の問い)**: 「R2 + cross-encoder」は、R2 に加えて cross-encoder の追加の学習($T_2$ ステップ、1 シードあたり 8 query × 8 候補 × $T_2$ 組の順伝播)を使う。
**同じ学習量での優劣を比べる実験ではない**。問いは「アプリで並べ替えを足す価値があるか」という実用の問いで、費用(順伝播の回数 $K$、観察 D)と合わせて読む。
並べ替えで到達できる Recall@10 の上限は coverage(上位 50 件に正例が入る割合)で、$\Delta_C$ の最大は coverage − $R(\mathrm{R2})$ である。

**介入の直接の作用点と、対比量までの距離**: 介入(並べ替え)が直接作用するのは、**上位 50 件の候補の順序** である。Recall@10 は、その順序を上位 10 件で二値化した量で、作用点に近い。
診断量として、並べ替え後の平均逆順位と正規化割引累積利得(上位 10 件)、候補 50 件のスコアの標準偏差の中央値(出力が定数に崩れていないかの検出)を併記する(判定には使わない)。

**対比量が閾値を超えうる状態と、仮説との整合**: $\Delta_C > 2\sigma_C$ となりうるのは、(i)cross-encoder が上位 50 件の中で正例を上位に並べ替える(仮説どおり)、(ii)R2 の学習が崩れている(P1 で検出)、
(iii)パイロットで選ばれた $T_2$ が実験 C に有利な向きの選択である(上の「パイロットの分岐」の偏り)。(iv)R2 だけの値が coverage に近く差が出ない(P2・P3 で検出。支持ではなく判定不能・前提不成立の側)。

**前提条件**: P0(R2・cross-encoder)、P1(R2 は実験 A・B の P1、cross-encoder は P1)、P2($R(\mathrm{R2}) \le$ coverage − 0.05)、P3(coverage ≥ 0.35 かつ coverage − $R(\mathrm{R2})$ ≥ 0.10)。

**規則で決めた学習率による偏り**: 段階 5 以降の cross-encoder の学習率は中心の値で、最適からずれると $R(\mathrm{R3})$ が下がり、$\Delta_C$ は負の向き(仮説に不利)に偏る(上の「学習率の較正」)。

**診断量(判定なし)**: 並べ替え後の平均逆順位・正規化割引累積利得、coverage、cross-encoder の最後の区間の訓練損失、スコアの標準偏差の中央値、学習に使える query の数と割合、
学習用の query・検証用の query・評価用の query での coverage(第 1 段 R2 が学習用の passage を見ていることによる、学習時と推論時の分布の違いの検出)。

#### 観察 D・E(判定基準を設けない)

- **観察 D(2 段階の検索の費用と指標)**: R2 のシード 0 の埋め込みの部分集合(件数 $N_{\mathrm{obs}}$)で、第 1 段を全探索と HNSW(025 の実装。$M = 5$、$\mathrm{efConstruction} = 100$。探索の幅は、全探索の上位 100 件との一致率が 0.9 に初めて届いた値)にし、
  cross-encoder(シード 0)で上位 $K = 10, 20, 50, 100$ 件を並べ替える。1 query あたりの第 1 段の距離計算の回数、第 2 段の順伝播の回数 $K$、並べ替えなしと並べ替えた後の Recall@10 と coverage を示す。
  関連性は部分集合の中の passage 単位(正解は query の passage だけ)で、部分集合の query の数は 6.1 節の水準で決まる(本番 600)。この表の Recall@10 の絶対値は、実験 A・C(コーパス全体)と比べない。
- **観察 E(検索結果の例)**: 同じ評価用の query に対する R1 と R2(シード 0)の上位の結果を示す。選び方は、評価用の query の並び順で、R1 と R2 の一方だけが正例を上位 10 件に入れた query の先頭から、
  R2 だけが入れたものを 2 件、R1 だけが入れたものを 1 件(結果を見て選ばない決定的な規則。存在しなければ、ある分だけ)。

#### 診断量: 024 と同じ形式の評価と R1 の再現(判定なし)

024 と同じ形式の記事単位の評価(024 の評価用の索引と query、固定長の passage と query)で、R1・R2・024 の Hub のモデル(C5 のシード 0)の Recall@10 を測る。
本トピックの R1 は、024 の C5 と入力の形式(query は抜いた文、passage は文を除いた形)・正例の作り方・バッチ(同じ記事の組を含む)が違うので、**ビット単位の一致も、024 の値との一致も期待しない**。
R1 のシード 0 の値と 024 の Hub の重みの値の差を、024 の C5 の記事単位の Recall@10 のシード間の標準偏差の目安(0.002〜0.009)と並べて印字する。判定には使わない。

#### 検出力の事前確認

本番の前に、基準の条件の値のシード間のばらつきをパイロットで測ることは、このセッションの時間の都合でできない(別の測定を設けない)。代わりに、同じ構成の 024 の本番(T4)の値から見積もる。

- 024 の C2・C5(記事単位の評価、3,244 query、5 シード): $R$ のシード間の標準偏差は 0.0023(C2)。実験 A の対比量の標準偏差は 0.0075(シード間の項 0.0035、ブートストラップの項 0.0066)、実験 B は 0.0049(0.0025、0.0042)、実験 D は 0.0074(0.0029、0.0068)で、
  ブートストラップの項(評価の query が有限であることによる変動)が支配的だった。閾値($2\sigma$)は 0.010〜0.018 だった。
- 本トピックの対比量の標準偏差の見込みは、同程度の $\sigma \approx 0.005$〜$0.010$(閾値 $2\sigma$ は **約 0.01〜0.02**)で、passage 単位のラベルは正例が 1 つの passage に限られる分、query ごとの hit の分散は大きくなりうる
  (評価の変動の下限は 024 の見積もり、上限は 2 条件の評価の変動が独立で打ち消しがまったく起きない場合の $\sqrt{2}$ 倍で、閾値は最大で約 0.03)。
- **想定される効果の大きさに比べた判断**: 想定される効果の大きさは測っていない(対比量の向きの情報を含むため)。検出できる最小の効果量(約 0.01〜0.03)が、Recall@10 の余地(基準の条件の値から天井まで)に比べて小さい見込みなので、**設計は見直さず本番に進む**。
  第 1 段のシード数を中(3)や低(2)に削る段階では、シード間の項が増えるが、支配的なブートストラップの項は変わらないので、閾値はほとんど増えない見込みである。

#### 仮説と判定の一覧

| 実験 | 仮説(対比量の向き) | 対比量 | 基準の条件 | 判定を除く診断量 |
|---|---|---|---|---|
| A | $\Delta_A > 0$ | $R(\mathrm{R2}) - R(\mathrm{R1})$(passage 単位のラベル) | R1 | 同じ記事の passage の割合、取り違えの割合、平均逆順位 |
| B | $\Delta_B > 0$ | $R(\mathrm{R1}) - R(\mathrm{R2})$(記事単位のラベル) | R2 | 平均逆順位、正規化割引累積利得 |
| C | $\Delta_C > 0$ | $R(\mathrm{R3}) - R(\mathrm{R2})$(passage 単位の評価) | R2 | 並べ替え後の平均逆順位、coverage、スコアの標準偏差 |

#### Hugging Face Hub への公開の条件(判定ではない。6.14 節)

公開の条件は、判定基準とは別に、アップロードするモデルが「使わない場合より悪くならない」ことの確認である。読み込み直した重みを評価して確かめ、満たさない場合はアップロードせず、その旨と値を印字する。

- **R2 のシード 0**(`kojikojiprg/ai-theories-text-embedding-en`の`inverse-cloze-task`ブランチ): 読み込み直した重みの passage 単位の評価での Recall@10 が、R1 のシード 0 の passage 単位の評価での Recall@10 を上回ること。
- **cross-encoder のシード 0**(`kojikojiprg/ai-theories-cross-encoder-en`): 読み込み直した重みで、シード 0 の R2 の上位 50 件を並べ替えた Recall@10 が、並べ替えなしの Recall@10 を上回ること(評価用の query)。

#### 設計の改訂の記録(本番の前)

次の 4 つの改訂を、本番の実行の前に入れた。いずれも、観測された結果の向きに依らない一般論(比較の妥当性、学習の信号の一貫性、入力の切り詰め、見積もりの余裕)による。
動作確認(縮小スケールの実行)の数値は、判定基準の根拠に使っていない。実験 A・B・C の対比量の定義、閾値の導出式(標準偏差の 2 倍)、期待する向きは変えていない。変えたのは、条件の作り方と評価のラベルの定義である。

| | 旧設計 | 新設計 | 理由 |
|---|---|---|---|
| 1. 入力の形式 | R1 は固定長の窓を query・正例にし、R2 は抜いた文と 1 文を除いた文脈を見る。実験 A は R2 の形式、実験 B は 024 の形式(固定長)で評価した | R1・R2 とも、query は窓から抜いた 1 文(文を残す確率 $r = 0.1$ も共通)。正例は、R2 が query を抜いた窓の文脈、R1 が同じ記事の別の窓(同じバッチのもう一方の組)の、文を除いた形。評価は、実験 A・B とも同じ索引(すべての passage を 1 文を除いた形にしたもの)・同じ query で、関連のラベルだけを変える(A: 正解は query の passage だけ、B: 正解は同じ記事の他の passage)。024 と同じ形式の記事単位の評価は診断量にする | 旧設計では、各モデルが学習で見た形式の評価で勝つだけで、A・B の両方が支持になりうる。関連性の定義の効果と、入力の形式の一致の効果を分けられない |
| 2. in-batch negatives | R2 も 1 つのバッチに同じ記事の組を入れない。R1 と R2 の負例はどちらも別の記事の passage だけ | 1 バッチ 64 組を 32 記事 × 2 組にし、同じ記事の組を意図的に同じバッチに入れる。R1 は同じ記事の組の passage を負例から除き(負例 62 個)、R2 は負例として数える(63 個)。「関連性の定義を、正例と in-batch の負例の両方に一貫して当てはめる」ことが、実験 A・B の介入 | 旧設計では、passage 単位の定義で同じ記事の他の passage が困難な負例になる(3.3 節)という本筋が、学習に現れない |
| 3. 抜く文の選び方 | 条件を満たす文(長さ 4〜96 トークン)から一様に選び、32 トークンを超える文は query の先頭で切る | query の長さの上限(32 トークン)以内に収まる文があればその中から選び、収まる文がない窓だけ長い文から選ぶ。学習の組と評価の索引の両方に適用する | 旧設計では、query の約 3 割が切り詰められ、文の後半の情報が query から失われる。切り詰めは学習と評価で同じ規則で行われる |
| 4. 見積もりの安全係数 | 学習の時間の見積もりは、本番の冒頭で測った 1 ステップの時間をそのまま使う | 学習(第 1 段・cross-encoder)の時間の見積もりに 1.1 倍の安全係数を掛ける | 025 の再実行に向けたパイロットで、T4 の実際の 1 ステップ(80〜81 ms)が本番の冒頭の計測値(75 ms)より約 7% 長かった。計測値の外れによる予算の超過を避ける |

**改訂に伴う変更**: 学習率の較正の指標は、新しい設計の同じ索引・同じ query の 2 つのラベルの Recall@10 の平均にした(改訂 1)。R1 は 024 の C5 と同じ入力の形式・バッチではなくなったので、R1 の再現の確認(024 の Hub のモデルとの比較)は、
024 と同じ形式の記事単位の評価(診断量)で行い、ビット単位の一致を期待しない。実験 B の評価の集合は、024 の評価用の索引(固定長)から、実験 A と同じ索引・同じ query に変わった。

### 6.2 スケーリングの計測と外挿(1 セッションの見積もり)

本番でデータ量や回数が大きくなる処理を、実行するデバイスの上で計測する。

- **学習**(1 ステップの処理が一定の反復): 第 1 段(R1・R2)と cross-encoder のそれぞれで、ウォームアップ 8 ステップの後、8・16・32 ステップの時間を測る。本番と同じ関数・同じ精度・同じバッチで測る。
  見積もりは **定常状態の 1 ステップの時間(32 ステップの計測から)をステップ数に比例させる**。べき指数 $b$ も推定し、べき乗則の外挿値と比例の値の差を印字する。
- **評価**: 検証用の評価(較正の 1 回ごと)、本番の評価(評価用の集合を、passage 単位のコーパス全体の索引と記事単位の索引で。R2 は第 1 段の上位 $K$ 件の採掘を含む)の時間を測る。評価用のコーパス全体の符号化は、最大の計測点が本番の量そのものである(外挿は不要)。
  あわせて、学習前の 008(R1・R2 と同じ構成)の検証用・評価用の Recall@10 を測る(P1(b) の基準と、学習前後の診断量。学習は行わない)。
- **cross-encoder の推論**: 候補の組の順伝播 1 組あたりの時間を測る(検証用・評価用の query の並べ替えの見積もりに使う)。
- **学習前のモデルの損失の測定**(P1(a) の分母): 32 ステップ分の時間を測り、学習 1 回ごとに最後の区間のステップ数だけ加える。
- **学習の組の生成**(第 1 段の R1・R2 の組、cross-encoder の組): ステップ数 64・128・256 の 3 点で計測し、べき指数を推定する。見積もりは 1 ステップあたりの時間を比例させる。
- **HNSW**(観察 D): 索引の件数 3 点($1024, 2048, 4096$)で構築の時間を測り、べき乗則で観察 D の件数(本番 16384・8192)まで外挿する。
- **参考**: バッチサイズを半分にしたときの第 1 段の 1 ステップの時間を測り、時間がバッチサイズに比例するか(固定費が支配的でないか)を印字する。
- **データの準備**: 5.3 節で、段階ごとの時間とピークのメモリを印字した。本番の冒頭で 1 回だけ行い(以降は全条件・全シードで共有する)、6.4 節の経過時間に含まれる。
- **判定・ブートストラップ・図・アップロードの準備**: 規模が小さく、合計しても数十秒〜数分である。固定の余裕(300 秒)に含める。

In [ ]:
_t0_scaling = time.time()
SCALING_STEP_COUNTS = (8, 16, 32)  # 水準によらず同じ
SCALING_WARMUP_STEPS = 8
SCHEDULE_STEP_COUNTS = (64, 128, 256)


def measure_first_stage(condition: str, step_counts, warmup_steps: int, batch_size: int = BATCH_SIZE) -> list[float]:
    # 本番と同じ学習の関数・精度で、定常状態の時間を測る(評価は行わない)
    total = max(warmup_steps + sum(step_counts), max(step_counts), warmup_steps)
    model = build_embedding_model(TIMING_SEED_INDEX).to(device)
    arrays = pair_arrays(condition, TIMING_SEED_INDEX, total)
    articles = pair_batches(TIMING_SEED_INDEX, total).articles

    def make(n: int) -> VariableBatches:
        # 同じ記事の 2 組はバッチの中で隣り合うので、先頭の batch_size 組を取れば組の対応は保たれる
        return VariableBatches(*(a[:n, :batch_size] for a in arrays), device, article_ids=articles[:n, :batch_size], mask_same_article=condition == "R1")

    def run(batches: VariableBatches) -> None:
        train_text_embedding_from_batches(
            model, batches, peak_learning_rate=FIRST_STAGE_LEARNING_RATE_CENTER[condition], warmup_steps=1,
            min_learning_rate=FIRST_STAGE_LEARNING_RATE_CENTER[condition] * MIN_LEARNING_RATE_RATIO, weight_decay=WEIGHT_DECAY,
            gradient_clip_threshold=GRADIENT_CLIP_THRESHOLD, temperature=TEMPERATURE, matryoshka_dimensions=MATRYOSHKA_DIMENSIONS,
            use_fp16_autocast=USE_FP16_AUTOCAST, init_loss_scale=INIT_LOSS_SCALE, loss_scale_growth_interval=LOSS_SCALE_GROWTH_INTERVAL,
        )

    warm = make(warmup_steps)
    timed_call(lambda: run(warm))
    times = []
    for n in step_counts:
        batches = make(n)
        times.append(timed_call(lambda b=batches: run(b)))
    del model
    empty_device_cache()
    return times


# 学習(R1・R2): 1 ステップの時間
STEP_SECONDS, SCALING_ROWS = {}, {}
for _condition in FIRST_STAGE_CONDITIONS:
    _times = measure_first_stage(_condition, SCALING_STEP_COUNTS, SCALING_WARMUP_STEPS)
    STEP_SECONDS[_condition] = _times[-1] / SCALING_STEP_COUNTS[-1]
    _fit = fit_power_law_exponent(SCALING_STEP_COUNTS, _times)
    _power = _times[-1] * (FIRST_STAGE_STEPS / SCALING_STEP_COUNTS[-1]) ** _fit.exponent
    SCALING_ROWS[_condition] = {"exponent": _fit.exponent, "power_minus_proportional": _power - STEP_SECONDS[_condition] * FIRST_STAGE_STEPS}
    print(
        f"{_condition}: {dict(zip(SCALING_STEP_COUNTS, rounded(_times, 2), strict=True))} 秒、1 ステップ {STEP_SECONDS[_condition] * 1000:.1f} ms、べき指数 b = {_fit.exponent:.3f}、"
        f"T_1 = {FIRST_STAGE_STEPS} でのべき乗則の外挿 - 比例の値 = {SCALING_ROWS[_condition]['power_minus_proportional']:+.1f} 秒"
    )
_half = measure_first_stage("R1", (SCALING_STEP_COUNTS[1],), SCALING_WARMUP_STEPS, batch_size=BATCH_SIZE // 2)[0] / SCALING_STEP_COUNTS[1]
print(
    f"参考 R1: バッチ {BATCH_SIZE // 2} の 1 ステップ {_half * 1000:.1f} ms、バッチ {BATCH_SIZE} は {STEP_SECONDS['R1'] * 1000:.1f} ms"
    f"(比 {STEP_SECONDS['R1'] / _half:.2f}。2 に近ければ時間はバッチサイズに比例し、1 に近ければ固定費が支配的)"
)

# 学習の組の生成(シードごとに 1 回。R1 と R2 で共通)
_times = []
for _steps in SCHEDULE_STEP_COUNTS:
    PAIR_BATCHES.clear()
    _start = time.time()
    pair_batches(TIMING_SEED_INDEX, _steps)
    _times.append(time.time() - _start)
_fit = fit_power_law_exponent(SCHEDULE_STEP_COUNTS, _times)
BATCH_SECONDS_PER_STEP = {c: _times[-1] / SCHEDULE_STEP_COUNTS[-1] for c in FIRST_STAGE_CONDITIONS}
print(f"学習の組の生成: {dict(zip(SCHEDULE_STEP_COUNTS, rounded(_times, 3), strict=True))} 秒、べき指数 b = {_fit.exponent:.3f}、1 ステップあたり {BATCH_SECONDS_PER_STEP['R1'] * 1000:.2f} ms(見積もりは比例)")
PAIR_BATCHES.clear()

# 学習前のモデルの損失の測定
_probe = build_embedding_model(TIMING_SEED_INDEX).to(device)
_timing_batches = first_stage_batches("R1", TIMING_SEED_INDEX, 32)
compute_pair_losses_from_batches(_probe, _timing_batches, range(32), TEMPERATURE, MATRYOSHKA_DIMENSIONS, USE_FP16_AUTOCAST)  # ウォームアップ
INITIAL_LOSS_SECONDS_PER_STEP = timed_call(lambda: compute_pair_losses_from_batches(_probe, _timing_batches, range(32), TEMPERATURE, MATRYOSHKA_DIMENSIONS, USE_FP16_AUTOCAST)) / 32
print(f"学習前のモデルの損失の測定: 1 ステップあたり {INITIAL_LOSS_SECONDS_PER_STEP * 1000:.1f} ms(学習 1 回につき最後の区間のステップ数だけ。T_1 = {FIRST_STAGE_STEPS} で {INITIAL_LOSS_SECONDS_PER_STEP * final_loss_window(FIRST_STAGE_STEPS):.1f} 秒)")
del _timing_batches

# 評価の時間と、学習前の 008 の値(R1・R2 と同じ構成、学習しない)
evaluate_calibration(_probe)  # ウォームアップ
CALIBRATION_EVALUATION_SECONDS = timed_call(lambda: evaluate_calibration(_probe))
UNTRAINED_VALIDATION_FIRST_STAGE = evaluate_calibration(_probe)
_start = time.time()
UNTRAINED_MAIN = evaluate_main(_probe, keep_embeddings=True)
sync_device()
MAIN_EVALUATION_SECONDS = time.time() - _start
_untrained_contexts = UNTRAINED_MAIN.pop("contexts")
UNTRAINED_MAIN.pop("queries")
MINING_SECONDS = timed_call(lambda: mine_candidates(_probe, _untrained_contexts))
print(
    f"評価の時間: 検証用 {CALIBRATION_EVALUATION_SECONDS:.2f} 秒、本番の評価(passage 単位のコーパス全体 + 記事単位)の合計 {MAIN_EVALUATION_SECONDS:.2f} 秒、R2 の第 1 段の上位 {RERANK_CANDIDATES} 件の採掘 {MINING_SECONDS:.2f} 秒"
    f"(学習用 {len(TRAIN_ENTRIES):,}・検証用 {len(VALIDATION_ENTRIES):,}・評価用 {len(EVALUATION_ENTRIES):,} query)"
)
print(
    f"学習前の 008(平均プール・因果マスク)の Recall@10: 検証用 passage 単位 {UNTRAINED_VALIDATION_FIRST_STAGE['passage_hits'].mean():.4f}・記事単位 {UNTRAINED_VALIDATION_FIRST_STAGE['article_hits'].mean():.4f}"
    f"(P1(b) の基準)、評価用 passage 単位 {UNTRAINED_MAIN['passage_hits'].mean():.4f}・記事単位 {UNTRAINED_MAIN['article_hits'].mean():.4f}。"
    f"ランダムな順位の期待値 passage 単位 {RANDOM_RECALL_PASSAGE:.6f}"
)
# コーパス全体の符号化の時間の外挿(索引の件数に比例するかを、1/4・1/2・全部で確かめる)
_sizes = [NUM_INDEX // 4, NUM_INDEX // 2, NUM_INDEX]
_encode_times = [timed_call(lambda n=n: embed_contexts(_probe, np.arange(n))) for n in _sizes]
_encode_fit = fit_power_law_exponent(_sizes, _encode_times)
print(f"索引の符号化: {dict(zip(_sizes, rounded(_encode_times, 3), strict=True))} 秒(べき指数 {_encode_fit.exponent:.3f}。最大の計測点が本番の量そのもの)")
del _untrained_contexts

# cross-encoder: 1 ステップの時間、推論の 1 組あたりの時間、学習の組の生成
_rng = np.random.default_rng(0)
_fake_mined = np.concatenate([TRAIN_ENTRIES[:, None], _rng.integers(0, NUM_INDEX, size=(len(TRAIN_ENTRIES), RERANK_CANDIDATES - 1))], axis=1)
_total_cross_encoder = SCALING_WARMUP_STEPS + sum(SCALING_STEP_COUNTS)
_cross_encoder_schedule_times = []
for _steps in SCHEDULE_STEP_COUNTS:
    _start = time.time()
    _cross_encoder_schedule = sample_cloze_reranking_schedule(TRAIN_ENTRIES, _fake_mined, _steps, BATCH_QUERIES, NUM_NEGATIVES, seed=1)
    _cross_encoder_schedule_times.append(time.time() - _start)
_fit = fit_power_law_exponent(SCHEDULE_STEP_COUNTS, _cross_encoder_schedule_times)
CROSS_ENCODER_SCHEDULE_SECONDS_PER_STEP = _cross_encoder_schedule_times[-1] / SCHEDULE_STEP_COUNTS[-1]
print(f"cross-encoder の学習の組の生成: {dict(zip(SCHEDULE_STEP_COUNTS, rounded(_cross_encoder_schedule_times, 3), strict=True))} 秒、べき指数 b = {_fit.exponent:.3f}、1 ステップあたり {CROSS_ENCODER_SCHEDULE_SECONDS_PER_STEP * 1000:.2f} ms(見積もりは比例)")
_cross_encoder_full = sample_cloze_reranking_schedule(TRAIN_ENTRIES, _fake_mined, _total_cross_encoder, BATCH_QUERIES, NUM_NEGATIVES, seed=2)


def measure_cross_encoder(step_counts, warmup_steps: int) -> list[float]:
    model = build_cross_encoder(TIMING_SEED_INDEX).to(device)

    def run(n: int) -> None:
        from src.training.reranking_variable import ClozeRerankingSchedule

        part = ClozeRerankingSchedule(_cross_encoder_full.query_ids[:n], _cross_encoder_full.candidate_ids[:n], _cross_encoder_full.num_usable)
        train_reranker_variable(
            model, CROSS_ENCODER_TABLES, part, peak_learning_rate=CROSS_ENCODER_LEARNING_RATE_CENTER, warmup_steps=1, min_learning_rate=CROSS_ENCODER_LEARNING_RATE_CENTER * MIN_LEARNING_RATE_RATIO,
            weight_decay=WEIGHT_DECAY, gradient_clip_threshold=GRADIENT_CLIP_THRESHOLD, use_fp16_autocast=USE_FP16_AUTOCAST,
            init_loss_scale=INIT_LOSS_SCALE, loss_scale_growth_interval=LOSS_SCALE_GROWTH_INTERVAL,
        )

    timed_call(lambda: run(warmup_steps))
    times = [timed_call(lambda n=n: run(n)) for n in step_counts]
    del model
    empty_device_cache()
    return times


_times = measure_cross_encoder(SCALING_STEP_COUNTS, SCALING_WARMUP_STEPS)
CROSS_ENCODER_STEP_SECONDS = _times[-1] / SCALING_STEP_COUNTS[-1]
_fit = fit_power_law_exponent(SCALING_STEP_COUNTS, _times)
print(
    f"cross-encoder の学習: {dict(zip(SCALING_STEP_COUNTS, rounded(_times, 2), strict=True))} 秒、1 ステップ {CROSS_ENCODER_STEP_SECONDS * 1000:.1f} ms、べき指数 b = {_fit.exponent:.3f}、"
    f"T_2 = {max(CROSS_ENCODER_STEP_CANDIDATES)} でのべき乗則の外挿 - 比例の値 = {_times[-1] * (max(CROSS_ENCODER_STEP_CANDIDATES) / SCALING_STEP_COUNTS[-1]) ** _fit.exponent - CROSS_ENCODER_STEP_SECONDS * max(CROSS_ENCODER_STEP_CANDIDATES):+.1f} 秒"
)
_half_cross_encoder = measure_cross_encoder((SCALING_STEP_COUNTS[1],), SCALING_WARMUP_STEPS)[0] / SCALING_STEP_COUNTS[1]
print(f"参考 cross-encoder: 全ステップ {CROSS_ENCODER_STEP_SECONDS * 1000:.1f} ms(参考: 同じ設定で 16 ステップの平均 {_half_cross_encoder * 1000:.1f} ms)")
_probe_cross_encoder = build_cross_encoder(TIMING_SEED_INDEX).to(device)
_pair_queries = RERANK_EVALUATION_ENTRIES[:40]
_pair_candidates = np.tile(np.arange(RERANK_CANDIDATES), (len(_pair_queries), 1)) + 0 * _pair_queries[:, None]
_pair_candidates = _fake_mined[np.searchsorted(TRAIN_ENTRIES, TRAIN_ENTRIES[:40])]  # 実際の候補の長さ分布に近い組(学習用の passage)
evaluate_rerank(_probe_cross_encoder, TRAIN_ENTRIES[:4], _pair_candidates[:4])  # ウォームアップ
CROSS_ENCODER_PAIR_SECONDS = timed_call(lambda: evaluate_rerank(_probe_cross_encoder, TRAIN_ENTRIES[:40], _pair_candidates)) / (40 * RERANK_CANDIDATES)
print(f"cross-encoder の推論: 1 組あたり {CROSS_ENCODER_PAIR_SECONDS * 1000:.3f} ms(候補 {RERANK_CANDIDATES} 件 x query 40 個から)")
del _probe_cross_encoder, _probe, _cross_encoder_full, _fake_mined
empty_device_cache()

# HNSW(観察 D): 索引の件数 3 点で構築の時間を測り、べき乗則で観察 D の件数まで外挿する
_hnsw_vectors = UNTRAINED_EMBEDDING_SAMPLE if "UNTRAINED_EMBEDDING_SAMPLE" in globals() else None
_probe2 = build_embedding_model(TIMING_SEED_INDEX).to(device)
_hnsw_pool = embed_contexts(_probe2, np.arange(min(NUM_INDEX, max(LEVEL_SETTINGS["HNSW_TIMING_SIZES"])))).cpu().numpy()
del _probe2
empty_device_cache()
_hnsw_times, _hnsw_search_times = [], []
for _n in LEVEL_SETTINGS["HNSW_TIMING_SIZES"]:
    _start = time.time()
    _index = HierarchicalNavigableSmallWorld(_hnsw_pool[:_n], HNSW_MAX_CONNECTIONS, HNSW_CONSTRUCTION_WIDTH, seed=OBSERVATION_SEED)
    _index.add(_n)
    _hnsw_times.append(time.time() - _start)
    _start = time.time()
    _index.search_batch(_hnsw_pool[:20], 11, OBSERVATION_SEARCH_WIDTHS[-1])
    _hnsw_search_times.append((time.time() - _start) / 20)
_hnsw_fit = fit_power_law_exponent(LEVEL_SETTINGS["HNSW_TIMING_SIZES"], _hnsw_times)
HNSW_SEARCH_SECONDS_PER_QUERY = _hnsw_search_times[-1]
print(
    f"HNSW の構築: {dict(zip(LEVEL_SETTINGS['HNSW_TIMING_SIZES'], rounded(_hnsw_times, 2), strict=True))} 秒、べき指数 b = {_hnsw_fit.exponent:.3f}、"
    f"観察 D の件数への外挿: " + "、".join(f"N = {n}: {_hnsw_times[-1] * (n / LEVEL_SETTINGS['HNSW_TIMING_SIZES'][-1]) ** _hnsw_fit.exponent:.0f} 秒" for n in LEVEL_SETTINGS["OBSERVATION_N_CANDIDATES"])
    + f"。探索 {HNSW_SEARCH_SECONDS_PER_QUERY * 1000:.1f} ms / query(探索の幅 {OBSERVATION_SEARCH_WIDTHS[-1]})"
)
del _hnsw_pool, _index


def hnsw_build_seconds(n: int) -> float:
    return _hnsw_times[-1] * (n / LEVEL_SETTINGS["HNSW_TIMING_SIZES"][-1]) ** _hnsw_fit.exponent


print("データの準備(5.3 節、1 回だけ): " + "、".join(f"{k} {v:.1f} 秒" for k, v in DATA_SECONDS_BY_STAGE.items()) + f"。合計 {DATA_SECONDS:.1f} 秒(6.4 節の経過時間に含まれる)")
SCALING_SECONDS = time.time() - _t0_scaling
print(f"スケーリングの計測 {SCALING_SECONDS:.1f} 秒")

### 6.3 パイロット(第 1 段の R2 と cross-encoder の学習ステップ数)

6.1 節で宣言した規則のとおり、本番と同じセッションの冒頭で、学習を伴うパイロットを行う。**測るのは単一の条件の量だけ** で、条件間の比較や対比量の計算はしない。評価に使うのは較正専用のシード(90)と検証用の query だけで、評価用の集合は使わない。

1. **第 1 段のパイロット**: R2 を、シード 90・$T_1$・中心の学習率で学習し、第 1 段の上位 $K = 50$ 件を求める(学習用・検証用・評価用の query)。この学習は、6.5 節で R2 の較正の格子の中心の点として使い回す。
   あわせて、各集合の query での coverage(上位 $K$ 件に正例が入る割合)を印字する。学習用の query での coverage が、検証用・評価用の query より大きければ、第 1 段が学習用の passage を見ていることによる学習時と推論時の分布の違いを示す。
2. **cross-encoder のパイロット**: $T_2 = 2048, 4096, 8192$(動作確認では 16, 32, 64)の順に、中心の学習率で学習し、(i)学習の成立 P1(a)、(ii)検証用の query での並べ替えの Recall@10 の増分(学習前の cross-encoder との差)が増分の標準偏差の 2 倍以上、
   (iii)検証用の query での並べ替えの Recall@10 が R2 だけの値を上回る、の 3 つがすべて成立した最初の $T_2$ で止める。どれも成立しなければ最大の候補を使う。選ばれた $T_2$ の最後の学習は、6.5 節で cross-encoder の較正の格子の中心の点として使い回す。

In [ ]:
_t0_pilot = time.time()
CALIBRATION_RECORDS: dict[tuple[str, float], dict] = {}  # (条件, 学習率) -> 第 1 段の較正専用シードの記録
CROSS_ENCODER_CALIBRATION_RECORDS: dict[tuple[int, float], dict] = {}  # (T_2, 学習率) -> cross-encoder の較正専用シードの記録


def learning_rate_key(learning_rate: float) -> float:
    return float(f"{learning_rate:.10g}")


# --- 1. 第 1 段のパイロット: R2(較正専用のシード、T_1、中心の学習率)---
_center = FIRST_STAGE_LEARNING_RATE_CENTER["R2"]
CALIBRATION_RECORDS[("R2", learning_rate_key(_center))] = train_first_stage("R2", CALIBRATION_SEED_INDEX, _center, FIRST_STAGE_STEPS, main=False, mine=True)
_pilot_r2 = CALIBRATION_RECORDS[("R2", learning_rate_key(_center))]
print(
    f"第 1 段のパイロット: R2(シード {CALIBRATION_SEED_INDEX}、T_1 = {FIRST_STAGE_STEPS}、学習率 {_center:.3g})、最後の区間の訓練損失 {_pilot_r2['final_train_loss']:.3f}"
    f"(同じ組での学習前のモデルの損失 {_pilot_r2['initial_model_loss_on_final_window']:.3f}、比 {_pilot_r2['final_train_loss'] / _pilot_r2['initial_model_loss_on_final_window']:.3f})、"
    f"検証用の Recall@10: passage 単位 {_pilot_r2['validation']['passage_hits'].mean():.4f}・記事単位 {_pilot_r2['validation']['article_hits'].mean():.4f}、{_pilot_r2['seconds']:.0f} 秒"
)
_coverage_rows = {}
for _name, _entries in (("train", TRAIN_ENTRIES), ("validation", VALIDATION_ENTRIES), ("evaluation", EVALUATION_ENTRIES)):
    _coverage_rows[_name] = float(usable_cloze_queries(MINED[CALIBRATION_SEED_INDEX][_name], _entries).mean())
PILOT_USABLE_QUERIES = int(usable_cloze_queries(MINED[CALIBRATION_SEED_INDEX]["train"], TRAIN_ENTRIES).sum())
print(
    f"第 1 段 R2 の上位 {RERANK_CANDIDATES} 件に正例が入る割合(coverage): 学習用の query {_coverage_rows['train']:.4f}・検証用 {_coverage_rows['validation']:.4f}・評価用 {_coverage_rows['evaluation']:.4f}。"
    f"R2 だけの Recall@10: 検証用 {MINED[CALIBRATION_SEED_INDEX]['validation_hits'].mean():.4f}・評価用 {MINED[CALIBRATION_SEED_INDEX]['evaluation_hits'].mean():.4f}。"
    f"学習に使える query(正例が上位 {RERANK_CANDIDATES} 件に入る学習用の query): {PILOT_USABLE_QUERIES:,} 個(学習用の query {len(TRAIN_ENTRIES):,} 個の {PILOT_USABLE_QUERIES / len(TRAIN_ENTRIES):.1%})"
)

# --- 2. cross-encoder のパイロット ---
def pilot_criteria(record: dict) -> dict:
    reasons = cross_encoder_failure_reasons(record)
    difference = record["validation_hits"].astype(np.float64) - record["untrained_validation_hits"].astype(np.float64)
    gain = float(difference.mean())
    sigma = float(bootstrap_mean(RERANK_VALIDATION_WEIGHTS, difference).std(ddof=1))
    no_rerank = float(record["validation_no_rerank_hits"].mean())
    return {
        "learning": not reasons, "learning_reasons": reasons, "gain": gain, "gain_sigma": sigma, "gain_ok": bool(gain > 0 and gain >= P1_SIGMA_MULTIPLIER * sigma),
        "above_no_rerank": bool(record["validation_recall"] > no_rerank), "no_rerank": no_rerank,
    }


PILOT_ROWS = []
CROSS_ENCODER_STEPS, PILOT_SUCCEEDED = max(CROSS_ENCODER_STEP_CANDIDATES), False
for _steps in CROSS_ENCODER_STEP_CANDIDATES:
    _record = train_cross_encoder(CALIBRATION_SEED_INDEX, CROSS_ENCODER_LEARNING_RATE_CENTER, _steps, main=False)
    CROSS_ENCODER_CALIBRATION_RECORDS[(_steps, learning_rate_key(CROSS_ENCODER_LEARNING_RATE_CENTER))] = _record
    _criteria = pilot_criteria(_record)
    _statistics = learning_loss_statistics(_record)
    PILOT_ROWS.append({"steps": _steps, "criteria": _criteria, "record": _record})
    print(
        f"cross-encoder のパイロット T_2 = {_steps}(学習率 {CROSS_ENCODER_LEARNING_RATE_CENTER:.3g}、シード {CALIBRATION_SEED_INDEX}): 最後の区間の訓練損失 {_record['final_train_loss']:.4f}"
        f"(学習前のモデルの損失 {_record['initial_model_loss_on_final_window']:.4f}、差の平均 / 標準誤差 {_statistics['mean'] / _statistics['standard_error'] if _statistics['standard_error'] > 0 else float('nan'):.1f}、一様な予測 {UNIFORM_LOSS_CROSS_ENCODER:.4f})、"
        f"検証用の query の並べ替えの Recall@10 {_record['validation_recall']:.4f}(学習前 {float(_record['untrained_validation_hits'].mean()):.4f}、増分 {_criteria['gain']:+.4f}、標準偏差 {_criteria['gain_sigma']:.4f}、R2 だけ {_criteria['no_rerank']:.4f}、coverage {_record['validation_coverage']:.4f})、"
        f"スコアの標準偏差の中央値 {_record['validation_score_std_median']:.4f}、clipping の発動 {_record['clip_rate']:.2f}、{_record['seconds']:.0f} 秒"
    )
    print(
        f"  (i)学習の成立 P1(a): {_criteria['learning']} {_criteria['learning_reasons'] or ''}、(ii)増分 >= {P1_SIGMA_MULTIPLIER:g} x 標準偏差: {_criteria['gain_ok']}、(iii)R2 だけを上回る: {_criteria['above_no_rerank']}"
    )
    if _criteria["learning"] and _criteria["gain_ok"] and _criteria["above_no_rerank"]:
        CROSS_ENCODER_STEPS, PILOT_SUCCEEDED = _steps, True
        break
_usable_cross = PILOT_USABLE_QUERIES
print(
    f"パイロットの結果: 3 条件がすべて成立した最小の T_2 = {CROSS_ENCODER_STEPS}" if PILOT_SUCCEEDED else f"パイロットの結果: どの T_2 でも 3 条件がすべては成立しなかった。規則どおり、最大の候補 T_2 = {CROSS_ENCODER_STEPS} を使う(実験 C は前提条件の成否で記録する)"
)
print(
    f"選ばれた T_2 = {CROSS_ENCODER_STEPS} が見る query 数 {BATCH_QUERIES * CROSS_ENCODER_STEPS:,} 個(学習に使える query {_usable_cross:,} 個に対して {BATCH_QUERIES * CROSS_ENCODER_STEPS / max(_usable_cross, 1):.2f} エポック)"
)
PILOT_SECONDS = time.time() - _t0_pilot
print(f"パイロット {PILOT_SECONDS / 60:.1f} 分")

### 6.4 実行計画の選択

6.2・6.3 節の計測とパイロットの後の経過時間から、8 通りの計画(6.1 節)のそれぞれの残りの実行時間を見積もる。

- 第 1 段の学習 1 回 = 1 ステップの時間 × $T_1$ + 学習の組の生成の時間(1 ステップあたりの時間 × $T_1$)+ 学習前のモデルの損失の測定の時間(最後の区間のステップ数 × 1 ステップあたりの時間)+ 評価の時間 + 固定費の余裕 2 秒。
  評価の時間は、較正の学習では検証用の評価 1 回、本番の学習では本番の評価(passage 単位のコーパス全体 + 記事単位)に、R2 では第 1 段の上位 $K$ 件の採掘を加えたもの。
- 第 1 段の較正 = 較正する条件(R2 は常に、R1 は段階 3 まで)ごとに、格子 3 点 + 拡張 1 点の **最悪の場合の 4 回**(パイロットで学習した R2 の中心の点は使い回すので、R2 は 3 回)。
- cross-encoder の学習 1 回 = 1 ステップの時間 × $T_2$ + 学習の組の生成の時間 + 学習前のモデルの損失の測定の時間(順伝播だけなので 1 ステップの時間の 1/3 として見積もる)+ 検証用の query の並べ替えの評価(学習前と学習後)+ 本番の学習ではさらに評価用の query の並べ替えの評価。
  cross-encoder の較正 = 最悪の場合の 4 回(パイロットで学習した中心の点は使い回すので 3 回。段階 5 以降はしない)。
- 本番 = 条件ごとのシード数 × 学習 1 回。観察 D の時間(HNSW の構築の外挿 + 探索 + 並べ替え)と、判定・ブートストラップ・図・アップロードの準備の余裕 300 秒。

**経過時間 + 見積もりが予算(120 分)に収まる計画のうち、番号の最も小さいものを選ぶ。** どれも収まらなければ、本番の学習の前に停止する。

In [ ]:
def first_stage_run_seconds(condition: str, kind: str) -> float:
    seconds = (STEP_SECONDS[condition] * TRAINING_SAFETY_FACTOR + BATCH_SECONDS_PER_STEP[condition]) * FIRST_STAGE_STEPS + RUN_OVERHEAD_SECONDS
    seconds += INITIAL_LOSS_SECONDS_PER_STEP * final_loss_window(FIRST_STAGE_STEPS) + CALIBRATION_EVALUATION_SECONDS
    if kind == "main":
        seconds += MAIN_EVALUATION_SECONDS + (MINING_SECONDS if condition == "R2" else 0.0)
    return seconds


def cross_encoder_run_seconds(kind: str, steps: int = CROSS_ENCODER_STEPS) -> float:
    seconds = (CROSS_ENCODER_STEP_SECONDS * TRAINING_SAFETY_FACTOR + CROSS_ENCODER_SCHEDULE_SECONDS_PER_STEP) * steps + RUN_OVERHEAD_SECONDS + CROSS_ENCODER_STEP_SECONDS / 3 * final_loss_window(steps)
    pairs_validation = len(RERANK_VALIDATION_ENTRIES) * RERANK_CANDIDATES
    seconds += 2 * pairs_validation * CROSS_ENCODER_PAIR_SECONDS  # 学習前と学習後の検証用の query の並べ替え
    if kind == "main":
        seconds += len(RERANK_EVALUATION_ENTRIES) * RERANK_CANDIDATES * CROSS_ENCODER_PAIR_SECONDS
    return seconds


def observation_seconds(observation_n: int) -> float:
    queries = LEVEL_SETTINGS["OBSERVATION_QUERIES"]
    search = queries * HNSW_SEARCH_SECONDS_PER_QUERY * len(OBSERVATION_SEARCH_WIDTHS) * 1.5  # 索引が大きいほど遅くなる分の余裕
    rerank = 2 * queries * max(OBSERVATION_RERANK_KS) * CROSS_ENCODER_PAIR_SECONDS  # 全探索と HNSW の上位 100 件
    return hnsw_build_seconds(observation_n) + search + rerank + 30.0


CALIBRATION_POINTS_WORST = len(LEARNING_RATE_GRID_MULTIPLIERS) + 1  # 格子 3 点 + 拡張 1 点


def estimate_stage(stage: int) -> dict:
    settings = stage_settings(stage)
    first_stage_calibration = (CALIBRATION_POINTS_WORST - 1) * first_stage_run_seconds("R2", "calibration")  # パイロットの R2 の中心の点は使い回す
    if settings["calibrate_r1"]:
        first_stage_calibration += CALIBRATION_POINTS_WORST * first_stage_run_seconds("R1", "calibration")
    cross_encoder_calibration = (CALIBRATION_POINTS_WORST - 1) * cross_encoder_run_seconds("calibration") if settings["calibrate_ce"] else 0.0
    seeds = settings["first_stage_seeds"]
    first_stage_main = seeds * (first_stage_run_seconds("R1", "main") + first_stage_run_seconds("R2", "main"))
    cross_encoder_main = settings["cross_encoder_seeds"] * cross_encoder_run_seconds("main")
    observation = observation_seconds(settings["observation_n"])
    total = first_stage_calibration + cross_encoder_calibration + first_stage_main + cross_encoder_main + observation + REPORTING_MARGIN_SECONDS
    return {
        "first_stage_calibration": first_stage_calibration, "cross_encoder_calibration": cross_encoder_calibration, "first_stage_main": first_stage_main,
        "cross_encoder_main": cross_encoder_main, "observation": observation, "total": total,
    }


ELAPSED_AT_SELECTION = time.time() - NOTEBOOK_START_TIME
STAGE_ESTIMATES = {k: estimate_stage(k) for k in range(NUM_STAGES)}
print(f"経過時間 {ELAPSED_AT_SELECTION / 60:.1f} 分(パイロットを含む)、予算 {SESSION_BUDGET_SECONDS / 60:.0f} 分。T_1 = {FIRST_STAGE_STEPS}、T_2 = {CROSS_ENCODER_STEPS}。学習の時間の見積もりには安全係数 {TRAINING_SAFETY_FACTOR} を掛ける")
for _k in range(NUM_STAGES):
    _e = STAGE_ESTIMATES[_k]
    _fits = ELAPSED_AT_SELECTION + _e["total"] <= SESSION_BUDGET_SECONDS
    print(
        f"  計画 {_k}({STAGE_CHANGES[_k]}): 第 1 段の較正 {_e['first_stage_calibration'] / 60:.1f} 分 + cross-encoder の較正 {_e['cross_encoder_calibration'] / 60:.1f} 分 + 第 1 段の本番 {_e['first_stage_main'] / 60:.1f} 分"
        f" + cross-encoder の本番 {_e['cross_encoder_main'] / 60:.1f} 分 + 観察 D {_e['observation'] / 60:.1f} 分 + 余裕 -> 残り {_e['total'] / 60:.1f} 分、終了の見込み {(ELAPSED_AT_SELECTION + _e['total']) / 60:.1f} 分({'収まる' if _fits else '超える'})"
    )
_feasible = [k for k in range(NUM_STAGES) if ELAPSED_AT_SELECTION + STAGE_ESTIMATES[k]["total"] <= SESSION_BUDGET_SECONDS]
AUTO_STAGE = _feasible[0] if _feasible else None
if FORCED_STAGE is not None:
    STAGE = FORCED_STAGE
    print(f"*** テスト専用の上書き: 計画 {FORCED_STAGE} を使う(見積もりによる選択は {AUTO_STAGE}) ***")
elif AUTO_STAGE is None:
    raise RuntimeError(
        f"最も下位の計画(計画 {NUM_STAGES - 1})でも見積もりが予算を超えるため、本番の学習の前に停止する。判定の対象の結果は何も得ていない。実行条件(判定基準・水準・前提条件以外)を直して再実行すること。"
    )
else:
    STAGE = AUTO_STAGE
SETTINGS = stage_settings(STAGE)
FIRST_STAGE_SEEDS, CROSS_ENCODER_SEEDS, OBSERVATION_N = SETTINGS["first_stage_seeds"], SETTINGS["cross_encoder_seeds"], SETTINGS["observation_n"]
SEEDS_A = SEEDS_B = FIRST_STAGE_SEEDS
SEEDS_C = CROSS_ENCODER_SEEDS
print(
    f"選ばれた計画: {STAGE}({STAGE_CHANGES[STAGE]})、第 1 段のシード数 {FIRST_STAGE_SEEDS}(実験 A・B)、cross-encoder のシード数 {CROSS_ENCODER_SEEDS}(実験 C)、R1 の較正 {SETTINGS['calibrate_r1']}、cross-encoder の較正 {SETTINGS['calibrate_ce']}、"
    f"観察 D の索引の件数 {OBSERVATION_N}、見積もり {STAGE_ESTIMATES[STAGE]['total'] / 60:.1f} 分"
)

### 6.5 学習率の較正

選ばれた計画の規則(6.1 節)のとおりに学習率を決める。第 1 段は $T_1$、cross-encoder は選ばれた $T_2$ で、較正専用のシード(90)の 1 シードで各格子点を学習する。**評価用の集合は較正に使わない。**
各格子点について、指標(第 1 段は検証用の passage 単位と記事単位の Recall@10 の平均 $M$、cross-encoder は検証用の query の並べ替えの Recall@10)とその標準偏差(記事を単位とするブートストラップ)、最後の区間の訓練損失、学習の成立の余裕を印字する。
パイロットで学習した R2 の中心の点と、選ばれた $T_2$ の cross-encoder の中心の点は、同じ設定であることをアサーションで確かめたうえで使い回す。

選択の規則: 指標が最大の点から標準偏差の 1 倍以内の候補(学習の成立の前提条件 P1(a) を満たす点だけ)のうち、学習の成立の余裕が最大の点(同点なら小さい学習率)。
最良が格子の端に来た場合は、その方向に公比 4 で 1 点だけ拡張して学習し、改めて同じ規則で選ぶ。

In [ ]:
_t0_calibration = time.time()
FIRST_STAGE_CALIBRATION: dict[str, dict] = {}
CROSS_ENCODER_CALIBRATION: dict = {}


def calibrate_first_stage(condition: str) -> dict:
    grid = list(first_stage_grid(condition))
    records: dict[float, dict] = {}

    def record_for(learning_rate: float) -> dict:
        key = learning_rate_key(learning_rate)
        if key not in records:
            if (condition, key) in CALIBRATION_RECORDS:  # パイロットの学習の使い回し(同じ設定であることを確かめる)
                reused = CALIBRATION_RECORDS[(condition, key)]
                assert reused["seed"] == CALIBRATION_SEED_INDEX and reused["num_steps"] == FIRST_STAGE_STEPS and math.isclose(reused["learning_rate"], learning_rate)
                records[key] = reused
            else:
                records[key] = train_first_stage(condition, CALIBRATION_SEED_INDEX, learning_rate, FIRST_STAGE_STEPS, main=False)
                CALIBRATION_RECORDS[(condition, key)] = records[key]
        return records[key]

    def choose(points: list[float]) -> tuple[int | None, list[dict]]:
        rows = [record_for(p) for p in points]
        values = [calibration_metric(r["validation"]) for r in rows]
        deviations = [calibration_metric_deviation(r["validation"]) for r in rows]
        margins = [1.0 - r["final_train_loss"] / r["initial_model_loss_on_final_window"] for r in rows]
        eligible = [not first_stage_failure_reasons(r) for r in rows]
        index = select_calibrated_index(points, values, deviations, margins, eligible)
        if index is None:  # P1(a) を満たす点がない: 余裕が最大の点を選び、P1 の不成立として進める
            index = max(range(len(points)), key=lambda i: (margins[i], -points[i]))
        return index, rows

    index, rows = choose(grid)
    extended = None
    if index == 0:
        extended = "lower"
        grid = [grid[0] / LEARNING_RATE_GRID_RATIO] + grid
        index, rows = choose(grid)
    elif index == len(grid) - 1:
        extended = "upper"
        grid = grid + [grid[-1] * LEARNING_RATE_GRID_RATIO]
        index, rows = choose(grid)
    return {
        "grid": grid, "chosen": grid[index], "interior": 0 < index < len(grid) - 1, "extended": extended,
        "metric": [calibration_metric(r["validation"]) for r in rows], "metric_deviation": [calibration_metric_deviation(r["validation"]) for r in rows],
        "passage": [float(r["validation"]["passage_hits"].mean()) for r in rows], "article": [float(r["validation"]["article_hits"].mean()) for r in rows],
        "final_train_loss": [r["final_train_loss"] for r in rows], "initial_loss": [r["initial_model_loss_on_final_window"] for r in rows],
        "p1a": [not first_stage_failure_reasons(r) for r in rows],
    }


def calibrate_cross_encoder() -> dict:
    grid = list(cross_encoder_grid())
    records: dict[float, dict] = {}

    def record_for(learning_rate: float) -> dict:
        key = learning_rate_key(learning_rate)
        if key not in records:
            if (CROSS_ENCODER_STEPS, key) in CROSS_ENCODER_CALIBRATION_RECORDS:  # パイロットの学習の使い回し
                reused = CROSS_ENCODER_CALIBRATION_RECORDS[(CROSS_ENCODER_STEPS, key)]
                assert reused["seed"] == CALIBRATION_SEED_INDEX and reused["num_steps"] == CROSS_ENCODER_STEPS and math.isclose(reused["learning_rate"], learning_rate)
                records[key] = reused
            else:
                records[key] = train_cross_encoder(CALIBRATION_SEED_INDEX, learning_rate, CROSS_ENCODER_STEPS, main=False)
                CROSS_ENCODER_CALIBRATION_RECORDS[(CROSS_ENCODER_STEPS, key)] = records[key]
        return records[key]

    def choose(points: list[float]) -> tuple[int, list[dict], bool]:
        rows = [record_for(p) for p in points]
        values = [r["validation_recall"] for r in rows]
        deviations = [float(bootstrap_mean(RERANK_VALIDATION_WEIGHTS, r["validation_hits"]).std(ddof=1)) for r in rows]
        statistics = [learning_loss_statistics(r) for r in rows]
        margins = [-s["mean"] / s["standard_error"] if s["standard_error"] > 0 else -math.inf for s in statistics]
        eligible = [not cross_encoder_failure_reasons(r) for r in rows]
        index = select_calibrated_index(points, values, deviations, margins, eligible)
        all_failed = index is None
        if index is None:  # P1(a) を満たす点がない: 余裕が最大の点を選び、P1 の不成立として進める
            index = max(range(len(points)), key=lambda i: (margins[i], -points[i]))
        return index, rows, all_failed

    index, rows, all_failed = choose(grid)
    extended = None
    if index == 0:
        extended = "lower"
        grid = [grid[0] / LEARNING_RATE_GRID_RATIO] + grid
        index, rows, all_failed = choose(grid)
    elif index == len(grid) - 1:
        extended = "upper"
        grid = grid + [grid[-1] * LEARNING_RATE_GRID_RATIO]
        index, rows, all_failed = choose(grid)
    return {
        "grid": grid, "chosen": grid[index], "interior": 0 < index < len(grid) - 1, "extended": extended, "all_failed": all_failed,
        "recall": [r["validation_recall"] for r in rows], "recall_deviation": [float(bootstrap_mean(RERANK_VALIDATION_WEIGHTS, r["validation_hits"]).std(ddof=1)) for r in rows],
        "final_train_loss": [r["final_train_loss"] for r in rows], "initial_loss": [r["initial_model_loss_on_final_window"] for r in rows],
        "z": [-learning_loss_statistics(r)["mean"] / learning_loss_statistics(r)["standard_error"] if learning_loss_statistics(r)["standard_error"] > 0 else float("nan") for r in rows],
        "p1a": [not cross_encoder_failure_reasons(r) for r in rows], "score_std": [r["validation_score_std_median"] for r in rows],
    }


def describe_first_stage(condition: str, result: dict) -> str:
    return (
        f"較正 {condition}: 学習率 {[float(f'{x:.3g}') for x in result['grid']]} -> 指標 M {rounded(result['metric'])}(標準偏差 {rounded(result['metric_deviation'])})、"
        f"passage 単位 {rounded(result['passage'])}・記事単位 {rounded(result['article'])}、最後の区間の訓練損失 {rounded(result['final_train_loss'], 3)}"
        f"(学習前のモデルの損失 {rounded(result['initial_loss'], 3)})、P1(a) {result['p1a']}、拡張 {result['extended'] or 'なし'}、選んだ学習率 {result['chosen']:.3g}({'内点' if result['interior'] else '端'})"
    )


LEARNING_RATE, LEARNING_RATE_SOURCE = {}, {}
FIRST_STAGE_CALIBRATION["R2"] = calibrate_first_stage("R2")
print(describe_first_stage("R2", FIRST_STAGE_CALIBRATION["R2"]))
LEARNING_RATE["R2"], LEARNING_RATE_SOURCE["R2"] = FIRST_STAGE_CALIBRATION["R2"]["chosen"], "較正"
if SETTINGS["calibrate_r1"]:
    FIRST_STAGE_CALIBRATION["R1"] = calibrate_first_stage("R1")
    print(describe_first_stage("R1", FIRST_STAGE_CALIBRATION["R1"]))
    LEARNING_RATE["R1"], LEARNING_RATE_SOURCE["R1"] = FIRST_STAGE_CALIBRATION["R1"]["chosen"], "較正"
else:
    LEARNING_RATE["R1"], LEARNING_RATE_SOURCE["R1"] = R1_RULE_LEARNING_RATE, "規則(024 の C5 の較正の値)"
    print(f"R1 の学習率: 較正しない。024 の C5 の較正で選ばれた値 {R1_RULE_LEARNING_RATE:.3g} を使う(P0 の対象外)")
if SETTINGS["calibrate_ce"]:
    CROSS_ENCODER_CALIBRATION = calibrate_cross_encoder()
    _r = CROSS_ENCODER_CALIBRATION
    print(
        f"較正 cross-encoder(T_2 = {CROSS_ENCODER_STEPS}): 学習率 {[float(f'{x:.3g}') for x in _r['grid']]} -> 検証用の並べ替えの Recall@10 {rounded(_r['recall'])}(標準偏差 {rounded(_r['recall_deviation'])})、"
        f"最後の区間の訓練損失 {rounded(_r['final_train_loss'], 3)}(学習前のモデルの損失 {rounded(_r['initial_loss'], 3)})、損失の差の平均 / 標準誤差の符号反転 {rounded(_r['z'], 1)}、P1(a) {_r['p1a']}、"
        f"スコアの標準偏差の中央値 {rounded(_r['score_std'], 4)}、拡張 {_r['extended'] or 'なし'}、選んだ学習率 {_r['chosen']:.3g}({'内点' if _r['interior'] else '端'})"
        + ("。すべての格子点で P1(a) が不成立: 余裕が最大の点を選び、実験 C は P1 の不成立として進める" if _r["all_failed"] else "")
    )
    LEARNING_RATE["cross_encoder"], LEARNING_RATE_SOURCE["cross_encoder"] = _r["chosen"], "較正"
else:
    LEARNING_RATE["cross_encoder"], LEARNING_RATE_SOURCE["cross_encoder"] = CROSS_ENCODER_RULE_LEARNING_RATE, "規則(中心の値)"
    print(f"cross-encoder の学習率: 較正しない。中心の値 {CROSS_ENCODER_RULE_LEARNING_RATE:.3g} を使う(P0 の対象外)")

# P0(較正): 各実験の対比量に入る条件のうち、自分自身を較正した条件で、選んだ学習率が内点であること
P0_DETAIL = {}
for _experiment, _conditions in {"A": ("R1", "R2"), "B": ("R1", "R2"), "C": ("R2", "cross_encoder")}.items():
    _calibrated = {}
    for _c in _conditions:
        if _c in ("R1", "R2") and _c in FIRST_STAGE_CALIBRATION:
            _calibrated[_c] = FIRST_STAGE_CALIBRATION[_c]["interior"]
        elif _c == "cross_encoder" and CROSS_ENCODER_CALIBRATION:
            _calibrated[_c] = CROSS_ENCODER_CALIBRATION["interior"]
    P0_DETAIL[_experiment] = {"calibrated": _calibrated, "by_rule": [c for c in _conditions if c not in _calibrated]}
    precondition_status[f"P0({_experiment})"] = all(_calibrated.values())
CALIBRATION_SECONDS = time.time() - _t0_calibration
print(f"学習率: {json.dumps({c: float(f'{v:.3g}') for c, v in LEARNING_RATE.items()})}、値の出どころ {json.dumps(LEARNING_RATE_SOURCE, ensure_ascii=False)}")
print(
    f"{RUN_TAG}P0(較正): "
    + "、".join(f"実験 {e}: {precondition_status[f'P0({e})']}(較正した条件の内点 {json.dumps(d['calibrated'])}、規則で決めた条件(対象外){d['by_rule'] or 'なし'})" for e, d in P0_DETAIL.items())
)
print(f"較正 {CALIBRATION_SECONDS / 60:.1f} 分(見積もり {(STAGE_ESTIMATES[STAGE]['first_stage_calibration'] + STAGE_ESTIMATES[STAGE]['cross_encoder_calibration']) / 60:.1f} 分、最悪の場合)")

### 6.6 第 1 段の本番の学習と評価

選ばれた計画のシード数の R1 と R2 をすべて学習する(同じシードの R1 と R2 は、記事の並びが同じ)。評価は評価用の集合で、最終ステップの重みについて行う(passage 単位はコーパス全体の索引、記事単位は 024 の評価用の索引)。
R2 は、cross-encoder の学習と評価に使う第 1 段の上位 $K$ 件も求める。1 行あたりの負例の数(R1 は同じ記事の組を除いて 62 個、R2 は 63 個)、文を文脈に残した組の割合、切り詰めた query の割合を、学習ごとに印字する。
診断量として、024 と同じ形式の記事単位の評価の Recall@10 を R1・R2・024 の Hub のモデルで測り、R1 のシード 0 と 024 の値の対応を印字する(組の作り方が違うので、一致は期待しない。判定には使わない)。

In [ ]:
_t0_main = time.time()
RUNS: dict[tuple[str, int], dict] = {}
for _s in range(FIRST_STAGE_SEEDS):
    for _c in FIRST_STAGE_CONDITIONS:
        RUNS[(_c, _s)] = train_first_stage(_c, _s, LEARNING_RATE[_c], FIRST_STAGE_STEPS, main=True)
        _r = RUNS[(_c, _s)]
        print(
            f"{_c} シード {_s}(学習率 {_r['learning_rate']:.3g}): 評価用 Recall@10 passage 単位のラベル {_r['passage_hits'].mean():.4f}・記事単位のラベル {_r['article_hits'].mean():.4f}"
            f"(024 と同じ形式の記事単位の評価 {_r['format024_hits'].mean():.4f}。検証用 {_r['validation']['passage_hits'].mean():.4f}・{_r['validation']['article_hits'].mean():.4f})、最後の区間の訓練損失 {_r['final_train_loss']:.3f}"
            f"(学習前のモデルの損失 {_r['initial_model_loss_on_final_window']:.3f}、比 {_r['final_train_loss'] / _r['initial_model_loss_on_final_window']:.3f})、clipping の発動 {_r['clip_rate']:.2f}、更新のスキップ {_r['skipped_steps']}、"
            f"1 行あたりの負例の数 {_r['negatives_per_row']:.1f}、文を文脈に残した組の割合 {_r['kept_share']:.3f}、切り詰めた query の割合 {_r['truncated_query_share']:.3f}、{_r['seconds']:.0f} 秒"
        )
    assert RUNS[("R1", _s)]["article_hash"] == RUNS[("R2", _s)]["article_hash"], "同じシードの R1 と R2 の記事の並びが一致しない"
FIRST_STAGE_MAIN_SECONDS = time.time() - _t0_main
print(f"第 1 段の本番の学習と評価 {FIRST_STAGE_MAIN_SECONDS / 60:.1f} 分(見積もり {STAGE_ESTIMATES[STAGE]['first_stage_main'] / 60:.1f} 分)")

# --- 診断量: 024 と同じ形式の記事単位の評価(R1・R2・024 の Hub の重みの 3 つ。判定には使わない)と、R1 の再現の確認 ---
from huggingface_hub import hf_hub_download  # noqa: E402

_hub_config = json.loads(Path(hf_hub_download(EMBEDDING_REPO_ID, "config.json")).read_text(encoding="utf-8"))
_hub_backbone = build_gpt(_hub_config)
_hub_backbone.load_state_dict(torch.load(hf_hub_download(EMBEDDING_REPO_ID, "model_state.pt"), map_location="cpu"))
_hub_model = TextEmbeddingModel(_hub_backbone, pooling=_hub_config["pooling"], attention=_hub_config["attention"]).to(device).eval()
_hub_queries, _hub_passages = embed_article_set(_hub_model, "evaluation")
HUB_FORMAT_024_RECALL = evaluate_embeddings(_hub_queries, _hub_passages, EVALUATION_SET).recall_at(10)
del _hub_model, _hub_backbone, _hub_queries, _hub_passages
empty_device_cache()
print(f"診断量: 024 と同じ形式の記事単位の評価(024 の評価用の索引と query。固定長)の Recall@10(シード平均): R1 {np.mean([RUNS[('R1', s)]['format024_hits'].mean() for s in range(FIRST_STAGE_SEEDS)]):.4f}、R2 {np.mean([RUNS[('R2', s)]['format024_hits'].mean() for s in range(FIRST_STAGE_SEEDS)]):.4f}、024 の Hub の重み(C5 のシード 0)を読み込み直した値 {HUB_FORMAT_024_RECALL:.4f}(024 のモデルカードの値 {REFERENCE_RECALL_024})。判定には使わない")
_r1 = RUNS[("R1", 0)]
print(
    f"R1 の再現の確認(診断量): 本トピックの R1 は、024 の C5 と入力の形式(query は抜いた文、passage は文を除いた形)・正例の作り方・バッチ(同じ記事の組を含む)が違うので、"
    f"ビット単位の一致も、024 の値との一致も期待しない。同じ形式の評価での R1 のシード 0 の値は {_r1['format024_hits'].mean():.4f}、024 の Hub の重みは {HUB_FORMAT_024_RECALL:.4f}(差 {_r1['format024_hits'].mean() - HUB_FORMAT_024_RECALL:+.4f}。"
    f"024 の C5 の記事単位 Recall@10 のシード間の標準偏差の目安は 0.002〜0.009)"
)

### 6.7 cross-encoder の本番の学習と評価

選ばれた計画のシード数の cross-encoder を学習する。シード $s$ の cross-encoder は、シード $s$ の R2 の上位 $K$ 件で学習し、評価する。評価は、R2 の上位 $K$ 件を並べ替えた後の Recall@10 を、評価用の query で測る。

In [ ]:
_t0_ce = time.time()
CROSS_ENCODER_RUNS: dict[int, dict] = {}
for _s in range(CROSS_ENCODER_SEEDS):
    CROSS_ENCODER_RUNS[_s] = train_cross_encoder(_s, LEARNING_RATE["cross_encoder"], CROSS_ENCODER_STEPS, main=True)
    _r = CROSS_ENCODER_RUNS[_s]
    _statistics = learning_loss_statistics(_r)
    print(
        f"cross-encoder シード {_s}(T_2 = {CROSS_ENCODER_STEPS}、学習率 {_r['learning_rate']:.3g}): 並べ替えた Recall@10 {_r['hits'].mean():.4f}(R2 だけ {_r['no_rerank_hits'].mean():.4f}、coverage {_r['coverage']:.4f})、"
        f"最後の区間の訓練損失 {_r['final_train_loss']:.3f}(学習前のモデルの損失 {_r['initial_model_loss_on_final_window']:.3f}、差の平均 / 標準誤差 {_statistics['mean'] / _statistics['standard_error'] if _statistics['standard_error'] > 0 else float('nan'):.1f})、"
        f"検証用の並べ替えの Recall@10 {_r['validation_recall']:.4f}(学習前 {float(_r['untrained_validation_hits'].mean()):.4f})、スコアの標準偏差の中央値 {_r['score_std_median']:.4f}、clipping の発動 {_r['clip_rate']:.2f}、{_r['seconds']:.0f} 秒"
    )
CROSS_ENCODER_MAIN_SECONDS = time.time() - _t0_ce
print(f"cross-encoder の本番の学習と評価 {CROSS_ENCODER_MAIN_SECONDS / 60:.1f} 分(見積もり {STAGE_ESTIMATES[STAGE]['cross_encoder_main'] / 60:.1f} 分)")

### 6.8 実験 A: passage 単位の評価での、関連性の定義の効果

In [ ]:
def first_stage_learning_check(conditions: tuple[str, ...], seeds: list[int]) -> dict:
    # 第 1 段の P1(6.1 節): (a)全シードで訓練損失が有限、最後の区間の訓練損失 <= 同じ組での学習前のモデルの損失 x P1_LOSS_RATIO、
    # (b)自分の定義の検証用の Recall@10(R1: 記事単位、R2: passage 単位) >= 学習前の 008 の値 + P1_VALIDATION_GAIN
    untrained = {"R1": float(UNTRAINED_VALIDATION_FIRST_STAGE["article_hits"].mean()), "R2": float(UNTRAINED_VALIDATION_FIRST_STAGE["passage_hits"].mean())}
    own_key = {"R1": "article_hits", "R2": "passage_hits"}
    failed_a, gains, ratios = [], {}, []
    for c in conditions:
        for s in seeds:
            if first_stage_failure_reasons(RUNS[(c, s)]):
                failed_a.append((c, s))
            ratios.append(RUNS[(c, s)]["final_train_loss"] / RUNS[(c, s)]["initial_model_loss_on_final_window"])
        gains[c] = np.array([float(RUNS[(c, s)]["validation"][own_key[c]].mean()) - untrained[c] for s in seeds])
    gain_ok = all(bool((g >= P1_VALIDATION_GAIN).all()) for g in gains.values())
    return {"ok": not failed_a and gain_ok, "failed_a": failed_a, "ratio_max": max(ratios), "gains": gains, "gain_ok": gain_ok}


def describe_first_stage_learning(result: dict) -> str:
    return (
        f"((a)の不成立 {result['failed_a'] or 'なし'}、最後の区間の訓練損失 / 同じ組での学習前のモデルの損失の最大 {result['ratio_max']:.3f} <= {P1_LOSS_RATIO}。"
        f"(b)自分の定義の検証用の Recall@10 の学習前からの増分の最小 " + "、".join(f"{c} {g.min():+.4f}" for c, g in result["gains"].items()) + f" >= {P1_VALIDATION_GAIN}: {result['gain_ok']})"
    )


def passage_recall(condition: str, seeds: int) -> np.ndarray:
    return np.array([RUNS[(condition, s)]["passage_hits"].mean() for s in range(seeds)])


def article_recall(condition: str, seeds: int) -> np.ndarray:
    return np.array([RUNS[(condition, s)]["article_hits"].mean() for s in range(seeds)])


def format024_recall(condition: str, seeds: int) -> np.ndarray:
    return np.array([RUNS[(condition, s)]["format024_hits"].mean() for s in range(seeds)])


def contrast_bootstrap(rows: list[np.ndarray], clusters: np.ndarray, num_first: int) -> np.ndarray:
    # 行 = 条件 x シードの query ごとの 0/1。先頭の num_first 行の平均 - 残りの行の平均の、記事を単位とする対応付きブートストラップの再標本の分布
    hits = np.stack(rows).astype(np.float64)
    boot = paired_cluster_bootstrap_ratio_of_sums(hits, np.ones(hits.shape[1]), clusters, BOOTSTRAP_RESAMPLES, BOOTSTRAP_SEED)
    return boot[:, :num_first].mean(axis=1) - boot[:, num_first:].mean(axis=1)


_seeds = list(range(SEEDS_A))
R_A_R1, R_A_R2 = passage_recall("R1", SEEDS_A), passage_recall("R2", SEEDS_A)
A_PER_SEED = R_A_R2 - R_A_R1
_boot_a = contrast_bootstrap([RUNS[("R2", s)]["passage_hits"] for s in _seeds] + [RUNS[("R1", s)]["passage_hits"] for s in _seeds], EVALUATION_QUERY_ARTICLES, SEEDS_A)
DELTA_A = float(A_PER_SEED.mean())
SIGMA_A = combined_sigma(A_PER_SEED, _boot_a)
_p1_a = first_stage_learning_check(("R1", "R2"), _seeds)
precondition_status["P1(A)"] = _p1_a["ok"]
precondition_status["P2(A)"] = bool(P2_FLOOR <= R_A_R1.mean() <= P2_CEILING)
A_PRECONDITIONS = ["P0(A)", "P1(A)", "P2(A)"]
A_COMPUTED = judge(DELTA_A, SIGMA_A["sigma"])
A_VERDICT = A_COMPUTED if all(precondition_status[k] for k in A_PRECONDITIONS) else "前提不成立"

print(f"{RUN_TAG}実験 A(シード {_seeds}、T_1 = {FIRST_STAGE_STEPS}、評価用の query {len(EVALUATION_ENTRIES)} 個、索引 {NUM_INDEX:,} passage、ラベル: passage 単位)")
print(f"  passage 単位の Recall@10 R(R1): {rounded(R_A_R1)}、R(R2): {rounded(R_A_R2)}")
print(f"  d_s = R(R2) - R(R1): {rounded(A_PER_SEED)}")
print(
    f"  Delta_A = {DELTA_A:+.4f}、sigma_A = {SIGMA_A['sigma']:.4f}(シード間 {SIGMA_A['seed_term']:.4f}・ブートストラップ {SIGMA_A['bootstrap_term']:.4f}、反復 {BOOTSTRAP_RESAMPLES:,})、閾値 2 sigma_A = {SIGMA_MULTIPLIER * SIGMA_A['sigma']:.4f}"
)
print(
    f"  前提条件: P0 {precondition_status['P0(A)']}、P1 {precondition_status['P1(A)']}{describe_first_stage_learning(_p1_a)}、"
    f"P2 {precondition_status['P2(A)']}(R(R1) の平均 {R_A_R1.mean():.4f} が [{P2_FLOOR}, {P2_CEILING}])"
)
print(f"  判定関数の結果 {A_COMPUTED} -> {RUN_TAG}最終判定: {A_VERDICT}")
print("  診断量(判定なし):")
for _c in ("R1", "R2"):
    _runs = [RUNS[(_c, s)] for s in range(SEEDS_A)]
    print(
        f"    {_c}: 上位 10 件のうち正例と同じ記事の passage の割合 {np.mean([r['same_article_top10_share'] for r in _runs]):.4f}、1 位が同じ記事の別の passage の割合 {np.mean([r['top1_same_article_wrong'] for r in _runs]):.4f}、"
        f"平均逆順位 {np.mean([r['passage_mean_reciprocal_rank'] for r in _runs]):.4f}、Recall@10 のシード間の標準偏差 {passage_recall(_c, SEEDS_A).std(ddof=1) if SEEDS_A >= 2 else float('nan'):.4f}、"
        f"最後の区間の訓練損失 {np.mean([r['final_train_loss'] for r in _runs]):.3f}、学習率 {LEARNING_RATE[_c]:.3g}(出どころ {LEARNING_RATE_SOURCE[_c]})、clipping の発動 {np.mean([r['clip_rate'] for r in _runs]):.2f}"
    )
print(
    f"    参照: 学習前の 008 の passage 単位の Recall@10(評価用) {UNTRAINED_MAIN['passage_hits'].mean():.4f}、ランダムな順位の期待値 {RANDOM_RECALL_PASSAGE:.6f}。"
    f"R1・R2 の記事単位のラベルの Recall@10(実験 B の量): R1 {article_recall('R1', SEEDS_A).mean():.4f}、R2 {article_recall('R2', SEEDS_A).mean():.4f}"
)

_fig, _ax = plt.subplots(figsize=(5.5, 4))
for _x, _c in enumerate(("R1", "R2")):
    _values = passage_recall(_c, SEEDS_A)
    _ax.scatter([_x] * len(_values), _values, alpha=0.6)
    _ax.plot([_x - 0.2, _x + 0.2], [_values.mean()] * 2, "k-")
_ax.axhline(UNTRAINED_MAIN["passage_hits"].mean(), color="gray", linestyle=":", label="untrained 008")
_ax.set_xticks([0, 1])
_ax.set_xticklabels(["R1 (article-level)", "R2 (Inverse Cloze Task)"])
_ax.set_ylabel("Recall@10 (passage-level evaluation)")
_ax.set_title(f"{PLOT_TAG}Experiment A")
_ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

### 6.9 実験 B: 記事単位の評価での、関連性の定義の効果

In [ ]:
_seeds = list(range(SEEDS_B))
R_B_R1, R_B_R2 = article_recall("R1", SEEDS_B), article_recall("R2", SEEDS_B)
B_PER_SEED = R_B_R1 - R_B_R2
_boot_b = contrast_bootstrap([RUNS[("R1", s)]["article_hits"] for s in _seeds] + [RUNS[("R2", s)]["article_hits"] for s in _seeds], EVALUATION_QUERY_ARTICLES, SEEDS_B)
DELTA_B = float(B_PER_SEED.mean())
SIGMA_B = combined_sigma(B_PER_SEED, _boot_b)
_p1_b = first_stage_learning_check(("R1", "R2"), _seeds)
precondition_status["P1(B)"] = _p1_b["ok"]
precondition_status["P2(B)"] = bool(P2_FLOOR <= R_B_R1.mean() <= P2_CEILING)
B_PRECONDITIONS = ["P0(B)", "P1(B)", "P2(B)"]
B_COMPUTED = judge(DELTA_B, SIGMA_B["sigma"])
B_VERDICT = B_COMPUTED if all(precondition_status[k] for k in B_PRECONDITIONS) else "前提不成立"

print(f"{RUN_TAG}実験 B(シード {_seeds}、T_1 = {FIRST_STAGE_STEPS}、評価用の query {len(EVALUATION_ENTRIES)} 個、索引 {NUM_INDEX:,} passage(実験 A と同じ)、ラベル: 記事単位。ランダムな順位の Recall@10 の期待値 {RANDOM_RECALL_ARTICLE_LABEL:.5f})")
print(f"  記事単位の Recall@10 R(R1): {rounded(R_B_R1)}、R(R2): {rounded(R_B_R2)}")
print(f"  d_s = R(R1) - R(R2): {rounded(B_PER_SEED)}")
print(
    f"  Delta_B = {DELTA_B:+.4f}、sigma_B = {SIGMA_B['sigma']:.4f}(シード間 {SIGMA_B['seed_term']:.4f}・ブートストラップ {SIGMA_B['bootstrap_term']:.4f}、反復 {BOOTSTRAP_RESAMPLES:,})、閾値 2 sigma_B = {SIGMA_MULTIPLIER * SIGMA_B['sigma']:.4f}"
)
print(
    f"  前提条件: P0 {precondition_status['P0(B)']}、P1 {precondition_status['P1(B)']}{describe_first_stage_learning(_p1_b)}、"
    f"P2 {precondition_status['P2(B)']}(R(R1) の平均 {R_B_R1.mean():.4f} が [{P2_FLOOR}, {P2_CEILING}])"
)
print(f"  判定関数の結果 {B_COMPUTED} -> {RUN_TAG}最終判定: {B_VERDICT}")
print("  診断量(判定なし):")
for _c in ("R1", "R2"):
    _runs = [RUNS[(_c, s)] for s in range(SEEDS_B)]
    print(
        f"    {_c}: 記事単位の平均逆順位 {np.mean([r['article_mean_reciprocal_rank'] for r in _runs]):.4f}、正規化割引累積利得(上位 10 件){np.mean([r['article_normalized_discounted_cumulative_gain'].mean() for r in _runs]):.4f}、"
        f"Recall@10 のシード間の標準偏差 {article_recall(_c, SEEDS_B).std(ddof=1) if SEEDS_B >= 2 else float('nan'):.4f}"
    )
print(f"    参照: 学習前の 008 の記事単位のラベルの Recall@10(評価用) {UNTRAINED_MAIN['article_hits'].mean():.4f}")
print(
    f"    診断量(判定なし): 024 と同じ形式の記事単位の評価(024 の評価用の索引と query、固定長)の Recall@10: R1 {format024_recall('R1', SEEDS_B).mean():.4f}、R2 {format024_recall('R2', SEEDS_B).mean():.4f}、"
    f"024 の Hub の重み {HUB_FORMAT_024_RECALL:.4f}、学習前の 008 {UNTRAINED_MAIN['format024_hits'].mean():.4f}。R1 と R2 の差(R1 - R2)のシードごと: {rounded(format024_recall('R1', SEEDS_B) - format024_recall('R2', SEEDS_B))}"
)

# 実験 A と B の判定の組み合わせの読み方(6.1 節で宣言した表。どちらかが前提不成立のときは適用しない)
COMBINATION_READING = {
    ("支持", "支持"): "交差: 各定義の検索器が、その定義のラベルの評価で優れる(同じ索引・同じ query・同じ入力の形式で、ラベルだけが違う)",
    ("支持", "判定不能"): "R2 の利点は passage 単位のラベルで示されたが、R1 の記事単位のラベルでの優位は示されない。交差とは言えない",
    ("支持", "反証"): "R2 が両方のラベルで優れる。定義の交差ではなく、passage 単位の定義の全面的な優位",
    ("判定不能", "支持"): "R1 の利点は記事単位のラベルで示されたが、R2 の passage 単位のラベルでの優位は示されない。交差とは言えない",
    ("判定不能", "判定不能"): "どちらのラベルでも差はノイズ床以内。この学習量・この設定で、定義の違いは検出されない",
    ("判定不能", "反証"): "R2 は記事単位のラベルでも優れ、passage 単位のラベルでは差がない",
    ("反証", "支持"): "R1 が両方のラベルで優れる。定義の交差ではなく、記事単位の定義の全面的な優位",
    ("反証", "判定不能"): "R1 は passage 単位のラベルで優れ、記事単位のラベルでは差がない",
    ("反証", "反証"): "逆の交差: 各定義の検索器が、その定義と反対のラベルの評価で優れる",
}
if "前提不成立" in (A_VERDICT, B_VERDICT):
    print(f"  A と B の組み合わせ: 実験 A {A_VERDICT}、実験 B {B_VERDICT} のため、組み合わせの読み方を適用しない")
else:
    print(f"  A と B の組み合わせ(実験 A {A_VERDICT}、実験 B {B_VERDICT}): {COMBINATION_READING[(A_VERDICT, B_VERDICT)]}")
    if not SETTINGS["calibrate_r1"]:
        print("  (段階 4 以降: R1 の学習率は規則で決めたので、実験 A の支持は、R1 の学習率が最適からずれた分の偏り(仮説に有利な向き)を含みうる。実験 B の反証・判定不能は、R1 に不利な向きの偏りを含みうる)")

_fig, _axes = plt.subplots(1, 2, figsize=(10, 3.8))
for _ax, _function, _title in ((_axes[0], passage_recall, "A: passage-label"), (_axes[1], article_recall, "B: article-label")):
    for _x, _c in enumerate(("R1", "R2")):
        _values = _function(_c, SEEDS_A)
        _ax.scatter([_x] * len(_values), _values, alpha=0.6)
        _ax.plot([_x - 0.2, _x + 0.2], [_values.mean()] * 2, "k-")
    _ax.set_xticks([0, 1])
    _ax.set_xticklabels(["R1", "R2"])
    _ax.set_ylabel("Recall@10")
    _ax.set_title(f"{PLOT_TAG}{_title}")
plt.tight_layout()
plt.show()

### 6.10 実験 C: 2 段階の検索の効果(アプリの構成の判断)

In [ ]:
_seeds = list(range(SEEDS_C))
R_C_RERANK = np.array([CROSS_ENCODER_RUNS[s]["hits"].mean() for s in _seeds])
R_C_FIRST = np.array([CROSS_ENCODER_RUNS[s]["no_rerank_hits"].mean() for s in _seeds])
for _s in _seeds:  # R2 だけの hits は、実験 A の R2 の passage 単位の hits と一致する(同じ順位の定義)
    assert np.array_equal(CROSS_ENCODER_RUNS[_s]["no_rerank_hits"], RUNS[("R2", _s)]["passage_hits"][RERANK_EVALUATION_POSITIONS]) if _s < SEEDS_A else True
C_PER_SEED = R_C_RERANK - R_C_FIRST
_boot_c = contrast_bootstrap([CROSS_ENCODER_RUNS[s]["hits"] for s in _seeds] + [CROSS_ENCODER_RUNS[s]["no_rerank_hits"] for s in _seeds], RERANK_EVALUATION_ARTICLES, SEEDS_C)
DELTA_C = float(C_PER_SEED.mean())
SIGMA_C = combined_sigma(C_PER_SEED, _boot_c)
COVERAGE_C = float(np.mean([CROSS_ENCODER_RUNS[s]["coverage"] for s in _seeds]))
# P1: R2(実験 A・B と同じ第 1 段の P1 のうち R2 の分)と cross-encoder の P1((a)全シードの訓練損失、(b)検証用の query の並べ替えの Recall@10 の増分)
_p1_r2 = first_stage_learning_check(("R2",), _seeds)
_failed_a = [s for s in _seeds if cross_encoder_failure_reasons(CROSS_ENCODER_RUNS[s])]
_gain = bootstrap_rerank_gain([CROSS_ENCODER_RUNS[s] for s in _seeds])
_gain_ok = bool(_gain["gain"] > 0 and _gain["gain"] >= P1_SIGMA_MULTIPLIER * _gain["sigma"])
precondition_status["P1(C)"] = bool(_p1_r2["ok"] and not _failed_a and _gain_ok)
precondition_status["P2(C)"] = bool(R_C_FIRST.mean() <= COVERAGE_C - P2_ROOM)
precondition_status["P3(C)"] = bool(COVERAGE_C >= P3_MIN_COVERAGE and COVERAGE_C - R_C_FIRST.mean() >= P3_MIN_HEADROOM)
C_PRECONDITIONS = ["P0(C)", "P1(C)", "P2(C)", "P3(C)"]
C_COMPUTED = judge(DELTA_C, SIGMA_C["sigma"])
C_VERDICT = C_COMPUTED if all(precondition_status[k] for k in C_PRECONDITIONS) else "前提不成立"

print(f"{RUN_TAG}実験 C(シード {_seeds}、T_2 = {CROSS_ENCODER_STEPS}、K = {RERANK_CANDIDATES}、並べ替えの query {len(RERANK_EVALUATION_ENTRIES)} 個)")
print(f"  Recall@10 R(R3): {rounded(R_C_RERANK)}、R(R2 だけ): {rounded(R_C_FIRST)}")
print(f"  d_s = R(R3) - R(R2): {rounded(C_PER_SEED)}")
print(
    f"  Delta_C = {DELTA_C:+.4f}、sigma_C = {SIGMA_C['sigma']:.4f}(シード間 {SIGMA_C['seed_term']:.4f}・ブートストラップ {SIGMA_C['bootstrap_term']:.4f}、反復 {BOOTSTRAP_RESAMPLES:,})、閾値 2 sigma_C = {SIGMA_MULTIPLIER * SIGMA_C['sigma']:.4f}"
)
print(
    f"  前提条件: P0 {precondition_status['P0(C)']}、P1 {precondition_status['P1(C)']}"
    f"(R2 の P1 {_p1_r2['ok']}、cross-encoder の(a)の不成立 {_failed_a or 'なし'}、(b)検証用の query での並べ替えの Recall@10 の増分のシード平均 {_gain['gain']:+.4f}(シードごと {rounded(_gain['per_seed_gain'])})、"
    f"標準偏差 {_gain['sigma']:.4f}(シード間 {_gain['seed_term']:.4f}・ブートストラップ {_gain['bootstrap_term']:.4f})、増分 / 標準偏差 {_gain['gain'] / _gain['sigma'] if _gain['sigma'] > 0 else float('inf'):.1f}(閾値 {P1_SIGMA_MULTIPLIER}): {_gain_ok})、"
    f"P2 {precondition_status['P2(C)']}(R(R2 だけ) の平均 {R_C_FIRST.mean():.4f} <= coverage {COVERAGE_C:.4f} - {P2_ROOM})、"
    f"P3 {precondition_status['P3(C)']}(coverage {COVERAGE_C:.4f} >= {P3_MIN_COVERAGE}、coverage - R2 だけ {COVERAGE_C - R_C_FIRST.mean():.4f} >= {P3_MIN_HEADROOM})"
)
print(f"  判定関数の結果 {C_COMPUTED} -> {RUN_TAG}最終判定: {C_VERDICT}")
print("  診断量(判定なし):")
_mean_reciprocal_rank_first = np.array([float((1.0 / RUNS[("R2", s)]["passage_rank"][RERANK_EVALUATION_POSITIONS]).mean()) for s in _seeds if s < SEEDS_A])
print(
    f"    R2 だけ(索引全体の順位): Recall@10 {R_C_FIRST.mean():.4f}、平均逆順位 {_mean_reciprocal_rank_first.mean() if len(_mean_reciprocal_rank_first) else float('nan'):.4f}、coverage(上位 {RERANK_CANDIDATES} 件に正例が入る割合){COVERAGE_C:.4f}"
)
print(
    f"    R3(R2 + cross-encoder): Recall@10 {R_C_RERANK.mean():.4f}(シード間の標準偏差 {R_C_RERANK.std(ddof=1) if SEEDS_C >= 2 else float('nan'):.4f})、平均逆順位 {np.mean([CROSS_ENCODER_RUNS[s]['mean_reciprocal_rank'] for s in _seeds]):.4f}、"
    f"正規化割引累積利得(上位 10 件){np.mean([CROSS_ENCODER_RUNS[s]['normalized_discounted_cumulative_gain'] for s in _seeds]):.4f}、最後の区間の訓練損失 {np.mean([CROSS_ENCODER_RUNS[s]['final_train_loss'] for s in _seeds]):.3f}(一様な予測 {UNIFORM_LOSS_CROSS_ENCODER:.3f})"
)
print(f"    並べ替えによる Recall@10 の変化(追加の学習量と交絡するので、実用の問いとして読む): {DELTA_C:+.4f}。1 query あたりの順伝播の回数 {RERANK_CANDIDATES} 回(cross-encoder は組ごと)")
print("    候補 50 件のスコアの標準偏差の中央値(崩壊の検出。判定には使わない): " + rounded([CROSS_ENCODER_RUNS[s]["score_std_median"] for s in _seeds], 4).__repr__())
print(
    "    学習時と推論時の分布の違いの検出: 第 1 段 R2 の上位 50 件に正例が入る割合(coverage) シードごとの 学習用 / 検証用 / 評価用: "
    + "、".join(
        f"{s}: {usable_cloze_queries(MINED[s]['train'], TRAIN_ENTRIES).mean():.3f} / {usable_cloze_queries(MINED[s]['validation'], VALIDATION_ENTRIES).mean():.3f} / {usable_cloze_queries(MINED[s]['evaluation'], EVALUATION_ENTRIES).mean():.3f}"
        for s in _seeds
    )
)
print(
    f"    学習に使える query: " + "、".join(f"シード {s}: {CROSS_ENCODER_SCHEDULES[(s, CROSS_ENCODER_STEPS)].num_usable:,} 個(見た query {BATCH_QUERIES * CROSS_ENCODER_STEPS:,} 個、{BATCH_QUERIES * CROSS_ENCODER_STEPS / CROSS_ENCODER_SCHEDULES[(s, CROSS_ENCODER_STEPS)].num_usable:.2f} エポック)" for s in _seeds)
)

_fig, _ax = plt.subplots(figsize=(5.5, 4))
for _x, (_label, _values) in enumerate((("R2 only", R_C_FIRST), ("R2 + cross-encoder", R_C_RERANK))):
    _ax.scatter([_x] * len(_values), _values, alpha=0.6)
    _ax.plot([_x - 0.2, _x + 0.2], [_values.mean()] * 2, "k-")
_ax.axhline(COVERAGE_C, color="gray", linestyle="--", label=f"coverage (K = {RERANK_CANDIDATES})")
_ax.set_xticks([0, 1])
_ax.set_xticklabels(["R2 only", "R2 + cross-encoder"])
_ax.set_ylabel(f"Recall@10 (passage-level, top-{RERANK_CANDIDATES} reranked)")
_ax.set_title(f"{PLOT_TAG}Experiment C")
_ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

### 6.11 観察 D・E(判定なし)

**観察 D(2 段階の検索の費用と指標)**: R2 のシード 0 の索引の埋め込みの部分集合(件数 $N_{\mathrm{obs}}$。評価用の query の passage をすべて含み、残りは索引からランダムに選ぶ)で、第 1 段を全探索と HNSW(025 の実装、$M = 5$、$\mathrm{efConstruction} = 100$)にし、
上位 100 件の先頭 $K = 10, 20, 50, 100$ 件を、cross-encoder のシード 0 で並べ替える。HNSW の探索の幅は、全探索の上位 100 件との一致率が 0.9 に初めて届いた値(格子 $101, 127, 160, 202, 254$ のうち)。
費用は、1 query あたりの第 1 段の距離計算の回数と、第 2 段の順伝播の回数 $K$。関連性は部分集合の中の passage 単位(正解は query の passage だけ)で、Recall@10 の絶対値は実験 A・C(コーパス全体)と比べない。

**観察 E(検索結果の例)**: 評価用の query の並び順で、R1 と R2(シード 0)の一方だけが正例を上位 10 件に入れた query を、R2 だけが入れたもの 2 件、R1 だけが入れたもの 1 件選ぶ(決定的な規則)。

In [ ]:
_t0_obs = time.time()
assert ("R2", UPLOAD_SEED) in SAVED_STATES and ("cross_encoder", UPLOAD_SEED) in SAVED_STATES and "R2" in KEPT_EMBEDDINGS and "R1" in KEPT_EMBEDDINGS
_rng = np.random.default_rng(OBSERVATION_SEED)
_query_positions = select_evenly(len(EVALUATION_ENTRIES), LEVEL_SETTINGS["OBSERVATION_QUERIES"])
_query_entries = EVALUATION_ENTRIES[_query_positions]
_others = np.setdiff1d(np.arange(NUM_INDEX), _query_entries)
_extra = _rng.choice(_others, size=min(len(_others), max(OBSERVATION_N - len(_query_entries), 0)), replace=False)
OBSERVATION_SUBSET = np.sort(np.concatenate([_query_entries, _extra]))  # 部分集合の passage の添字(索引の中の番号。昇順)
_position_in_subset = np.searchsorted(OBSERVATION_SUBSET, _query_entries)
assert (OBSERVATION_SUBSET[_position_in_subset] == _query_entries).all() and len(np.unique(OBSERVATION_SUBSET)) == len(OBSERVATION_SUBSET)
_contexts = KEPT_EMBEDDINGS["R2"]["contexts"]
_queries = KEPT_EMBEDDINGS["R2"]["queries"][_query_positions]
_subset_vectors = _contexts[torch.from_numpy(OBSERVATION_SUBSET)]
_depth = max(OBSERVATION_RERANK_KS)

# 第 1 段: 全探索
_exact_local, _ = exact_search(_queries.to(device), _subset_vectors.to(device), _depth)
_exact_candidates = OBSERVATION_SUBSET[_exact_local]
# 第 1 段: HNSW
_start = time.time()
_hnsw = HierarchicalNavigableSmallWorld(_subset_vectors.numpy(), HNSW_MAX_CONNECTIONS, HNSW_CONSTRUCTION_WIDTH, seed=OBSERVATION_SEED)
_hnsw.add(len(OBSERVATION_SUBSET))
HNSW_BUILD_SECONDS = time.time() - _start
_hnsw_rows = []
for _width in OBSERVATION_SEARCH_WIDTHS:
    _ids, _costs = _hnsw.search_batch(_queries.numpy(), _depth, _width)
    _recall = float(np.mean([len(set(_ids[i].tolist()) & set(_exact_local[i].tolist())) / _depth for i in range(len(_ids))]))
    _hnsw_rows.append((_width, _recall, float(_costs.mean()), _ids))
    if _recall >= OBSERVATION_TARGET_RECALL:
        break
OBSERVATION_WIDTH, OBSERVATION_HNSW_RECALL, OBSERVATION_HNSW_COST, _hnsw_ids = _hnsw_rows[-1][0], _hnsw_rows[-1][1], _hnsw_rows[-1][2], _hnsw_rows[-1][3]
assert (_hnsw_ids >= 0).all(), "HNSW の候補が足りない query がある"
_hnsw_candidates = OBSERVATION_SUBSET[_hnsw_ids]

_model = build_cross_encoder(UPLOAD_SEED).to(device)
_model.load_state_dict({k: v.to(device) for k, v in SAVED_STATES[("cross_encoder", UPLOAD_SEED)].items()})
OBSERVATION_TABLE = {}
for _name, _candidates, _cost in (("exact search", _exact_candidates, float(len(OBSERVATION_SUBSET))), (f"HNSW (search width = {OBSERVATION_WIDTH})", _hnsw_candidates, OBSERVATION_HNSW_COST)):
    _scores = score_candidates_variable(_model, CLOZE_QUERY_TOKENS, CLOZE_QUERY_LENGTHS, CONTEXT_TOKENS, CONTEXT_LENGTHS, _query_entries, _candidates, SCORE_BATCH, USE_FP16_AUTOCAST)
    OBSERVATION_TABLE[_name] = {"cost": _cost}
    for _k in OBSERVATION_RERANK_KS:
        _after = rerank_single_positive(_scores[:, :_k], _candidates[:, :_k], _query_entries, NUM_INDEX)
        _before = rerank_single_positive(np.broadcast_to(-np.arange(_k, dtype=np.float64), (len(_query_entries), _k)), _candidates[:, :_k], _query_entries, NUM_INDEX)  # 第 1 段の順位のまま
        OBSERVATION_TABLE[_name][_k] = {"after": _after.recall_at(10), "before": _before.recall_at(10), "coverage": _after.coverage()}
del _model
empty_device_cache()
print(
    f"{RUN_TAG}観察 D(判定なし。部分集合 N = {len(OBSERVATION_SUBSET)}、query {len(_query_entries)} 個、R2 のシード 0 の埋め込み、cross-encoder のシード 0 で並べ替え、ランダムな順位の Recall@10 の期待値 {10 / len(OBSERVATION_SUBSET):.4f})"
)
print(
    f"  HNSW の構築 {HNSW_BUILD_SECONDS:.1f} 秒(M = {HNSW_MAX_CONNECTIONS}、efConstruction = {HNSW_CONSTRUCTION_WIDTH})。探索の幅の格子と全探索の上位 {_depth} 件との一致率: "
    + "、".join(f"ef = {w}: {r:.3f}" for w, r, _, _ in _hnsw_rows)
    + f"。使う探索の幅 {OBSERVATION_WIDTH}"
)
for _name, _row in OBSERVATION_TABLE.items():
    print(f"  第 1 段: {_name}、第 1 段の距離計算の回数(1 query あたり){_row['cost']:,.0f}")
    for _k in OBSERVATION_RERANK_KS:
        _m = _row[_k]
        print(f"    K = {_k:>3}: 第 2 段の順伝播 {_k} 回、coverage {_m['coverage']:.4f}、Recall@10: 並べ替えなし {_m['before']:.4f} -> cross-encoder で並べ替え {_m['after']:.4f}")

_fig, _ax = plt.subplots(figsize=(6.5, 4))
for _name in OBSERVATION_TABLE:
    _ax.plot(OBSERVATION_RERANK_KS, [OBSERVATION_TABLE[_name][_k]["after"] for _k in OBSERVATION_RERANK_KS], "o-", label=f"{_name}: reranked")
    _ax.plot(OBSERVATION_RERANK_KS, [OBSERVATION_TABLE[_name][_k]["before"] for _k in OBSERVATION_RERANK_KS], "x--", label=f"{_name}: first-stage order")
_ax.set_xscale("log")
_ax.set_xlabel("K (candidates reranked)")
_ax.set_ylabel("Recall@10 (passage-level, subset)")
_ax.set_title(f"{PLOT_TAG}Observation D (N = {len(OBSERVATION_SUBSET)})")
_ax.legend(fontsize=6)
plt.tight_layout()
plt.show()

# --- 観察 E: 検索結果の例 ---
_rank_r1, _rank_r2 = RUNS[("R1", 0)]["passage_rank"], RUNS[("R2", 0)]["passage_rank"]
_only_r2 = np.nonzero((_rank_r2 <= 10) & (_rank_r1 > 10))[0][:2]
_only_r1 = np.nonzero((_rank_r1 <= 10) & (_rank_r2 > 10))[0][:1]
_selected = list(_only_r2) + list(_only_r1)


def snippet(entry: int, limit: int = 160) -> str:
    return tokenizer.decode(CLOZE.context_tokens[entry, : CLOZE.context_lengths[entry]].tolist())[:limit].replace("\n", "\\n")


print(f"{RUN_TAG}観察 E(判定なし。評価用の query のうち、R2 だけが正例を上位 10 件に入れた {len(_only_r2)} 件、R1 だけが入れた {len(_only_r1)} 件。シード 0)")
for _position in _selected:
    _entry = int(EVALUATION_ENTRIES[_position])
    _query_text = tokenizer.decode(CLOZE.query_tokens[_entry, : CLOZE.query_lengths[_entry]].tolist()).replace("\n", "\\n")
    print(f"  query(抜いた文): {_query_text[:200]!r}  正例の passage: {snippet(_entry, 100)!r}")
    for _name in ("R1", "R2"):
        _scores = (KEPT_EMBEDDINGS[_name]["queries"][_position] @ KEPT_EMBEDDINGS[_name]["contexts"].T).numpy()
        _top = np.argsort(-_scores)[:3]
        _rank = (_rank_r1 if _name == "R1" else _rank_r2)[_position]
        print(f"    {_name}(正例の順位 {_rank}): " + " / ".join(f"{'[正例]' if int(t) == _entry else ('[同じ記事]' if ENTRY_ARTICLES[int(t)] == ENTRY_ARTICLES[_entry] else '')}{snippet(int(t), 60)!r}" for t in _top))
OBSERVATION_SECONDS = time.time() - _t0_obs
print(f"観察 D・E {OBSERVATION_SECONDS / 60:.1f} 分(見積もり {STAGE_ESTIMATES[STAGE]['observation'] / 60:.1f} 分)")

### 6.12 不変条件のアサーションと`SMOKE_TEST`の配線

- 実効水準の照合: 実際の学習の長さ・シード数・ブートストラップの反復回数が、5.2 節と 6.4 節で印字した値と一致する。
- 同じシードの R1 と R2 の記事の並びが一致する(6.6 節)。R2 の組はシードごとに違う。評価の順位の長さが評価用の query の数と一致する。
- 総パラメータ数が R1 と R2 で同じで、cross-encoder は本体のパラメータ数 + ヘッド(257 個)である。
- 保持した埋め込み(R2 のシード 0)から順位を再計算すると、記録した順位と一致する。第 1 段の上位 $K$ 件の先頭が、全探索の最上位と一致する。
- アップロード用に保持した重み(R2 と cross-encoder のシード 0)が存在し、トークナイザなど同梱しないものを含まない。

In [ ]:
# --- 実効水準の照合(SMOKE_TEST の配線)---
assert CURRENT_LEVEL_NAME == ("smoke" if SMOKE_TEST else "prod")
for (_c, _s), _r in RUNS.items():
    assert _r["num_steps"] == FIRST_STAGE_STEPS == len(_r["train_loss"]) == LEVELS[CURRENT_LEVEL_NAME]["FIRST_STAGE_STEPS"], (_c, _s)
    assert len(_r["passage_rank"]) == len(_r["article_rank"]) == len(EVALUATION_ENTRIES) and len(_r["format024_hits"]) == EVALUATION_SET.num_queries
assert {s for (_c, s) in RUNS} == set(range(FIRST_STAGE_SEEDS)) and set(CROSS_ENCODER_RUNS) == set(range(CROSS_ENCODER_SEEDS))
for _s, _r in CROSS_ENCODER_RUNS.items():
    assert _r["num_steps"] == CROSS_ENCODER_STEPS == len(_r["train_loss"]) and CROSS_ENCODER_STEPS in LEVELS[CURRENT_LEVEL_NAME]["CROSS_ENCODER_STEP_CANDIDATES"], _s
    assert len(_r["hits"]) == len(RERANK_EVALUATION_ENTRIES)
assert BOOTSTRAP_RESAMPLES == LEVELS[CURRENT_LEVEL_NAME]["BOOTSTRAP_RESAMPLES"] and _boot_a.shape == (BOOTSTRAP_RESAMPLES,)
assert CROSS_ENCODER_SEEDS <= FIRST_STAGE_SEEDS and set(SETTINGS) == {"observation_n", "cross_encoder_seeds", "first_stage_seeds", "calibrate_r1", "calibrate_ce"}
assert SETTINGS == stage_settings(STAGE) and OBSERVATION_N == LEVELS[CURRENT_LEVEL_NAME]["OBSERVATION_N_CANDIDATES"][1 if STAGE >= 1 else 0]
print(
    f"実効水準: 水準 {CURRENT_LEVEL_NAME!r}、T_1 = {FIRST_STAGE_STEPS}(全 {len(RUNS)} 回の第 1 段の学習で一致)、T_2 = {CROSS_ENCODER_STEPS}(全 {len(CROSS_ENCODER_RUNS)} 回の cross-encoder の学習で一致)、"
    f"第 1 段のシード数 {FIRST_STAGE_SEEDS}・cross-encoder のシード数 {CROSS_ENCODER_SEEDS}、ブートストラップ {BOOTSTRAP_RESAMPLES:,} 回、計画 {STAGE}: OK"
)

# --- 記事の並び・組・パラメータ数 ---
for _s in range(FIRST_STAGE_SEEDS):
    assert RUNS[("R1", _s)]["article_hash"] == RUNS[("R2", _s)]["article_hash"], _s
assert len({RUNS[("R2", s)]["schedule_hash"] for s in range(FIRST_STAGE_SEEDS)}) == FIRST_STAGE_SEEDS and len({RUNS[("R1", s)]["schedule_hash"] for s in range(FIRST_STAGE_SEEDS)}) == FIRST_STAGE_SEEDS
assert RUNS[("R1", 0)]["schedule_hash"] != RUNS[("R2", 0)]["schedule_hash"], "R1 と R2 の組が同じ(介入がない)"
# R1 と R2 の組の関係: 同じバッチの query と passage のトークンの集合は同じで、正例の対応(R1: 同じ記事のもう一方の組の窓)だけが違う。R1 だけが同じ記事の組を負例から除く
_q2, _ql2, _c2, _cl2 = pair_arrays("R2", 0, 8)
_q1, _ql1, _c1, _cl1 = pair_arrays("R1", 0, 8)
_swap = np.arange(BATCH_SIZE) ^ 1
assert np.array_equal(_q1, _q2) and np.array_equal(_ql1, _ql2) and np.array_equal(_c1, _c2[:, _swap]) and np.array_equal(_cl1, _cl2[:, _swap]), "R1 と R2 の組の対応が設計と違う"
_articles_check = pair_batches(0, 8).articles
assert (_articles_check[:, _swap] == _articles_check).all(), "入れ替える相手が同じ記事でない"
_batches_r1, _batches_r2 = first_stage_batches("R1", 0, 8), first_stage_batches("R2", 0, 8)
assert _batches_r1.mask_same_article and not _batches_r2.mask_same_article
assert (_batches_r1.negatives_per_row() == BATCH_SIZE - 2).all() and (_batches_r2.negatives_per_row() == BATCH_SIZE - 1).all()
assert int(_batches_r1.ignore(0).sum()) == BATCH_SIZE and bool((_batches_r1.ignore(0).diagonal() == 0).all())
NEGATIVES_PER_ROW = {"R1": float(_batches_r1.negatives_per_row().mean()), "R2": float(_batches_r2.negatives_per_row().mean())}
del _batches_r1, _batches_r2
_base_parameters = sum(p.numel() for p in build_embedding_model().parameters())
_cross_encoder_parameters = sum(p.numel() for p in build_cross_encoder(0).parameters())
assert _cross_encoder_parameters == _base_parameters + EMBEDDING_DIMENSION + 1, (_cross_encoder_parameters, _base_parameters)
print(f"記事の並び: 同じシードの R1 と R2 で一致({FIRST_STAGE_SEEDS} シード)、組はシードごと・条件ごとに違う。R1 の passage は R2 の文脈を同じ記事の組の間で入れ替えたもの(query は同じ)で、R1 だけが同じ記事の組を負例から除く(1 行あたりの負例の数: R1 {NEGATIVES_PER_ROW['R1']:.0f}、R2 {NEGATIVES_PER_ROW['R2']:.0f})。総パラメータ数: R1・R2 {_base_parameters:,}(同じ構造)、cross-encoder {_cross_encoder_parameters:,}(本体 + ヘッド {EMBEDDING_DIMENSION + 1}): OK")

# --- 保持した埋め込みからの再計算・第 1 段の候補 ---
_kept = KEPT_EMBEDDINGS["R2"]
_recomputed = rank_single_positive(_kept["queries"].to(device), _kept["contexts"].to(device), EVALUATION_ENTRIES)
assert np.array_equal(_recomputed, RUNS[("R2", UPLOAD_SEED)]["passage_rank"]), "保持した埋め込みから再計算した順位が、記録した順位と一致しない"
for _s in sorted(MINED):
    for _name in ("train", "validation", "evaluation"):
        assert (MINED[_s][_name] >= 0).all() and MINED[_s][_name].shape[1] == RERANK_CANDIDATES
assert (MINED[UPLOAD_SEED]["evaluation"][:, 0] == exact_search(_kept["queries"].to(device), _kept["contexts"].to(device), 1)[0][:, 0]).all(), "第 1 段の上位 K 件の先頭が全探索の最上位と一致しない"
print("保持した埋め込み: 再計算した順位が記録した順位と一致(R2 のシード 0)、第 1 段の上位 K 件の先頭が全探索の最上位と一致: OK")
assert set(SAVED_STATES) == {("R2", UPLOAD_SEED), ("cross_encoder", UPLOAD_SEED)}, set(SAVED_STATES)
print(f"アップロード用に保持した重み: {sorted(SAVED_STATES)}(トークナイザなどは保持しない): OK")
TOTAL_ELAPSED_SECONDS = time.time() - NOTEBOOK_START_TIME
print(f"ここまでの経過時間 {TOTAL_ELAPSED_SECONDS / 60:.1f} 分(予算 {SESSION_BUDGET_SECONDS / 60:.0f} 分、選択時の見積もりの終了 {(ELAPSED_AT_SELECTION + STAGE_ESTIMATES[STAGE]['total']) / 60:.1f} 分)")

### 6.13 判定結果の一覧

In [ ]:
print(f"{RUN_TAG}判定結果(計画 {STAGE}: {STAGE_CHANGES[STAGE]}、T_1 = {FIRST_STAGE_STEPS}、T_2 = {CROSS_ENCODER_STEPS})")
for _name, _delta, _sigma, _computed, _verdict, _keys in (
    ("A", DELTA_A, SIGMA_A, A_COMPUTED, A_VERDICT, A_PRECONDITIONS),
    ("B", DELTA_B, SIGMA_B, B_COMPUTED, B_VERDICT, B_PRECONDITIONS),
    ("C", DELTA_C, SIGMA_C, C_COMPUTED, C_VERDICT, C_PRECONDITIONS),
):
    print(
        f"  実験 {_name}: 対比量 {_delta:+.4f}、標準偏差 {_sigma['sigma']:.4f}、閾値 {SIGMA_MULTIPLIER * _sigma['sigma']:.4f}、前提条件 "
        + "、".join(f"{k} {precondition_status[k]}" for k in _keys)
        + f"、判定関数の結果 {_computed}、最終判定 {_verdict}"
    )
print(f"  学習率: {json.dumps({c: float(f'{v:.3g}') for c, v in LEARNING_RATE.items()})}、出どころ {json.dumps(LEARNING_RATE_SOURCE, ensure_ascii=False)}")

### 6.14 Hugging Face Hub へのアップロード(R2 のシード 0・cross-encoder のシード 0)

後続のアプリ(`apps/`)の入力にする 2 つのモデルを、`UPLOAD_ARTIFACTS`が`True`のときだけアップロードする。アップロードの前に、書き出した重みを新しいモデルに読み込み直して評価し、モデルカードの指標にはその値を使う。
**公開の条件(判定ではない)** を満たさない場合はアップロードしない。

- R2 のシード 0 → `kojikojiprg/ai-theories-text-embedding-en`の新しいブランチ`inverse-cloze-task`(`main`の 024 のモデルと同じ構造なので、同じリポジトリのブランチにする)。公開の条件: 読み込み直した重みの passage 単位の評価での Recall@10 が、R1 のシード 0 の値を上回ること。
- cross-encoder のシード 0 → 新しいリポジトリ`kojikojiprg/ai-theories-cross-encoder-en`。公開の条件: 読み込み直した重みで、R2 のシード 0 の上位 $K$ 件を並べ替えた Recall@10 が、並べ替えなしの値を上回ること(評価用の query)。

`HF_TOKEN`は Colab Secrets から取得する(取得できなければ、停止せずスキップする)。トークンの値は印字・記録しない。アップロードの後、`list_repo_files()`で意図したファイルが存在することを確認する。

In [ ]:
_t0_upload = time.time()
import tempfile  # noqa: E402

EMBEDDING_CARD_TEMPLATE = '''---
language: en
license: mit
tags:
- ai-theories
- text-embedding
- retrieval
- inverse-cloze-task
- matryoshka-representation-learning
- scratch-implementation
---

# ai-theories テキスト埋め込みモデル(英語、Inverse Cloze Task、`{branch}` ブランチ)

`ai-theories`(https://github.com/kojikojiprg/ai-theories)プロジェクトの成果物。`main` ブランチ(024 の記事単位の定義で学習したモデル)と **同じ構造** で、**学習の組の作り方が違う** モデルである。
008 の小型 GPT(`kojikojiprg/ai-theories-small-gpt-en`)を encoder として、Inverse Cloze Task(passage から 1 文を抜いて query にし、その文を除いた残りを正例の文脈にする)と
Matryoshka Representation Learning(入れ子の次元 {matryoshka_dimensions})で、全パラメータを更新したもの。スクラッチ実装であり、研究・教育目的のモデルである。品質保証は行っていない。商用・実運用での利用は想定しない。

## 由来

- [026. Inverse Cloze Task と passage 単位の 2 段階検索](https://github.com/kojikojiprg/ai-theories/blob/main/theories/07_retrieval/026_inverse_cloze_task_and_two_stage_retrieval.ipynb)の
  条件 R2 のシード 0 の学習後の重み。結果を見て選んだものではなく、事前に固定した条件とシードである。
- 学習: {num_steps} ステップ、バッチ {batch_size} 組、温度 {temperature}、学習率 {learning_rate:.3g}、AdamW(重み減衰 {weight_decay})、warmup + cosine、gradient clipping {clip}。
  1 バッチは {articles_per_batch} 記事 x 2 組で、同じ記事の組を意図的に同じバッチに入れる(同じ記事の組の passage も負例として数える)。
  Inverse Cloze Task: 長さ {passage_length} トークンの窓から、長さ {min_query}〜{max_removed} トークンの文を 1 つ選んで query(先頭 {query_length} トークンまで)にし、残りを文脈にする。確率 {keep_probability} で文を文脈に残す。
  学習データは `kojikojiprg/ai-theories-corpus-en` の、先頭の 356 記事を除く記事のうち学習用に分けた {num_train_articles} 記事のみ(検証用・評価用の記事は含まない)。

## 使うために必要な他のアーティファクト

- **トークナイザは同梱していない。** `kojikojiprg/ai-theories-tokenizer-en` を使用してください(語彙サイズ 8192、特殊トークンなし)。
- 本体の構成は `config.json` を参照。

## 埋め込みの作り方

`main` ブランチと同じ(因果マスク、平均プール、L2 正規化、先頭 m 次元の切り詰めと再正規化)。**違い**: 学習で見た入力が可変長である(query は抜いた文で 4〜{query_length} トークン、文脈は 32〜{passage_length} トークン)。
可変長の入力は右側をパディングして長さを渡す(因果マスクのもとでは、右側のパディングは手前の位置の出力に影響しない。平均プールはパディングを除く)。

```python
import json
import torch
from huggingface_hub import hf_hub_download
from src.models.text_embedding import TextEmbeddingModel
from src.models.variable_length import encode_tokens_with_lengths

config = json.load(open(hf_hub_download("{repo_id}", "config.json", revision="{branch}")))
# GPTLanguageModel の構築は 008(kojikojiprg/ai-theories-small-gpt-en)の config.json と同じ(026 のノートブックの build_gpt() を参照)
backbone = build_gpt(config)
backbone.load_state_dict(torch.load(hf_hub_download("{repo_id}", "model_state.pt", revision="{branch}"), map_location="cpu"))
model = TextEmbeddingModel(backbone, pooling="mean", attention="causal").eval()
embeddings = encode_tokens_with_lengths(model, padded_token_ids, lengths, batch_size=256)[256]  # 右側をパディングしたトークンと有効な長さ
```

## 評価(アップロードした重みそのものを読み込み直して測った値)

評価用の記事(300 記事)の passage から作った query({num_queries} 個。passage から抜いた文)を、コーパス全体の索引({num_index} passage。すべて「1 文を除いた形」)に対して、全件の内積で検索した。正例は query の passage だけ。

| 評価 | このモデル(R2 のシード 0) | R1 のシード 0(記事単位の定義、`main` と同じ構造) |
|---|---|---|
| passage 単位の Recall@10 | {recall_passage:.4f} | {recall_passage_r1:.4f} |
| 記事単位のラベルの Recall@10(同じ索引・同じ query。正解は同じ記事の他の passage) | {recall_article:.4f} | {recall_article_r1:.4f} |
| 024 と同じ形式の記事単位の評価の Recall@10(024 の評価用の索引と query) | {recall_format024:.4f} | {recall_format024_r1:.4f} |

- ランダムな順位の passage 単位の Recall@10 の期待値: {random_recall:.6f}。
- 上の値は {smoke_note}

## 注意

- 学習データは英語 Wikipedia の長大な記事(Special:LongPages の長い順に選んだ記事)に限られ、汎用の検索器として使えるものではない。
- passage 単位の評価の索引と query は、ノートブックの実行時点のコーパス(`article_offsets` を含む `metadata.json`)に依存する。
'''

CROSS_ENCODER_CARD_TEMPLATE = '''---
language: en
license: mit
tags:
- ai-theories
- cross-encoder
- reranking
- retrieval
- scratch-implementation
---

# ai-theories cross-encoder(英語、passage 単位の並べ替えモデル)

`ai-theories`(https://github.com/kojikojiprg/ai-theories)プロジェクトの成果物。008 の小型 GPT(`kojikojiprg/ai-theories-small-gpt-en`)を起点に、query と passage を連結して符号化し、
終端の位置の出力から並べ替えのスコアを出す cross-encoder として、全パラメータを更新して学習したもの。第 1 段の検索器(`kojikojiprg/ai-theories-text-embedding-en` の `inverse-cloze-task` ブランチ)の
上位 {rerank_candidates} 件から採掘した正例・負例で学習した。スクラッチ実装であり、研究・教育目的のモデルである。品質保証は行っていない。商用・実運用での利用は想定しない。

## 由来

- [026. Inverse Cloze Task と passage 単位の 2 段階検索](https://github.com/kojikojiprg/ai-theories/blob/main/theories/07_retrieval/026_inverse_cloze_task_and_two_stage_retrieval.ipynb)の
  cross-encoder のシード 0 の学習後の重み。結果を見て選んだものではなく、事前に固定したシードである。
- 学習: {num_steps} ステップ、1 ステップ {batch_queries} query x(正例 1 + 負例 {num_negatives})、学習率 {learning_rate:.3g}、AdamW(重み減衰 {weight_decay})、warmup + cosine、gradient clipping {clip}。
  損失は 1 個の正例と {num_negatives} 個の負例の softmax 交差エントロピー。query は passage から抜いた文(先頭 {query_length} トークンまで)、正例はその文を除いた passage、負例は第 1 段の上位 {rerank_candidates} 件から正例を除いて選んだ passage。
  学習データは `kojikojiprg/ai-theories-corpus-en` の、先頭の 356 記事を除く記事のうち学習用に分けた {num_train_articles} 記事のみ(検証用・評価用の記事は含まない)。

## 使うために必要な他のアーティファクト

- **トークナイザは同梱していない。** `kojikojiprg/ai-theories-tokenizer-en` を使用してください(語彙サイズ 8192、特殊トークンなし)。
- **第 1 段の検索器は同梱していない。** 並べ替える候補は、`kojikojiprg/ai-theories-text-embedding-en` の `inverse-cloze-task` ブランチの全探索または近似最近傍探索の上位 {rerank_candidates} 件を想定している。
- 本体の構成は `config.json` を参照。

## 入力とスコア

入力の列は `[query のトークン] [区切り] [passage のトークン] [終端]` を **左から詰めて** 並べたもの(右側をパディング。長さの配列を渡す)。008 のトークナイザには特殊トークンがないので、語彙を増やさず、
コーパスに一度も現れない単一バイトのトークンを使う: 区切りは 0x1F(ID {separator_id})、終端は 0x1E(ID {terminal_id})。スコアは、因果マスクのもとで query 全体と passage 全体を見ている **終端の位置** の、
最終正規化層の後の隠れ状態に、線形の層(`head`)を掛けたスカラーである。

```python
import json
import torch
from huggingface_hub import hf_hub_download
from src.models.variable_length import VariableLengthCrossEncoder

config = json.load(open(hf_hub_download("{repo_id}", "config.json")))
backbone = build_gpt(config)  # 026 のノートブックの build_gpt() を参照
model = VariableLengthCrossEncoder(backbone, config["separator_token_id"], config["terminal_token_id"])
model.load_state_dict(torch.load(hf_hub_download("{repo_id}", "model_state.pt"), map_location="cpu"))
model.eval()
with torch.no_grad():
    scores = model(query_tokens, passage_tokens, query_lengths, passage_lengths)  # 形状 (B,)。トークンは右側をパディング
```

## 評価(アップロードした重みそのものを読み込み直して測った値)

評価用の記事の passage から作った query({num_queries} 個)について、コーパス全体({num_index} passage)に対する第 1 段(R2 のシード 0 の全探索)の上位 {rerank_candidates} 件を並べ替えて、Recall@10 を測った(正例は query の passage だけ)。

| 指標 | 並べ替えなし(第 1 段の順位) | このモデルで並べ替え |
|---|---|---|
| Recall@10 | {first_stage_recall:.4f} | {recall:.4f} |
| 平均逆順位(Mean Reciprocal Rank) | {first_stage_mrr:.4f} | {mrr:.4f} |

- 並べ替えで到達できる Recall@10 の上限(上位 {rerank_candidates} 件に正例が入る query の割合): {coverage:.4f}。
- 上の値は {smoke_note}

## 注意

- 学習データは英語 Wikipedia の長大な記事に限られ、汎用の並べ替えモデルとして使えるものではない。
- 関連性は「query の文を抜いた元の passage ただ 1 つ」と定義した評価であり、同じ内容を含む別の passage は負例として扱われる。
'''


def write_embedding_files(directory: Path) -> dict:
    state = SAVED_STATES[("R2", UPLOAD_SEED)]
    torch.save(state, directory / "model_state.pt")
    config = dict(REFERENCE_CONFIG) | {
        "embedding_dimension": EMBEDDING_DIMENSION, "pooling": "mean", "attention": "causal", "matryoshka_dimensions": list(MATRYOSHKA_DIMENSIONS), "temperature": TEMPERATURE,
        "query_length": QUERY_LENGTH, "passage_length": PASSAGE_LENGTH, "variable_length_inputs": True, "training": "inverse_cloze_task", "keep_probability": KEEP_PROBABILITY,
        "tokenizer_repo": TOKENIZER_REPO_ID, "base_model_repo": REFERENCE_MODEL_REPO_ID, "source_notebook": "theories/07_retrieval/026_inverse_cloze_task_and_two_stage_retrieval.ipynb",
    }
    (directory / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
    reloaded = build_embedding_model(0)
    reloaded.backbone.load_state_dict(torch.load(directory / "model_state.pt", map_location="cpu"))
    assert all(torch.equal(a, b) for a, b in zip(reloaded.backbone.state_dict().values(), state.values(), strict=True)), "書き出した重みが一致しない"
    reloaded = reloaded.to(device).eval()
    contexts, queries = embed_contexts(reloaded), embed_cloze_queries(reloaded, EVALUATION_ENTRIES)
    passage_ranks = rank_single_positive(queries, contexts, EVALUATION_ENTRIES)
    article = rank_article_label(queries, contexts, EVALUATION_ENTRIES, ENTRY_ARTICLES)
    format_queries, format_passages = embed_article_set(reloaded, "evaluation")
    format_024 = evaluate_embeddings(format_queries, format_passages, EVALUATION_SET)
    del reloaded, contexts, queries
    empty_device_cache()
    recorded = float(RUNS[("R2", UPLOAD_SEED)]["passage_hits"].mean())
    assert abs(float((passage_ranks <= 10).mean()) - recorded) <= 2 / len(EVALUATION_ENTRIES), (float((passage_ranks <= 10).mean()), recorded)  # 学習直後の値(記録)と、読み込み直した重みでの値が一致する
    card = EMBEDDING_CARD_TEMPLATE.format(
        branch=EMBEDDING_UPLOAD_BRANCH, repo_id=EMBEDDING_REPO_ID, matryoshka_dimensions=", ".join(str(m) for m in MATRYOSHKA_DIMENSIONS), num_steps=FIRST_STAGE_STEPS, batch_size=BATCH_SIZE,
        temperature=TEMPERATURE, learning_rate=LEARNING_RATE["R2"], weight_decay=WEIGHT_DECAY, clip=GRADIENT_CLIP_THRESHOLD, passage_length=PASSAGE_LENGTH, min_query=MIN_QUERY_TOKENS,
        max_removed=MAX_REMOVED_TOKENS, query_length=QUERY_LENGTH, keep_probability=KEEP_PROBABILITY, num_train_articles=f"{len(SPLIT.train):,}", num_queries=f"{len(EVALUATION_ENTRIES):,}",
        num_index=f"{NUM_INDEX:,}", recall_passage=float((passage_ranks <= 10).mean()), recall_passage_r1=float(RUNS[("R1", UPLOAD_SEED)]["passage_hits"].mean()),
        recall_article=float((article.rank <= 10).mean()), recall_article_r1=float(RUNS[("R1", UPLOAD_SEED)]["article_hits"].mean()), recall_format024=format_024.recall_at(10),
        recall_format024_r1=float(RUNS[("R1", UPLOAD_SEED)]["format024_hits"].mean()), articles_per_batch=ARTICLES_PER_BATCH, random_recall=RANDOM_RECALL_PASSAGE,
        smoke_note="動作確認の値であり、意味を持たない。" if SMOKE_TEST else "本番の実行の値である。",
    )
    (directory / "README.md").write_text(card, encoding="utf-8")
    return {"passage_recall": float((passage_ranks <= 10).mean()), "card": card, "files": sorted(p.name for p in directory.iterdir())}


def write_cross_encoder_files(directory: Path) -> dict:
    state = SAVED_STATES[("cross_encoder", UPLOAD_SEED)]
    torch.save(state, directory / "model_state.pt")
    config = dict(REFERENCE_CONFIG) | {
        "model_type": "cross_encoder", "separator_token_id": SEPARATOR_TOKEN_ID, "terminal_token_id": TERMINAL_TOKEN_ID,
        "head": "linear(d_model -> 1) applied to the final-normalized hidden state of the terminal position", "variable_length_inputs": True,
        "query_length": QUERY_LENGTH, "passage_length": PASSAGE_LENGTH, "rerank_candidates": RERANK_CANDIDATES, "tokenizer_repo": TOKENIZER_REPO_ID,
        "base_model_repo": REFERENCE_MODEL_REPO_ID, "first_stage_repo": EMBEDDING_REPO_ID, "first_stage_branch": EMBEDDING_UPLOAD_BRANCH,
        "source_notebook": "theories/07_retrieval/026_inverse_cloze_task_and_two_stage_retrieval.ipynb",
    }
    (directory / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
    reloaded = build_cross_encoder(0)
    reloaded.load_state_dict(torch.load(directory / "model_state.pt", map_location="cpu"))
    assert all(torch.equal(a, b) for a, b in zip(reloaded.state_dict().values(), state.values(), strict=True)), "書き出した重みが一致しない"
    reloaded = reloaded.to(device)
    evaluated, scores = evaluate_rerank(reloaded, RERANK_EVALUATION_ENTRIES, MINED[UPLOAD_SEED]["evaluation"][RERANK_EVALUATION_POSITIONS])
    del reloaded
    empty_device_cache()
    assert abs(evaluated.recall_at(10) - float(CROSS_ENCODER_RUNS[UPLOAD_SEED]["hits"].mean())) <= 2 / len(RERANK_EVALUATION_ENTRIES), evaluated.recall_at(10)
    first = rerank_single_positive(
        np.broadcast_to(-np.arange(RERANK_CANDIDATES, dtype=np.float64), (len(RERANK_EVALUATION_ENTRIES), RERANK_CANDIDATES)), MINED[UPLOAD_SEED]["evaluation"][RERANK_EVALUATION_POSITIONS], RERANK_EVALUATION_ENTRIES, NUM_INDEX
    )
    card = CROSS_ENCODER_CARD_TEMPLATE.format(
        repo_id=CROSS_ENCODER_REPO_ID, rerank_candidates=RERANK_CANDIDATES, num_steps=CROSS_ENCODER_STEPS, batch_queries=BATCH_QUERIES, num_negatives=NUM_NEGATIVES, learning_rate=LEARNING_RATE["cross_encoder"],
        weight_decay=WEIGHT_DECAY, clip=GRADIENT_CLIP_THRESHOLD, query_length=QUERY_LENGTH, num_train_articles=f"{len(SPLIT.train):,}", separator_id=SEPARATOR_TOKEN_ID, terminal_id=TERMINAL_TOKEN_ID,
        num_queries=f"{len(RERANK_EVALUATION_ENTRIES):,}", num_index=f"{NUM_INDEX:,}", first_stage_recall=first.recall_at(10), recall=evaluated.recall_at(10), first_stage_mrr=first.mean_reciprocal_rank(),
        mrr=evaluated.mean_reciprocal_rank(), coverage=evaluated.coverage(), smoke_note="動作確認の値であり、意味を持たない。" if SMOKE_TEST else "本番の実行の値である。",
    )
    (directory / "README.md").write_text(card, encoding="utf-8")
    return {"recall": evaluated.recall_at(10), "first_stage_recall": first.recall_at(10), "card": card, "files": sorted(p.name for p in directory.iterdir())}


def get_hub_token():
    try:
        from google.colab import userdata  # noqa: E402

        return userdata.get("HF_TOKEN")
    except Exception:  # noqa: BLE001  # Colab 以外・Secrets 未設定・取得失敗のいずれでも、停止せずスキップする
        return None


with tempfile.TemporaryDirectory() as _embedding_dir, tempfile.TemporaryDirectory() as _cross_encoder_dir:
    _embedding = write_embedding_files(Path(_embedding_dir))
    _cross_encoder = write_cross_encoder_files(Path(_cross_encoder_dir))
    assert _embedding["files"] == ["README.md", "config.json", "model_state.pt"] and _cross_encoder["files"] == ["README.md", "config.json", "model_state.pt"], "トークナイザなど、同梱しないファイルが含まれている"
    print(f"{RUN_TAG}モデルカードの下書きと書き出したファイル {_embedding['files']} を作成し、重みを読み込み直して評価した")
    print("--- R2 のモデルカードの先頭 ---")
    print("\n".join(_embedding["card"].splitlines()[:14]))
    print("--- cross-encoder のモデルカードの先頭 ---")
    print("\n".join(_cross_encoder["card"].splitlines()[:14]))
    # 公開の条件(判定ではない)
    EMBEDDING_CONDITION = bool(_embedding["passage_recall"] > float(RUNS[("R1", UPLOAD_SEED)]["passage_hits"].mean()))
    CROSS_ENCODER_CONDITION = bool(_cross_encoder["recall"] > _cross_encoder["first_stage_recall"])
    print(f"{RUN_TAG}公開の条件(判定ではない): R2 のシード 0(読み込み直した重み)の passage 単位の Recall@10 {_embedding['passage_recall']:.4f} > R1 のシード 0 の {float(RUNS[('R1', UPLOAD_SEED)]['passage_hits'].mean()):.4f}: {EMBEDDING_CONDITION}")
    print(f"{RUN_TAG}公開の条件(判定ではない): cross-encoder のシード 0(読み込み直した重み)の並べ替えの Recall@10 {_cross_encoder['recall']:.4f} > 並べ替えなし {_cross_encoder['first_stage_recall']:.4f}: {CROSS_ENCODER_CONDITION}")
    if not UPLOAD_ARTIFACTS:
        print("UPLOAD_ARTIFACTS = False のため、アップロードは行わない(モデルカードの下書きの確認のみ)")
    elif not (EMBEDDING_CONDITION or CROSS_ENCODER_CONDITION):
        print("どちらのモデルも公開の条件を満たさないため、アップロードを行わない")
    else:
        _token = get_hub_token()
        if not _token:
            print("Colab Secrets から HF_TOKEN を取得できないため、アップロードをスキップする")
        else:
            from huggingface_hub import HfApi

            _api = HfApi(token=_token)
            if EMBEDDING_CONDITION:
                _api.create_branch(EMBEDDING_REPO_ID, branch=EMBEDDING_UPLOAD_BRANCH, repo_type="model", exist_ok=True)
                for _name in _embedding["files"]:
                    _api.upload_file(path_or_fileobj=str(Path(_embedding_dir) / _name), path_in_repo=_name, repo_id=EMBEDDING_REPO_ID, repo_type="model", revision=EMBEDDING_UPLOAD_BRANCH)
                _remote = sorted(_api.list_repo_files(EMBEDDING_REPO_ID, repo_type="model", revision=EMBEDDING_UPLOAD_BRANCH))
                assert not [n for n in _embedding["files"] if n not in _remote], f"アップロードしたはずのファイルがリポジトリにない: {_remote}"
                print(f"アップロード完了: {EMBEDDING_REPO_ID}(ブランチ {EMBEDDING_UPLOAD_BRANCH})のファイル {_remote}(意図したファイルがすべて存在することを list_repo_files() で確認)")
            else:
                print("R2 のシード 0 は公開の条件を満たさないため、アップロードしない")
            if CROSS_ENCODER_CONDITION:
                _api.create_repo(CROSS_ENCODER_REPO_ID, repo_type="model", exist_ok=True, private=False)
                for _name in _cross_encoder["files"]:
                    _api.upload_file(path_or_fileobj=str(Path(_cross_encoder_dir) / _name), path_in_repo=_name, repo_id=CROSS_ENCODER_REPO_ID, repo_type="model")
                _remote = sorted(_api.list_repo_files(CROSS_ENCODER_REPO_ID, repo_type="model"))
                assert not [n for n in _cross_encoder["files"] if n not in _remote], f"アップロードしたはずのファイルがリポジトリにない: {_remote}"
                print(f"アップロード完了: {CROSS_ENCODER_REPO_ID} のファイル {_remote}(意図したファイルがすべて存在することを list_repo_files() で確認)")
            else:
                print("cross-encoder のシード 0 は公開の条件を満たさないため、アップロードしない")
print(f"アップロードの準備 {time.time() - _t0_upload:.1f} 秒")

## 7. 結果・考察 / Results and Discussion

本番の実行の後に、実際のセル出力と突き合わせて、各節の本文を書く。各節に書く内容は次のとおり。

### 7.1 実行の概要

- 実行環境(5.1 節の出力: GPU、ライブラリのバージョン、コミット、実行日時)。
- 選ばれた計画(6.4 節)、$T_1$、パイロットで選ばれた $T_2$ とパイロットの結果(6.3 節)、学習率とその出どころ(6.5 節)、各実験のシード数。
- 前提条件の成否の一覧と、判定の一覧(6.13 節)。

### 7.2 前提条件

- P0〜P3 の値と成否を実験ごとに。成立しなかった前提条件があれば、その理由の事後的な解釈は 7.11 節に分ける。
- 前提条件の閾値の下限(6.1 節)と、本番の増分・標準偏差の比較。

### 7.3 実験 A: passage 単位の評価での、関連性の定義の効果

- $R(\mathrm{R1})$・$R(\mathrm{R2})$ のシード平均、$\Delta_A$、$\sigma_A$、閾値、最終判定。
- 作用点に近い診断量(上位 10 件のうち同じ記事の passage の割合、1 位が同じ記事の別の passage である割合、平均逆順位)の読み取り。切り詰めた query の割合(学習・評価)。

### 7.4 実験 B: 記事単位の評価での、関連性の定義の効果

- $R(\mathrm{R1})$・$R(\mathrm{R2})$ のシード平均、$\Delta_B$、$\sigma_B$、閾値、最終判定。診断量(平均逆順位、正規化割引累積利得)と、024 と同じ形式の記事単位の評価(R1・R2・024 の Hub のモデル)の読み取り。

### 7.5 実験 C: 2 段階の検索の効果

- $R(\mathrm{R2})$・$R(\mathrm{R3})$ のシード平均、$\Delta_C$、$\sigma_C$、閾値、最終判定。coverage と並べ替えの余地。
- 実用の問いとしての読み方(追加の学習量と費用との関係)。

### 7.6 実験 A と B の組み合わせ

- 6.1 節で宣言した読み方の表のうち、該当する行・列。規則で決めた学習率の偏りがある場合の割り引き。

### 7.7 観察 D・E

- 観察 D の表(第 1 段の方法、費用、$K$ ごとの指標)の読み取り。
- 観察 E の検索結果の例から読める、R1 と R2 の違い(話題は合っているが passage を取り違える、など。観察であり、検証済みの結論ではない)。

### 7.8 診断量と R1 の再現の確認

- R1 のシード 0 と 024 の値の対応。学習前の 008 の値との比較。学習時と推論時の分布の違いの検出(coverage の学習用・検証用・評価用の比較)。

### 7.9 実行時間と見積もりの対応

- 6.2 節の見積もり、選択時の見積もり、実際の時間の対応。

### 7.10 Hugging Face Hub へのアップロード

- 公開の条件の成否と、アップロードした(または、しなかった)ファイルとその確認。

### 7.11 事後的な解釈(検証済みの結論ではない)

- 事前に宣言した判定とは別に、結果を見た後に立てた解釈を、節を分けて書く。最終判定が支持以外の実験(反証・判定不能・前提不成立)について、次の 4 つを必ず書く: (1)原因の候補(設計と現象の両面)、(2)根拠(セル出力の診断量。根拠がなければその旨)、(3)確かめる方法(実行しない)、(4)教訓。
- 前提不成立の実験があれば、「なぜ前提が崩れたか」と「判定関数の参考値がなぜその向きになったか」を別々の問いとして扱う。

### 7.12 制約と一般化の範囲

- コーパス(英語 Wikipedia の長大な記事 9,470 本、リスト系が約 30%)、1 つのチェックポイント(008)、$T_1 = 1024$ の学習量(約 0.059 エポック)、文の分割の規則の既知の誤り、R2 が同じ記事の困難な負例を見ることによる実験 A の対比量への寄与、1 行あたりの負例の数の違い(62 個と 63 個)、$T_2$ の選択の偏り、規則で決めた学習率の偏り(段階 4 以降)などの制約を書く。